In [13]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split

# ===================== 配置区 =====================
path = r"H:\HW\ruc_Class25Q2_train_price.csv"

# 划分参数
test_size = 0.2
random_state = 111

# 删除的列
cols_to_drop = [
    "套内面积","梯户比例","别墅类型","交易权属","上次交易","房屋用途","房屋年限","产权所属","抵押信息",
    "房屋优势","核心卖点","户型介绍","周边配套","交通出行","区县","板块_comm","环线位置","环线",
    "年份","开发商","物业公司","建筑结构_comm","物业办公电话","产权描述","供水","供电","供暖",
    "燃气费","供热费","停车位","coord_x","coord_y","客户反馈","物业类别"
]
# =================================================


print("=" * 80)
print("数据清洗 + 训练/测试集划分")
print("=" * 80)

# 1. 读取CSV
print("\n[1/4] 读取数据")
enc = None
for e in ["utf-8-sig", "gbk", "utf-8"]:
    try:
        df = pd.read_csv(path, encoding=e)
        enc = e
        print(f"  ✓ 成功读取（编码：{enc}）")
        break
    except Exception:
        pass

if enc is None:
    raise RuntimeError("无法读取CSV，请确认文件路径和编码")

print(f"  原始数据: {len(df)} 行 × {len(df.columns)} 列")

# 2. 划分训练/测试集（最开始！）
print(f"\n[2/4] 划分训练/测试集（test_size={test_size}, random_state={random_state}）")
df_train, df_test = train_test_split(
    df, 
    test_size=test_size, 
    random_state=random_state,
    shuffle=True
)

# 添加标识列
df_train = df_train.copy()
df_test = df_test.copy()
df_train['is_train'] = 1
df_test['is_train'] = 0

print(f"  训练集: {len(df_train)} 行 ({len(df_train)/len(df)*100:.1f}%)")
print(f"  测试集: {len(df_test)} 行 ({len(df_test)/len(df)*100:.1f}%)")

# 3. 合并回一个DataFrame（保持顺序：训练集在前）
df_combined = pd.concat([df_train, df_test], axis=0, ignore_index=True)
print(f"  合并后: {len(df_combined)} 行")

# 4. 删除不需要的列
print(f"\n[3/4] 删除列")
existing_drop_cols = [c for c in cols_to_drop if c in df_combined.columns]
print(f"  删除 {len(existing_drop_cols)} 列: {existing_drop_cols[:5]}{'...' if len(existing_drop_cols) > 5 else ''}")

df_clean = df_combined.drop(columns=existing_drop_cols, errors="ignore")

print(f"  保留列: {len(df_clean.columns)} 个")

# 5. 保存文件
print(f"\n[4/4] 保存文件")
out_path = os.path.splitext(path)[0] + "_clean.csv"
df_clean.to_csv(out_path, index=False, encoding="utf-8-sig")

# 统计输出
print("\n" + "=" * 80)
print("处理完成")
print("=" * 80)
print(f"输出文件: {out_path}")
print(f"总行数: {len(df_clean)}")
print(f"  训练集: {(df_clean['is_train']==1).sum()} 行")
print(f"  测试集: {(df_clean['is_train']==0).sum()} 行")
print(f"总列数: {len(df_clean.columns)}")
print(f"\n保留的列:")
for i, col in enumerate(df_clean.columns, 1):
    print(f"  {i:2d}. {col}")
print("=" * 80)

数据清洗 + 训练/测试集划分

[1/4] 读取数据
  ✓ 成功读取（编码：utf-8-sig）
  原始数据: 103871 行 × 55 列

[2/4] 划分训练/测试集（test_size=0.2, random_state=111）
  训练集: 83096 行 (80.0%)
  测试集: 20775 行 (20.0%)
  合并后: 103871 行

[3/4] 删除列
  删除 34 列: ['套内面积', '梯户比例', '别墅类型', '交易权属', '上次交易']...
  保留列: 22 个

[4/4] 保存文件

处理完成
输出文件: H:\HW\ruc_Class25Q2_train_price_clean.csv
总行数: 103871
  训练集: 83096 行
  测试集: 20775 行
总列数: 22

保留的列:
   1. 城市
   2. 区域
   3. 板块
   4. Price
   5. 房屋户型
   6. 所在楼层
   7. 建筑面积
   8. 房屋朝向
   9. 建筑结构
  10. 装修情况
  11. 配备电梯
  12. 交易时间
  13. lon
  14. lat
  15. 建筑年代
  16. 房屋总数
  17. 楼栋总数
  18. 绿 化 率
  19. 容 积 率
  20. 物 业 费
  21. 停车费用
  22. is_train


In [14]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# ========= 配置 =========
path = r"H:\HW\ruc_Class25Q2_train_price_clean.csv"

# winsorize 截尾比例（仅用于训练集）
LOWER_P = 0.01
UPPER_P = 0.01
# ======================

print("=" * 80)
print("房屋户型解析（训练集Winsorize + 测试集不截尾）")
print("=" * 80)

# 1. 读取CSV
print("\n[1/5] 读取数据")
enc = None
for e in ["utf-8-sig", "gbk", "utf-8"]:
    try:
        df = pd.read_csv(path, encoding=e)
        enc = e
        break
    except Exception:
        pass

if enc is None:
    raise RuntimeError("无法读取CSV")

if "房屋户型" not in df.columns:
    raise KeyError("缺少 '房屋户型' 列")

if "is_train" not in df.columns:
    raise KeyError("缺少 'is_train' 列，请先运行数据划分代码")

print(f"  总样本: {len(df)}")
print(f"  训练集: {(df['is_train']==1).sum()}")
print(f"  测试集: {(df['is_train']==0).sum()}")


# 2. 中文数字转换函数
def cn2num(token):
    if token is None:
        return np.nan
    token = str(token).strip()
    try:
        return float(token)
    except Exception:
        pass
    cmap = {"零":0,"一":1,"二":2,"两":2,"三":3,"四":4,"五":5,"六":6,"七":7,"八":8,"九":9}
    if token == "十":
        return 10.0
    if "十" in token:
        parts = token.split("十")
        tens = 1 if parts[0] in ("", "零") else cmap.get(parts[0], np.nan)
        ones = 0 if len(parts) == 1 or parts[1] == "" else cmap.get(parts[1], np.nan)
        if pd.isna(tens) or pd.isna(ones):
            return np.nan
        return float(tens * 10 + ones)
    if token in cmap:
        return float(cmap[token])
    return np.nan


# 3. 提取室/厅/厨/卫函数
pattern_parts = {
    "室": re.compile(r"([零一二两三四五六七八九十\d]+)\s*室"),
    "厅": re.compile(r"([零一二两三四五六七八九十\d]+)\s*厅"),
    "厨": re.compile(r"([零一二两三四五六七八九十\d]+)\s*厨"),
    "卫": re.compile(r"([零一二两三四五六七八九十\d]+)\s*卫"),
}

def extract_counts(text):
    if pd.isna(text):
        return pd.Series({"室": np.nan, "厅": np.nan, "厨": np.nan, "卫": np.nan})
    s = str(text)
    out = {}
    for k, pat in pattern_parts.items():
        m = pat.search(s)
        if m:
            out[k] = cn2num(m.group(1))
        else:
            out[k] = np.nan
    return pd.Series(out)


# 4. 分离训练集和测试集
print("\n[2/5] 分离训练/测试集")
df_train = df[df['is_train'] == 1].copy()
df_test = df[df['is_train'] == 0].copy()


# 5. 训练集：提取 + Winsorize + 计算中位数 + 填充
print("\n[3/5] 处理训练集")
print("  提取室/厅/厨/卫...")
counts_train = df_train["房屋户型"].apply(extract_counts)
df_train = pd.concat([df_train, counts_train], axis=1)

print(f"  Winsorize（截尾比例：{LOWER_P}/{UPPER_P}）...")
train_medians = {}  # 保存训练集中位数

for col in ["室", "厅", "厨", "卫"]:
    s = pd.to_numeric(df_train[col], errors="coerce")
    
    if s.notna().sum() == 0:
        print(f"    ⚠️ {col} 列全缺失")
        train_medians[col] = 0
        df_train[col] = 0
        continue
    
    # Winsorize截尾
    q_low = s.quantile(LOWER_P)
    q_high = s.quantile(1 - UPPER_P)
    s_clip = s.clip(lower=q_low, upper=q_high)
    
    # 计算中位数（截尾后的）
    med = s_clip.median()
    train_medians[col] = med
    
    # 填充训练集缺失值
    s_filled = s_clip.fillna(med).round().astype(int)
    df_train[col] = s_filled
    
    missing_count = s.isna().sum()
    print(f"    {col}: 截尾[{q_low:.1f}, {q_high:.1f}], 中位数={med:.1f}, 填充{missing_count}个缺失")

print(f"  ✓ 训练集处理完成")


# 6. 测试集：提取 + 不做Winsorize + 用训练集中位数填充
print("\n[4/5] 处理测试集")
print("  提取室/厅/厨/卫...")
counts_test = df_test["房屋户型"].apply(extract_counts)
df_test = pd.concat([df_test, counts_test], axis=1)

print("  用训练集中位数填充（不做Winsorize）...")
for col in ["室", "厅", "厨", "卫"]:
    s = pd.to_numeric(df_test[col], errors="coerce")
    med = train_medians[col]
    
    # 直接用训练集中位数填充，不做截尾
    missing_count = s.isna().sum()
    s_filled = s.fillna(med).round().astype(int)
    df_test[col] = s_filled
    
    print(f"    {col}: 填充{missing_count}个缺失（训练集中位数={med:.1f}）")

print(f"  ✓ 测试集处理完成")


# 7. 合并并保存
print("\n[5/5] 合并并保存")
df_final = pd.concat([df_train, df_test], axis=0, ignore_index=True)

# 删除原"房屋户型"列
df_final = df_final.drop(columns=["房屋户型"])

# 保存
out_path = Path(path).with_name(Path(path).stem + "1.csv")
df_final.to_csv(out_path, index=False, encoding="utf-8-sig")

# 统计输出
print("=" * 80)
print("处理完成")
print("=" * 80)
print(f"输入编码: {enc}")
print(f"输出文件: {out_path}")

print(f"\n训练集中位数（Winsorize后）:")
for col in ["室", "厅", "厨", "卫"]:
    print(f"  {col}: {train_medians[col]:.1f}")

print(f"\n新列统计（合并后）:")
print(df_final[["室","厅","厨","卫"]].describe())

print(f"\n按训练/测试集分组统计:")
print("\n【训练集】（已Winsorize）")
print(df_final[df_final['is_train']==1][["室","厅","厨","卫"]].describe())

print("\n【测试集】（未Winsorize，保留原始分布）")
print(df_final[df_final['is_train']==0][["室","厅","厨","卫"]].describe())

print("=" * 80)

房屋户型解析（训练集Winsorize + 测试集不截尾）

[1/5] 读取数据
  总样本: 103871
  训练集: 83096
  测试集: 20775

[2/5] 分离训练/测试集

[3/5] 处理训练集
  提取室/厅/厨/卫...
  Winsorize（截尾比例：0.01/0.01）...
    室: 截尾[1.0, 5.0], 中位数=3.0, 填充1045个缺失
    厅: 截尾[0.0, 3.0], 中位数=2.0, 填充1045个缺失
    厨: 截尾[0.0, 1.0], 中位数=1.0, 填充1045个缺失
    卫: 截尾[1.0, 4.0], 中位数=1.0, 填充471个缺失
  ✓ 训练集处理完成

[4/5] 处理测试集
  提取室/厅/厨/卫...
  用训练集中位数填充（不做Winsorize）...
    室: 填充254个缺失（训练集中位数=3.0）
    厅: 填充254个缺失（训练集中位数=2.0）
    厨: 填充254个缺失（训练集中位数=1.0）
    卫: 填充109个缺失（训练集中位数=1.0）
  ✓ 测试集处理完成

[5/5] 合并并保存
处理完成
输入编码: utf-8-sig
输出文件: H:\HW\ruc_Class25Q2_train_price_clean1.csv

训练集中位数（Winsorize后）:
  室: 3.0
  厅: 2.0
  厨: 1.0
  卫: 1.0

新列统计（合并后）:
                   室              厅              厨              卫
count  103871.000000  103871.000000  103871.000000  103871.000000
mean        2.610758       1.532670       0.991518       1.436628
std         0.954604       0.570654       0.111076       0.635679
min         0.000000       0.000000       0.000000       0.000000
25%       

In [15]:
import pandas as pd
import numpy as np
import re

# ========= 配置：请修改为你的CSV路径（覆盖原文件，不生成新文件）=========
path = r"H:\HW\ruc_Class25Q2_train_price_clean1.csv"
# =====================================================================

# 读取CSV并探测编码
enc = None
for e in ["utf-8-sig", "gbk", "utf-8"]:
    try:
        df = pd.read_csv(path, encoding=e)
        enc = e
        break
    except Exception:
        pass
if enc is None:
    raise RuntimeError("无法读取CSV，请检查路径或尝试 UTF-8/GBK 编码")

# 兼容：若列名使用“配对电梯”，改名为“配备电梯”
if "配备电梯" not in df.columns and "配对电梯" in df.columns:
    df = df.rename(columns={"配对电梯": "配备电梯"})

# 校验必要列
need_cols = ["所在楼层", "配备电梯"]
missing_cols = [c for c in need_cols if c not in df.columns]
if missing_cols:
    raise KeyError(f"缺少必要列：{missing_cols}，请确认列名是否正确")

# 将中文数字转换为阿拉伯数字（支持到99，含“两”、“十”、“二十五”等）
def cn2num(token):
    if token is None:
        return np.nan
    token = str(token).strip()
    # 直接数字
    try:
        return float(token)
    except Exception:
        pass
    cmap = {"零":0,"一":1,"二":2,"两":2,"三":3,"四":4,"五":5,"六":6,"七":7,"八":8,"九":9}
    if token == "十":
        return 10.0
    if "十" in token:
        parts = token.split("十")
        tens = 1 if parts[0] in ("", "零") else cmap.get(parts[0], np.nan)
        ones = 0 if len(parts) == 1 or parts[1] == "" else cmap.get(parts[1], np.nan)
        if pd.isna(tens) or pd.isna(ones):
            return np.nan
        return float(tens * 10 + ones)
    if token in cmap:
        return float(cmap[token])
    return np.nan

# 提取“共X层”的总楼层数（支持带或不带括号）
re_total = re.compile(r"共\s*([零一二两三四五六七八九十\d]+)\s*层")

def extract_total_floors(text):
    if pd.isna(text):
        return np.nan
    s = str(text)
    m = re_total.search(s)
    if not m:
        return np.nan
    return cn2num(m.group(1))

# 识别楼层类别 -> 六大类之一
def classify_floor(text):
    if pd.isna(text):
        return None
    s0 = re.sub(r"\s+", "", str(text))
    # 优先匹配具体描述
    if ("地下" in s0) or ("负" in s0):   # 如：地下室、负一层
        return "地下室"
    if "顶层" in s0:
        return "顶层"
    if "底层" in s0:
        return "底层"
    if ("低楼层" in s0) or ("低层" in s0):
        return "低楼层"
    if ("中楼层" in s0) or ("中层" in s0):
        return "中楼层"
    if ("高楼层" in s0) or ("高层" in s0):
        return "高楼层"
    return None  # 未识别

# 标准化“配备电梯”到 有/无/NaN（先否定后肯定，避免“没有电梯”被误判为“有”）
def normalize_lift(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace(" ", "")
    if s == "":
        return np.nan
    # 先匹配否定
    neg_keys = ["无电梯", "没有电梯", "不带电梯", "不配备", "否", "无", "没电梯", "未配"]
    if any(k in s for k in neg_keys):
        return "无"
    # 再匹配肯定
    pos_keys = ["有电梯", "带电梯", "配备电梯", "配电梯", "有梯", "带梯", "有", "是"]
    if any(k in s for k in pos_keys):
        return "有"
    # 兼容 0/1、Y/N、True/False
    if s in {"1", "Y", "y", "true", "True"}:
        return "有"
    if s in {"0", "N", "n", "false", "False"}:
        return "无"
    return np.nan

# 1) 楼层类别 one-hot：地下室/底层/低楼层/中楼层/高楼层/顶层
floor_cat = df["所在楼层"].apply(classify_floor)
floor_categories = ["地下室", "底层", "低楼层", "中楼层", "高楼层", "顶层"]
for cat in floor_categories:
    df[f"{cat}_01"] = (floor_cat == cat).astype("int8")

# 2) 提取总楼层数，用于电梯缺失填补
total_floors = df["所在楼层"].apply(extract_total_floors)

# 3) 配备电梯标准化 + 缺失填补（总楼层>6 -> 有，否则无）
lift_norm = df["配备电梯"].apply(normalize_lift)
rule_based = pd.Series(np.where(total_floors > 6, "有", "无"), index=df.index)  # 与索引对齐
lift_imputed = lift_norm.where(~lift_norm.isna(), rule_based)

# 生成电梯 one-hot 两列
df["配备电梯_有_01"] = (lift_imputed == "有").astype("int8")
df["配备电梯_无_01"] = (lift_imputed == "无").astype("int8")

# 4) 删除原始列
df = df.drop(columns=[c for c in ["所在楼层", "配备电梯"] if c in df.columns])

# 5) 覆盖保存到原文件（使用原始编码）
df.to_csv(path, index=False, encoding=enc or "utf-8-sig")

print("已完成处理：")
print("- 新增楼层六个01变量：", [f"{c}_01" for c in floor_categories])
print("- 新增电梯两个01变量：['配备电梯_有_01', '配备电梯_无_01']")
print("- 已删除原列：'所在楼层', '配备电梯'")
print(f"文件已覆盖保存：{path}（编码：{enc}）")

已完成处理：
- 新增楼层六个01变量： ['地下室_01', '底层_01', '低楼层_01', '中楼层_01', '高楼层_01', '顶层_01']
- 新增电梯两个01变量：['配备电梯_有_01', '配备电梯_无_01']
- 已删除原列：'所在楼层', '配备电梯'
文件已覆盖保存：H:\HW\ruc_Class25Q2_train_price_clean1.csv（编码：utf-8-sig）


In [16]:
import pandas as pd
import numpy as np
import re

# ========= 配置：请修改为你的CSV路径（覆盖原文件，不生成新文件）=========
path = r"H:\HW\ruc_Class25Q2_train_price_clean1.csv"
# =====================================================================

# 读取CSV并探测编码
enc = None
for e in ["utf-8-sig", "gbk", "utf-8"]:
    try:
        df = pd.read_csv(path, encoding=e)
        enc = e
        break
    except Exception:
        pass
if enc is None:
    raise RuntimeError("无法读取CSV，请检查路径或尝试 UTF-8/GBK 编码")

# 校验必要列
need_cols = ["房屋朝向", "建筑结构", "装修情况"]
missing_cols = [c for c in need_cols if c not in df.columns]
if missing_cols:
    raise KeyError(f"缺少必要列：{missing_cols}，请确认列名是否正确")

# -------------------------
# 1) 房屋朝向 -> 东/南/西/北 四个01变量（可多选，也可能全0）
# -------------------------
def normalize_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip().replace(" ", "")

ori_series = df["房屋朝向"].apply(normalize_text)

for d in ["东", "南", "西", "北"]:
    df[f"朝向_{d}_01"] = ori_series.str.contains(d, na=False).astype("int8")

# -------------------------
# 2) 建筑结构 -> 统一到六类：钢混结构/混合结构/未知结构/框架结构/钢结构/砖混结构
#    规则：砖结构 与 无数据 一律并入 混合结构
# -------------------------
def normalize_structure(x):
    s = normalize_text(x)
    # 修正规范化：把“混合机构”这种笔误当成“混合结构”
    s = s.replace("机构", "结构")

    # 无数据或空值 -> 混合结构（按要求）
    if s == "" or s in {"无数据", "暂无", "NA", "nan", "None", "无"}:
        return "混合结构"

    # 显式未知
    if "未知" in s:
        return "未知结构"

    # 砖混优先匹配
    if "砖混" in s:
        return "砖混结构"

    # 钢筋混凝土 / 钢混
    if ("钢筋混凝土" in s) or ("钢混" in s) or ("钢筋" in s and "混凝土" in s):
        return "钢混结构"

    # 框架
    if "框架" in s:
        return "框架结构"

    # 钢结构（注意不要误把钢筋混凝土归到钢结构，上面已先匹配）
    if "钢结构" in s or (("钢" in s) and ("结构" in s) and ("钢筋混凝土" not in s)):
        return "钢结构"

    # 砖结构 统一并入 混合结构（按要求）
    if ("砖结构" in s) or ("砖木" in s) or (("砖" in s) and ("砖混" not in s)):
        return "混合结构"

    # 混合结构
    if "混合" in s:
        return "混合结构"

    # 其余未识别 -> 未知结构（更稳妥）
    return "未知结构"

struct_norm = df["建筑结构"].apply(normalize_structure)
struct_categories = ["钢混结构", "混合结构", "未知结构", "框架结构", "钢结构", "砖混结构"]
for cat in struct_categories:
    df[f"建筑结构_{cat}_01"] = (struct_norm == cat).astype("int8")

# -------------------------
# 3) 装修情况 -> 四类：精装/简装/毛坯/其他（无数据视为“其他”）
# -------------------------
def normalize_decoration(x):
    s = normalize_text(x)
    if s == "" or s in {"无数据", "暂无", "NA", "nan", "None", "无"}:
        return "其他"

    # 毛坯类
    if ("毛坯" in s) or ("清水" in s):
        return "毛坯"

    # 精装类（豪装/豪华装修并入精装）
    if ("精装" in s) or ("精装修" in s) or ("豪装" in s) or ("豪华" in s):
        return "精装"

    # 简装类（普通装修/中装并入简装）
    if ("简装" in s) or ("简装修" in s) or ("普通装修" in s) or ("中装" in s) or ("中等装修" in s):
        return "简装"

    # 其他
    if "其他" in s or "未知" in s:
        return "其他"

    # 无法识别 -> 其他
    return "其他"

dec_norm = df["装修情况"].apply(normalize_decoration)
dec_categories = ["精装", "简装", "毛坯", "其他"]
for cat in dec_categories:
    df[f"装修_{cat}_01"] = (dec_norm == cat).astype("int8")

# -------------------------
# 4) 删除原始三列
# -------------------------
df = df.drop(columns=[c for c in ["房屋朝向", "建筑结构", "装修情况"] if c in df.columns])

# -------------------------
# 5) 覆盖保存到原文件（使用原始编码）
# -------------------------
df.to_csv(path, index=False, encoding=enc or "utf-8-sig")

print("已完成处理：")
print("- 新增朝向四个01变量：['朝向_东_01','朝向_南_01','朝向_西_01','朝向_北_01']")
print("- 新增建筑结构六个01变量：", [f"建筑结构_{c}_01" for c in struct_categories])
print("- 新增装修四个01变量：", [f"装修_{c}_01" for c in dec_categories])
print("- 已删除原列：'房屋朝向', '建筑结构', '装修情况'")
print(f"文件已覆盖保存：{path}（编码：{enc}）")

已完成处理：
- 新增朝向四个01变量：['朝向_东_01','朝向_南_01','朝向_西_01','朝向_北_01']
- 新增建筑结构六个01变量： ['建筑结构_钢混结构_01', '建筑结构_混合结构_01', '建筑结构_未知结构_01', '建筑结构_框架结构_01', '建筑结构_钢结构_01', '建筑结构_砖混结构_01']
- 新增装修四个01变量： ['装修_精装_01', '装修_简装_01', '装修_毛坯_01', '装修_其他_01']
- 已删除原列：'房屋朝向', '建筑结构', '装修情况'
文件已覆盖保存：H:\HW\ruc_Class25Q2_train_price_clean1.csv（编码：utf-8-sig）


In [17]:
import pandas as pd
import numpy as np
import re
import os
import stat
import tempfile
from pathlib import Path

# ========= 配置 =========
path = r"H:\HW\ruc_Class25Q2_train_price_clean1.csv"
# ========================

print("=" * 80)
print("交易时间 + 房龄处理（训练集统计量填充）")
print("=" * 80)

# 1. 读取CSV
print("\n[1/6] 读取数据")
enc = None
for e in ["utf-8-sig", "gbk", "utf-8"]:
    try:
        df = pd.read_csv(path, encoding=e)
        enc = e
        break
    except Exception:
        pass
if enc is None:
    raise RuntimeError("无法读取CSV")

print(f"  编码: {enc}")
print(f"  总样本: {len(df)}")

# 校验必要列
need_cols = ["交易时间", "建筑年代", "is_train"]
missing_cols = [c for c in need_cols if c not in df.columns]
if missing_cols:
    raise KeyError(f"缺少必要列：{missing_cols}")

train_count = (df['is_train']==1).sum()
test_count = (df['is_train']==0).sum()
print(f"  训练集: {train_count}")
print(f"  测试集: {test_count}")


# 2. 处理交易时间（整个数据集）
print("\n[2/6] 处理交易时间（保留年月 YYYY/M）")
ts = pd.to_datetime(df["交易时间"], errors="coerce")
trade_year = ts.dt.year
trade_month = ts.dt.month
trade_ym = pd.Series(
    np.where(ts.notna(), trade_year.astype(str) + "/" + trade_month.astype(str), np.nan),
    index=df.index
)
df["交易时间"] = trade_ym

missing_time = ts.isna().sum()
print(f"  有效交易时间: {ts.notna().sum()} / {len(df)}")
if missing_time > 0:
    print(f"  ⚠️ 缺失: {missing_time}")


# 3. 解析建筑年代（整个数据集）
print("\n[3/6] 解析建筑年代（取最后年份）")

def parse_built_year_last(s):
    if pd.isna(s):
        return np.nan
    txt = str(s)
    txt = txt.replace("年", "")
    txt = re.sub(r"\s+", "", txt)
    txt = re.sub(r"[—–－至到~～]", "-", txt)
    m = re.search(r"(\d{4})(?:\D+(\d{4}))?", txt)
    if not m:
        return np.nan
    y1 = m.group(1)
    y2 = m.group(2)
    try:
        if y2:
            return float(y2)
        return float(y1)
    except Exception:
        return np.nan

built_year_last = df["建筑年代"].apply(parse_built_year_last)

built_missing = built_year_last.isna().sum()
print(f"  成功解析: {built_year_last.notna().sum()} / {len(df)}")
if built_missing > 0:
    print(f"  ⚠️ 缺失: {built_missing}")


# 4. 计算房龄（整个数据集）
print("\n[4/6] 计算房龄（交易年 - 建造年）")
age = pd.Series(np.nan, index=df.index, dtype="float64")
valid_mask = trade_year.notna() & built_year_last.notna()
age.loc[valid_mask] = (
    trade_year[valid_mask].astype(float) - built_year_last[valid_mask]
)
age = age.clip(lower=0)  # 负值裁剪为0

print(f"  成功计算: {age.notna().sum()} / {len(df)}")
print(f"  房龄范围: min={age.min():.1f}, max={age.max():.1f}, mean={age.mean():.1f}")


# 5. 分离训练集和测试集处理
print("\n[5/6] 分离训练/测试集处理")

# 分离
df_train = df[df['is_train'] == 1].copy()
df_test = df[df['is_train'] == 0].copy()

age_train = age[df['is_train'] == 1].copy()
age_test = age[df['is_train'] == 0].copy()

built_missing_train = built_year_last[df['is_train'] == 1].isna()
built_missing_test = built_year_last[df['is_train'] == 0].isna()

# 训练集：Winsorize + 计算中位数
print("  【训练集】Winsorize + 计算中位数")

def winsorize_series(s, lower=0.01, upper=0.01):
    s = pd.to_numeric(s, errors="coerce")
    mask = s.notna()
    if mask.sum() == 0:
        return s, np.nan, np.nan
    q_low = s[mask].quantile(lower)
    q_high = s[mask].quantile(1 - upper)
    s_clip = s.clip(q_low, q_high)
    return s_clip, q_low, q_high

age_train_wins, q_low, q_high = winsorize_series(age_train, lower=0.01, upper=0.01)
median_age_train = age_train_wins.median(skipna=True)

print(f"    截尾范围: [{q_low:.1f}, {q_high:.1f}]")
print(f"    中位数: {median_age_train:.1f}")

# 训练集：用中位数填充建筑年代缺失的房龄
age_train_final = age_train_wins.copy()
train_fill_count = (built_missing_train & age_train_final.isna()).sum()
if not np.isnan(median_age_train):
    age_train_final.loc[built_missing_train & age_train_final.isna()] = median_age_train
print(f"    填充缺失: {train_fill_count} 个（建筑年代缺失）")

df_train["房龄"] = age_train_final.astype("float32")

# 测试集：不Winsorize，直接用训练集中位数填充建筑年代缺失的房龄
print("  【测试集】用训练集中位数填充")
age_test_final = age_test.copy()
test_fill_count = (built_missing_test & age_test_final.isna()).sum()
if not np.isnan(median_age_train):
    age_test_final.loc[built_missing_test & age_test_final.isna()] = median_age_train
print(f"    填充缺失: {test_fill_count} 个（建筑年代缺失）")

df_test["房龄"] = age_test_final.astype("float32")


# 6. 合并并保存
print("\n[6/6] 合并并保存")
df_final = pd.concat([df_train, df_test], axis=0, ignore_index=True)

# 删除建筑年代
df_final = df_final.drop(columns=["建筑年代"])

# 保存
def overwrite_csv_atomic(df_, target_path, encoding="utf-8-sig"):
    p = Path(target_path)
    try:
        os.chmod(p, stat.S_IWRITE)
    except Exception:
        pass
    tmp = None
    try:
        with tempfile.NamedTemporaryFile(mode="w", delete=False, dir=str(p.parent), 
                                         suffix=".tmp", encoding=encoding, newline="") as f:
            tmp = Path(f.name)
            df_.to_csv(f, index=False)
        os.replace(tmp, p)
    finally:
        try:
            if tmp and tmp.exists():
                tmp.unlink()
        except Exception:
            pass

overwrite_csv_atomic(df_final, path, encoding=enc or "utf-8-sig")

# 统计输出
print("=" * 80)
print("处理完成")
print("=" * 80)
print(f"文件: {path}")
print(f"编码: {enc}")

print(f"\n训练集统计量:")
print(f"  房龄中位数（Winsorize后）: {median_age_train:.1f}")
print(f"  截尾范围: [{q_low:.1f}, {q_high:.1f}]")

print(f"\n填充情况:")
print(f"  训练集填充: {train_fill_count} 个（建筑年代缺失）")
print(f"  测试集填充: {test_fill_count} 个（建筑年代缺失）")

print(f"\n房龄统计（合并后）:")
print(df_final["房龄"].describe())

print(f"\n按训练/测试集分组:")
print("\n训练集（已Winsorize）:")
print(df_final[df_final['is_train']==1]["房龄"].describe())

print("\n测试集（未Winsorize）:")
print(df_final[df_final['is_train']==0]["房龄"].describe())

print(f"\n已删除列: ['建筑年代']")
print(f"交易时间已转换为: YYYY/M 格式")
print("=" * 80)

交易时间 + 房龄处理（训练集统计量填充）

[1/6] 读取数据
  编码: utf-8-sig
  总样本: 103871
  训练集: 83096
  测试集: 20775

[2/6] 处理交易时间（保留年月 YYYY/M）
  有效交易时间: 103871 / 103871

[3/6] 解析建筑年代（取最后年份）
  成功解析: 68770 / 103871
  ⚠️ 缺失: 35101

[4/6] 计算房龄（交易年 - 建造年）
  成功计算: 68770 / 103871
  房龄范围: min=0.0, max=89.0, mean=13.1

[5/6] 分离训练/测试集处理
  【训练集】Winsorize + 计算中位数
    截尾范围: [2.0, 35.0]
    中位数: 11.0
    填充缺失: 27992 个（建筑年代缺失）
  【测试集】用训练集中位数填充
    填充缺失: 7109 个（建筑年代缺失）

[6/6] 合并并保存
处理完成
文件: H:\HW\ruc_Class25Q2_train_price_clean1.csv
编码: utf-8-sig

训练集统计量:
  房龄中位数（Winsorize后）: 11.0
  截尾范围: [2.0, 35.0]

填充情况:
  训练集填充: 27992 个（建筑年代缺失）
  测试集填充: 7109 个（建筑年代缺失）

房龄统计（合并后）:
count    103871.000000
mean         12.379220
std           6.358341
min           0.000000
25%           9.000000
50%          11.000000
75%          14.000000
max          50.000000
Name: 房龄, dtype: float64

按训练/测试集分组:

训练集（已Winsorize）:
count    83096.000000
mean        12.380211
std          6.339230
min          2.000000
25%          9.000000
50%         11.00

In [18]:
import pandas as pd
import re

# ========= 配置 =========
path = r"H:\HW\ruc_Class25Q2_train_price_clean1.csv"
# ========================

# 读取CSV
enc = None
for e in ["utf-8-sig", "gbk", "utf-8"]:
    try:
        df = pd.read_csv(path, encoding=e)
        enc = e
        break
    except Exception:
        pass
if enc is None:
    raise RuntimeError("无法读取CSV")

# 英文月份映射
MONTH_MAP = {
    'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
    'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
}

def parse_trade_time(s):
    """转换 Jul-24 → 2024/7"""
    if pd.isna(s):
        return s
    txt = str(s).strip()
    
    # 格式1: Jul-24
    m = re.match(r'([a-zA-Z]{3})-(\d{2})', txt, re.IGNORECASE)
    if m:
        month_abbr = m.group(1).lower()
        year_2digit = int(m.group(2))
        if month_abbr in MONTH_MAP:
            month = MONTH_MAP[month_abbr]
            year = 2000 + year_2digit if year_2digit <= 50 else 1900 + year_2digit
            return f"{year}/{month}"
    
    # 格式2: 标准日期
    try:
        dt = pd.to_datetime(txt, errors='coerce')
        if pd.notna(dt):
            return f"{dt.year}/{dt.month}"
    except:
        pass
    
    return s  # 保持原值

# 转换交易时间
df["交易时间"] = df["交易时间"].apply(parse_trade_time)

# 保存
df.to_csv(path, index=False, encoding=enc or "utf-8-sig")

print(f"✓ 交易时间已转换为 YYYY/M 格式")
print(f"文件: {path}")

✓ 交易时间已转换为 YYYY/M 格式
文件: H:\HW\ruc_Class25Q2_train_price_clean1.csv


In [19]:
import pandas as pd
import numpy as np
import re
import os
import stat
import time
import tempfile
from pathlib import Path

# ========= 配置 =========
path = r"H:\HW\ruc_Class25Q2_train_price_clean1.csv"
lower_pct = 0.01
upper_pct = 0.01
target_cols = ["房屋总数", "楼栋总数"]
# ========================

print("=" * 60)
print("房屋总数 + 楼栋总数处理（训练集统计量填充）")
print("=" * 60)

# 1. 读取CSV
enc = None
for e in ["utf-8-sig", "gbk", "utf-8"]:
    try:
        df = pd.read_csv(path, encoding=e)
        enc = e
        break
    except Exception:
        pass
if enc is None:
    raise RuntimeError("无法读取CSV")

# 校验必要列
need_cols = target_cols + ["is_train"]
missing = [c for c in need_cols if c not in df.columns]
if missing:
    raise KeyError(f"缺少必要列：{missing}")

print(f"总样本: {len(df)} (训练:{(df['is_train']==1).sum()}, 测试:{(df['is_train']==0).sum()})")


# 2. 工具函数
def extract_first_int(x):
    """提取首个连续整数"""
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    full = "０１２３４５６７８９，,"
    half = "0123456789,,"
    s = s.translate(str.maketrans(full, half))
    s = s.replace(",", "")
    if s.isdigit():
        try:
            return float(int(s))
        except:
            return np.nan
    m = re.search(r"(\d+)", s)
    if not m:
        return np.nan
    try:
        return float(int(m.group(1)))
    except:
        return np.nan

def winsorize_series(s, lower=0.01, upper=0.01):
    """Winsorize截尾"""
    s = pd.to_numeric(s, errors="coerce")
    mask = s.notna()
    if mask.sum() == 0:
        return s, np.nan, np.nan
    q_low = s[mask].quantile(lower)
    q_high = s[mask].quantile(1 - upper)
    s_clip = s.clip(q_low, q_high)
    return s_clip, q_low, q_high


# 3. 逐列处理
for col in target_cols:
    print(f"\n处理列: {col}")
    print("-" * 60)
    
    # 提取数字（整个数据集）
    raw = df[col]
    num = raw.apply(extract_first_int)
    print(f"  提取数字: {num.notna().sum()}/{len(df)} 成功")
    
    # 分离训练集和测试集
    num_train = num[df['is_train'] == 1].copy()
    num_test = num[df['is_train'] == 0].copy()
    
    # 训练集：Winsorize + 计算中位数
    num_train_wins, q_low, q_high = winsorize_series(num_train, lower_pct, upper_pct)
    median_train = num_train_wins.median(skipna=True)
    
    print(f"  【训练集】")
    print(f"    Winsorize: [{q_low:.1f}, {q_high:.1f}]")
    print(f"    中位数: {median_train:.1f}")
    
    # 训练集填充
    num_train_final = num_train_wins.fillna(median_train) if not np.isnan(median_train) else num_train_wins
    train_filled = num_train_wins.isna().sum()
    print(f"    填充: {train_filled} 个缺失")
    
    # 测试集：不Winsorize，直接用训练集中位数填充
    num_test_final = num_test.fillna(median_train) if not np.isnan(median_train) else num_test
    test_filled = num_test.isna().sum()
    print(f"  【测试集】")
    print(f"    填充: {test_filled} 个缺失（用训练集中位数）")
    
    # 合并并转换为非负整数
    num_final = pd.concat([num_train_final, num_test_final], axis=0)
    num_final = num_final.reindex(df.index)  # 恢复原始顺序
    num_final = num_final.clip(lower=0).round().astype("Int64")
    
    df[col] = num_final


# 4. 保存
def overwrite_csv_atomic(df_, target_path, encoding="utf-8-sig", retries=3, wait=1.5):
    p = Path(target_path)
    try:
        os.chmod(p, stat.S_IWRITE)
    except:
        pass
    
    last_err = None
    for i in range(retries):
        tmp = None
        try:
            with tempfile.NamedTemporaryFile(
                mode="w", delete=False, dir=str(p.parent), suffix=".tmp",
                encoding=encoding, newline=""
            ) as f:
                tmp = Path(f.name)
                df_.to_csv(f, index=False)
            os.replace(tmp, p)
            print(f"\n✓ 已保存: {p.resolve()}")
            return True
        except PermissionError as e:
            last_err = e
            try:
                if tmp and tmp.exists():
                    tmp.unlink()
            except:
                pass
            if i < retries - 1:
                print(f"文件被占用，{wait}s后重试 {i+1}/{retries}...")
                time.sleep(wait)
        except Exception as e:
            raise
    
    # 另存为
    alt = p.with_name(p.stem + "_processed.csv")
    df_.to_csv(alt, index=False, encoding=encoding)
    print(f"\n⚠️ 已另存为: {alt.resolve()}")
    return False

overwrite_csv_atomic(df, path, encoding=enc or "utf-8-sig")

print("=" * 60)
print("✓ 处理完成")
print("=" * 60)

房屋总数 + 楼栋总数处理（训练集统计量填充）
总样本: 103871 (训练:83096, 测试:20775)

处理列: 房屋总数
------------------------------------------------------------
  提取数字: 96740/103871 成功
  【训练集】
    Winsorize: [15.0, 9342.0]
    中位数: 1371.0
    填充: 5677 个缺失
  【测试集】
    填充: 1454 个缺失（用训练集中位数）

处理列: 楼栋总数
------------------------------------------------------------
  提取数字: 96740/103871 成功
  【训练集】
    Winsorize: [1.0, 321.0]
    中位数: 15.0
    填充: 5677 个缺失
  【测试集】
    填充: 1454 个缺失（用训练集中位数）

✓ 已保存: H:\HW\ruc_Class25Q2_train_price_clean1.csv
✓ 处理完成


In [20]:
import pandas as pd
import numpy as np
import re
import os
import stat
import time
import tempfile
from pathlib import Path

# ========= 配置 =========
path = r"H:\HW\ruc_Class25Q2_train_price_clean1.csv"
lower_pct = 0.01
upper_pct = 0.01
target_names = {"绿化率", "容积率"}
# ========================

print("=" * 60)
print("绿化率 + 容积率处理（训练集统计量填充）")
print("=" * 60)

# 1. 读取CSV
enc = None
for e in ["utf-8-sig", "gbk", "utf-8"]:
    try:
        df = pd.read_csv(path, encoding=e)
        enc = e
        break
    except Exception:
        pass
if enc is None:
    raise RuntimeError("无法读取CSV")

# 列名映射（去空格）
def nospace(s: str) -> str:
    return "".join(str(s).split())

col_map = {nospace(c): c for c in df.columns}
missing = [name for name in target_names if name not in col_map]
if missing:
    raise KeyError(f"缺少必要列：{missing}")

# 校验is_train
if "is_train" not in df.columns:
    raise KeyError("缺少必要列：is_train")

col_green = col_map["绿化率"]
col_far = col_map["容积率"]

print(f"总样本: {len(df)} (训练:{(df['is_train']==1).sum()}, 测试:{(df['is_train']==0).sum()})")


# 2. 工具函数
def normalize_str(x):
    if pd.isna(x):
        return ""
    s = str(x).strip()
    full = "０１２３４５６７８９％，,"
    half = "0123456789%,,"
    s = s.translate(str.maketrans(full, half))
    s = s.replace(",", "")
    s = re.sub(r"\s+", "", s)
    s = s.replace("—", "-").replace("－", "-").replace("–", "-").replace("~", "-").replace("～", "-")
    return s

def extract_first_float(x):
    s = normalize_str(x)
    m = re.search(r"(\d+(?:\.\d+)?)", s)
    if not m:
        return np.nan
    try:
        return float(m.group(1))
    except:
        return np.nan

def winsorize_series(s, lower=0.01, upper=0.01):
    s = pd.to_numeric(s, errors="coerce")
    mask = s.notna()
    if mask.sum() == 0:
        return s, np.nan, np.nan
    q_low = s[mask].quantile(lower)
    q_high = s[mask].quantile(1 - upper)
    s_clip = s.clip(q_low, q_high)
    return s_clip, q_low, q_high


# 3. 处理函数（分离训练集和测试集）
def process_column_split(series, col_name, is_train_mask, lower=0.01, upper=0.01):
    print(f"\n处理列: {col_name}")
    print("-" * 60)
    
    # 提取数字（整个数据集）
    num = series.apply(extract_first_float)
    print(f"  提取数字: {num.notna().sum()}/{len(series)} 成功")
    
    # 分离训练集和测试集
    num_train = num[is_train_mask].copy()
    num_test = num[~is_train_mask].copy()
    
    # 训练集：Winsorize + 计算中位数
    num_train_wins, q_low, q_high = winsorize_series(num_train, lower, upper)
    median_train = num_train_wins.median(skipna=True)
    
    print(f"  【训练集】")
    print(f"    Winsorize: [{q_low:.2f}, {q_high:.2f}]")
    print(f"    中位数: {median_train:.2f}")
    
    # 训练集填充
    num_train_final = num_train_wins.fillna(median_train) if not np.isnan(median_train) else num_train_wins
    train_filled = num_train_wins.isna().sum()
    print(f"    填充: {train_filled} 个缺失")
    
    # 测试集：不Winsorize，直接用训练集中位数填充
    num_test_final = num_test.fillna(median_train) if not np.isnan(median_train) else num_test
    test_filled = num_test.isna().sum()
    print(f"  【测试集】")
    print(f"    填充: {test_filled} 个缺失（用训练集中位数）")
    
    # 合并
    num_final = pd.concat([num_train_final, num_test_final], axis=0)
    num_final = num_final.reindex(series.index)  # 恢复原始顺序
    
    return num_final.astype("float32"), median_train


# 4. 处理绿化率和容积率
is_train = (df['is_train'] == 1)

df[col_green], med_green = process_column_split(
    df[col_green], "绿化率", is_train, lower_pct, upper_pct
)

df[col_far], med_far = process_column_split(
    df[col_far], "容积率", is_train, lower_pct, upper_pct
)


# 5. 保存
def overwrite_csv_atomic(df_, target_path, encoding="utf-8-sig", retries=3, wait=1.5):
    p = Path(target_path)
    try:
        os.chmod(p, stat.S_IWRITE)
    except:
        pass
    
    last_err = None
    for i in range(retries):
        tmp = None
        try:
            with tempfile.NamedTemporaryFile(
                mode="w", delete=False, dir=str(p.parent), suffix=".tmp",
                encoding=encoding, newline=""
            ) as f:
                tmp = Path(f.name)
                df_.to_csv(f, index=False)
            os.replace(tmp, p)
            print(f"\n✓ 已保存: {p.resolve()}")
            return True
        except PermissionError as e:
            last_err = e
            try:
                if tmp and tmp.exists():
                    tmp.unlink()
            except:
                pass
            if i < retries - 1:
                print(f"文件被占用，{wait}s后重试 {i+1}/{retries}...")
                time.sleep(wait)
        except Exception as e:
            raise
    
    # 另存为
    alt = p.with_name(p.stem + "_processed.csv")
    df_.to_csv(alt, index=False, encoding=encoding)
    print(f"\n⚠️ 已另存为: {alt.resolve()}")
    return False

overwrite_csv_atomic(df, path, encoding=enc or "utf-8-sig")

print("=" * 60)
print("✓ 处理完成")
print("=" * 60)

绿化率 + 容积率处理（训练集统计量填充）
总样本: 103871 (训练:83096, 测试:20775)

处理列: 绿化率
------------------------------------------------------------
  提取数字: 70988/103871 成功
  【训练集】
    Winsorize: [3.50, 65.00]
    中位数: 34.00
    填充: 26280 个缺失
  【测试集】
    填充: 6603 个缺失（用训练集中位数）

处理列: 容积率
------------------------------------------------------------
  提取数字: 70717/103871 成功
  【训练集】
    Winsorize: [0.67, 10.00]
    中位数: 2.50
    填充: 26473 个缺失
  【测试集】
    填充: 6681 个缺失（用训练集中位数）

✓ 已保存: H:\HW\ruc_Class25Q2_train_price_clean1.csv
✓ 处理完成


In [21]:

import pandas as pd
import numpy as np
import re
import os
import stat
import time
import tempfile
from pathlib import Path

# ========= 配置 =========
path = r"H:\HW\ruc_Class25Q2_train_price_clean1.csv"
lower_pct = 0.01
upper_pct = 0.01
target_name = "物业费"
force_csv_two_decimals = False
# ========================

print("=" * 60)
print("物业费处理（训练集统计量填充）")
print("=" * 60)

# 1. 读取CSV
enc = None
for e in ["utf-8-sig", "gbk", "utf-8"]:
    try:
        df = pd.read_csv(path, encoding=e)
        enc = e
        break
    except Exception:
        pass
if enc is None:
    raise RuntimeError("无法读取CSV")

# 列名映射（去空格）
def nospace(s: str) -> str:
    return "".join(str(s).split())

col_map = {nospace(c): c for c in df.columns}
if target_name not in col_map:
    raise KeyError(f"缺少必要列：{target_name}")

col_fee = col_map[target_name]

# 校验is_train
if "is_train" not in df.columns:
    raise KeyError("缺少必要列：is_train")

print(f"总样本: {len(df)} (训练:{(df['is_train']==1).sum()}, 测试:{(df['is_train']==0).sum()})")


# 2. 工具函数
def normalize_str(x):
    if pd.isna(x):
        return ""
    s = str(x).strip()
    trans = str.maketrans({
        "０":"0","１":"1","２":"2","３":"3","４":"4",
        "５":"5","６":"6","７":"7","８":"8","９":"9",
        "．":".","，":",","％":"%","／":"/","￥":"¥",
        "－":"-","—":"-","–":"-",
        "～":"~","〜":"~",
    })
    s = s.translate(trans)
    s = s.replace("至", "-").replace("到", "-")
    s = re.sub(r"\s+", "", s)
    s = s.replace(",", "")
    return s

def parse_property_fee(v):
    if pd.isna(v):
        return np.nan
    s = normalize_str(v)
    if s == "" or s.lower() in {"无", "暂无", "待定", "na", "none", "nan"}:
        return np.nan
    
    # 去单位
    s = re.sub(r"(元|￥|rmb|cny)", "", s, flags=re.I)
    s = re.sub(r"(每?\s*月)", "", s, flags=re.I)
    s = re.sub(r"(㎡|m2|m²|平米|平方米)", "", s, flags=re.I)
    s = s.replace("/", "").replace("·", "")
    
    # 区间识别
    m_range = re.search(r"(\d+(?:\.\d+)?)\s*[-~]\s*(\d+(?:\.\d+)?)", s)
    if m_range:
        a = float(m_range.group(1))
        b = float(m_range.group(2))
        return (a + b) / 2.0
    
    # 取首个浮点数
    m = re.search(r"(\d+(?:\.\d+)?)", s)
    return float(m.group(1)) if m else np.nan

def winsorize_series(s, lower=0.01, upper=0.01):
    s = pd.to_numeric(s, errors="coerce")
    mask = s.notna()
    if mask.sum() == 0:
        return s, np.nan, np.nan
    q_low = s[mask].quantile(lower)
    q_high = s[mask].quantile(1 - upper)
    s_clip = s.clip(q_low, q_high)
    return s_clip, q_low, q_high


# 3. 处理物业费
print(f"\n处理列: {target_name}")
print("-" * 60)

# 提取数值（整个数据集）
fee_num = df[col_fee].apply(parse_property_fee)
print(f"  提取数值: {fee_num.notna().sum()}/{len(df)} 成功")

# 分离训练集和测试集
is_train = (df['is_train'] == 1)
fee_train = fee_num[is_train].copy()
fee_test = fee_num[~is_train].copy()

# 训练集：Winsorize + 计算中位数
fee_train_wins, q_low, q_high = winsorize_series(fee_train, lower_pct, upper_pct)
median_train = fee_train_wins.median(skipna=True)

print(f"  【训练集】")
print(f"    Winsorize: [{q_low:.2f}, {q_high:.2f}]")
print(f"    中位数: {median_train:.2f}")

# 训练集填充
fee_train_final = fee_train_wins.fillna(median_train) if not np.isnan(median_train) else fee_train_wins
train_filled = fee_train_wins.isna().sum()
print(f"    填充: {train_filled} 个缺失")

# 测试集：不Winsorize，直接用训练集中位数填充
fee_test_final = fee_test.fillna(median_train) if not np.isnan(median_train) else fee_test
test_filled = fee_test.isna().sum()
print(f"  【测试集】")
print(f"    填充: {test_filled} 个缺失（用训练集中位数）")

# 合并
fee_final = pd.concat([fee_train_final, fee_test_final], axis=0)
fee_final = fee_final.reindex(df.index)  # 恢复原始顺序

# 非负化 + 保留两位小数
fee_final = fee_final.clip(lower=0).round(2).astype("float32")

df[col_fee] = fee_final


# 4. 保存
def overwrite_csv_atomic(df_, target_path, encoding="utf-8-sig", retries=3, wait=1.5, float_fmt=None):
    p = Path(target_path)
    try:
        os.chmod(p, stat.S_IWRITE)
    except:
        pass
    
    last_err = None
    for i in range(retries):
        tmp = None
        try:
            with tempfile.NamedTemporaryFile(
                mode="w", delete=False, dir=str(p.parent), suffix=".tmp",
                encoding=encoding, newline=""
            ) as f:
                tmp = Path(f.name)
                if float_fmt:
                    df_.to_csv(f, index=False, float_format=float_fmt)
                else:
                    df_.to_csv(f, index=False)
            os.replace(tmp, p)
            print(f"\n✓ 已保存: {p.resolve()}")
            return True
        except PermissionError as e:
            last_err = e
            try:
                if tmp and tmp.exists():
                    tmp.unlink()
            except:
                pass
            if i < retries - 1:
                print(f"文件被占用，{wait}s后重试 {i+1}/{retries}...")
                time.sleep(wait)
        except Exception as e:
            raise
    
    # 另存为
    alt = p.with_name(p.stem + "_processed.csv")
    if float_fmt:
        df_.to_csv(alt, index=False, encoding=encoding, float_format=float_fmt)
    else:
        df_.to_csv(alt, index=False, encoding=encoding)
    print(f"\n⚠️ 已另存为: {alt.resolve()}")
    return False

overwrite_csv_atomic(
    df, path, encoding=enc or "utf-8-sig",
    float_fmt="%.2f" if force_csv_two_decimals else None
)

print("=" * 60)
print("✓ 处理完成")
print("=" * 60)

# ---------- 可选：快速自测 ----------
# s = pd.Series(["0.5-2元/月/㎡","1.4元/月/㎡","2 元 / m2 / 月","2.5~3 元/平米/月","待定","—","￥1.80/m²·月"])
# print(s.apply(parse_property_fee).tolist())


物业费处理（训练集统计量填充）
总样本: 103871 (训练:83096, 测试:20775)

处理列: 物业费
------------------------------------------------------------
  提取数值: 72713/103871 成功
  【训练集】
    Winsorize: [0.45, 16.01]
    中位数: 1.90
    填充: 24870 个缺失
  【测试集】
    填充: 6288 个缺失（用训练集中位数）

✓ 已保存: H:\HW\ruc_Class25Q2_train_price_clean1.csv
✓ 处理完成


In [22]:
import pandas as pd

path = r"H:\HW\ruc_Class25Q2_train_price_clean1.csv"  # 改成你的CSV路径

# 读取（尝试常见编码）
enc = None
for e in ["utf-8-sig", "gbk", "utf-8"]:
    try:
        df = pd.read_csv(path, encoding=e)
        enc = e
        break
    except Exception:
        pass
if enc is None:
    raise RuntimeError("无法读取CSV，请检查路径或编码")

# 删除“停车费”（按去空格匹配）
key = "停车费用"
col_map = {"".join(str(c).split()): c for c in df.columns}
if key in col_map:
    df.drop(columns=[col_map[key]], inplace=True)
    print(f"已删除列：{col_map[key]}")
else:
    print("未找到列：停车费用")

# 保存
df.to_csv(path, index=False, encoding=enc or "utf-8-sig")
print("已保存")

已删除列：停车费用
已保存


In [23]:

import pandas as pd
import numpy as np
import re
from pathlib import Path

# ========= 配置区域 =========
# 输入文件（你的 clean1）
path_in = r"H:\HW\ruc_Class25Q2_train_price_clean1.csv"

# 输出文件：同目录生成 clean2（若文件名包含 clean1，则替换为 clean2，否则追加 _clean2 后缀）
p_in = Path(path_in)
stem = p_in.stem
if re.search(r"clean1", stem, flags=re.I):
    stem_out = re.sub(r"clean1", "clean2", stem, flags=re.I)
else:
    stem_out = stem + "_clean2"
path_out = str(p_in.with_name(stem_out + p_in.suffix))

# —— 城市编码与基准选择（线性模型推荐删除一个基准列以避免完全共线性）——
city_col_exact = "城市"   # 城市编码列（0–11）
n_cities = 12            # 编码范围 0..11
prefer_keep_00 = True    # 优先保留 city_00：若 00 是最高频，则用次高频作为基准
fallback_baseline_idx = 11  # 若数据异常（全缺失或只有 00），回退基准
include_nan_dummy = False   # 是否为缺失编码生成 city_nan 列
drop_original_city_code = True  # 建议 True：删除原始“城市”列，避免误用
# ===========================

# 读取（尝试常见编码）
enc = None
for e in ["utf-8-sig", "gbk", "utf-8"]:
    try:
        df = pd.read_csv(path_in, encoding=e)
        enc = e
        break
    except Exception:
        pass
if enc is None:
    raise RuntimeError("无法读取CSV，请检查路径或编码。")

# 校验城市列是否存在
if city_col_exact not in df.columns:
    raise KeyError(f"未找到列：{city_col_exact}。现有列示例：{list(df.columns)[:10]}")

# 将“城市”清洗为 0..(n_cities-1) 的可空整数，越界/无法解析 -> 缺失
codes = pd.to_numeric(df[city_col_exact], errors="coerce").astype("Int16")
codes = codes.where((codes >= 0) & (codes < n_cities), pd.NA)

# 计算各城市频次（忽略缺失），按 频次降序、编码升序 排序
vc = codes.value_counts(dropna=True).astype(int)
freq_df = (
    vc.rename_axis("city").reset_index(name="cnt")
    .sort_values(["cnt", "city"], ascending=[False, True])
)

# 自动选择基准
if not freq_df.empty:
    baseline_idx = int(freq_df.iloc[0]["city"])
    # 若优先保留 00 且最高频为 00，则取次高频；如果没有次高频就回退
    if prefer_keep_00 and baseline_idx == 0:
        if len(freq_df) >= 2:
            baseline_idx = int(freq_df.iloc[1]["city"])
        else:
            baseline_idx = fallback_baseline_idx
else:
    baseline_idx = fallback_baseline_idx

# 兜底：确保基准在合法范围
if not (0 <= baseline_idx < n_cities):
    baseline_idx = min(max(baseline_idx, 0), n_cities - 1)

# 构造完整类别并 One-Hot（保证列齐全）
categories = list(range(n_cities))
cat = pd.Categorical(codes, categories=categories, ordered=False)
dummies = pd.get_dummies(
    pd.Series(cat, name="city"),
    prefix="city", prefix_sep="_",
    dtype="uint8",
    dummy_na=include_nan_dummy
)

# 统一列名为两位零填充：city_00, city_01, ...
rename_map = {f"city_{i}": f"city_{i:02d}" for i in categories}
dummies.rename(columns=rename_map, inplace=True)

# 保证所有期望列存在且有序
expected_cols = [f"city_{i:02d}" for i in categories]
for c in expected_cols:
    if c not in dummies.columns:
        dummies[c] = np.uint8(0)
order_cols = expected_cols + (["city_nan"] if include_nan_dummy else [])
dummies = dummies[order_cols]

# 删除基准列（避免线性模型的虚拟变量陷阱）
baseline_col = f"city_{baseline_idx:02d}"
if baseline_col in dummies.columns:
    dummies.drop(columns=[baseline_col], inplace=True)
else:
    raise KeyError(f"未找到基准列 {baseline_col}，请检查 n_cities/baseline_idx 设置。")

# 合并结果
df_out = pd.concat([df, dummies], axis=1)
if drop_original_city_code:
    df_out.drop(columns=[city_col_exact], inplace=True)

# 写出 clean2
df_out.to_csv(path_out, index=False, encoding=enc or "utf-8-sig")

# 信息输出
print(f"已生成新文件：{Path(path_out).resolve()}")
print(f"选择的基准城市编码：{baseline_idx:02d}（已删除列 {baseline_col}）")
print("城市频次（前十）：")
print(freq_df.head(10).to_string(index=False))
print(f"最终 One-Hot 列数：{dummies.shape[1]}；示例列：{dummies.columns[:min(12, dummies.shape[1])].tolist()}")


已生成新文件：H:\HW\ruc_Class25Q2_train_price_clean2.csv
选择的基准城市编码：02（已删除列 city_02）
城市频次（前十）：
 city   cnt
    2 24996
    3 21472
    0 16491
   10 15057
    1  6437
    8  5931
    4  4363
    5  3582
    6  2281
    9  1323
最终 One-Hot 列数：11；示例列：['city_00', 'city_01', 'city_03', 'city_04', 'city_05', 'city_06', 'city_07', 'city_08', 'city_09', 'city_10', 'city_11']


In [24]:

import pandas as pd
from pathlib import Path
import re

# ============ 配置 ============
# 输入文件（建议是你当前的 clean2）
path_in = r"H:\HW\ruc_Class25Q2_train_price_clean2.csv"

# 输出文件：同目录生成 *_clean2_lin.csv（不覆盖你的 clean2）
p_in = Path(path_in)
stem = p_in.stem
if re.search(r"clean2", stem, flags=re.I):
    stem_out = re.sub(r"clean2", "clean2_lin", stem, flags=re.I)
else:
    stem_out = stem + "_clean2_lin"
path_out = str(p_in.with_name(stem_out + p_in.suffix))

# 若某组 ≥98% 行满足单选（行和在 {0,1}），才删一列基准；否则跳过该组
single_choice_threshold = 0.98

# “朝向”明确为多选：无论检测结果如何，都不删该组
force_keep_orientation = True

# ============ 读取 ============
enc = None
for e in ["utf-8-sig", "gbk", "utf-8"]:
    try:
        df = pd.read_csv(path_in, encoding=e)
        enc = e
        break
    except Exception:
        pass
if enc is None:
    raise RuntimeError("无法读取CSV，请检查路径或编码。")

cols = set(df.columns)

# ============ 分组规则 ============
# 1) 楼层（按你提供的列名）
floor_candidates = ["地下室_01", "底层_01", "低楼层_01", "中楼层_01", "高楼层_01", "顶层_01"]
group_floor = [c for c in floor_candidates if c in cols]

# 2) 配备电梯
group_elevator = sorted([c for c in cols if c.startswith("配备电梯_") and c.endswith("_01")])

# 3) 朝向（明确多选）
group_orientation = sorted([c for c in cols if c.startswith("朝向_") and c.endswith("_01")])

# 4) 建筑结构
group_structure = sorted([c for c in cols if c.startswith("建筑结构_") and c.endswith("_01")])

# 5) 装修
group_decor = sorted([c for c in cols if c.startswith("装修_") and c.endswith("_01")])

groups = {
    "楼层": group_floor,
    "配备电梯": group_elevator,
    "朝向": group_orientation,
    "建筑结构": group_structure,
    "装修": group_decor,
}

def is_single_choice(df, cols, threshold=single_choice_threshold):
    if len(cols) <= 1:
        return False
    s = df[cols].fillna(0).sum(axis=1)
    ratio = s.isin([0, 1]).mean()
    return ratio >= threshold

def choose_baseline_by_freq(df, cols):
    # 以“1”的数量作为出现频率，最高频作为基准
    counts = df[cols].fillna(0).sum().astype(int)
    # 若全为0或空，返回 None
    if counts.empty or counts.max() == 0:
        return None, counts.sort_values(ascending=False)
    # 最高频（若并列，由列名排序保证稳定）
    counts_sorted = counts.sort_values(ascending=False, kind="mergesort")
    baseline = counts_sorted.index[0]
    return baseline, counts_sorted

dropped = []

for gname, gcols in groups.items():
    if len(gcols) <= 1:
        continue  # 组里不足2列，跳过

    # 朝向：明确多选，强制保留
    if gname == "朝向" and force_keep_orientation:
        print(f"保留组（多选）：{gname}，列：{gcols}")
        continue

    # 其他组：仅当基本满足“单选”特征时才删除一列
    if not is_single_choice(df, gcols):
        print(f"跳过组（非单选或数据不规范）：{gname}，列：{gcols}")
        continue

    baseline, counts_sorted = choose_baseline_by_freq(df, gcols)
    if baseline is None:
        print(f"组 {gname} 未找到可用基准（可能全为0），跳过。")
        continue

    df.drop(columns=[baseline], inplace=True)
    dropped.append((gname, baseline, counts_sorted.to_dict()))

# 写出
df.to_csv(path_out, index=False, encoding=enc or "utf-8-sig")

print(f"已写出：{Path(path_out).resolve()}")
if dropped:
    print("各组删除的基准列（按最高频）：")
    for g, b, cnts in dropped:
        print(f"- 组[{g}] 删列：{b}；频次摘要：{ {k: int(v) for k, v in list(cnts.items())[:5]} } ...")
else:
    print("未删除任何列可能各组不是单选或只有1列，或列名未匹配）。")


保留组（多选）：朝向，列：['朝向_东_01', '朝向_北_01', '朝向_南_01', '朝向_西_01']
已写出：H:\HW\ruc_Class25Q2_train_price_clean2_lin.csv
各组删除的基准列（按最高频）：
- 组[楼层] 删列：中楼层_01；频次摘要：{'中楼层_01': 36251, '高楼层_01': 32132, '低楼层_01': 30668, '顶层_01': 2264, '底层_01': 1854} ...
- 组[配备电梯] 删列：配备电梯_有_01；频次摘要：{'配备电梯_有_01': 77852, '配备电梯_无_01': 26019} ...
- 组[建筑结构] 删列：建筑结构_钢混结构_01；频次摘要：{'建筑结构_钢混结构_01': 82519, '建筑结构_混合结构_01': 9601, '建筑结构_未知结构_01': 4808, '建筑结构_砖混结构_01': 4201, '建筑结构_框架结构_01': 1676} ...
- 组[装修] 删列：装修_精装_01；频次摘要：{'装修_精装_01': 46585, '装修_其他_01': 24400, '装修_简装_01': 20522, '装修_毛坯_01': 12364} ...


In [25]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# ========= 配置 =========
path_in = r"H:\HW\ruc_Class25Q2_train_price_clean2_lin.csv"
price_col_pref = "Price"
area_col_pref = "建筑面积"
winsor_q_price = (0.005, 0.995)
winsor_q_area = (0.005, 0.995)
keep_raw_copy = False
# ========================

print("=" * 60)
print("Price + 建筑面积处理（训练集统计量）")
print("=" * 60)

# 1. 读取CSV
def read_csv_smart(path):
    enc = None
    for e in ["utf-8-sig", "gbk", "utf-8"]:
        try:
            df = pd.read_csv(path, encoding=e)
            enc = e
            return df, enc
        except Exception:
            pass
    raise RuntimeError("无法读取CSV")

df, enc = read_csv_smart(path_in)

# 校验is_train
if "is_train" not in df.columns:
    raise KeyError("缺少必要列：is_train")

print(f"总样本: {len(df)} (训练:{(df['is_train']==1).sum()}, 测试:{(df['is_train']==0).sum()})")


# 2. 定位列
def find_col_case_insensitive(df, preferred, alternatives=None, startswith=None):
    lc = {c.lower(): c for c in df.columns}
    if preferred.lower() in lc:
        return lc[preferred.lower()]
    if alternatives:
        for a in alternatives:
            if a.lower() in lc:
                return lc[a.lower()]
    if startswith:
        for c in df.columns:
            if c.startswith(startswith):
                return c
    return None

price_col = find_col_case_insensitive(
    df, preferred=price_col_pref,
    alternatives=["price", "PRICE", "价格", "总价"]
)
if price_col is None:
    raise KeyError(f"未找到价格列")

area_col = find_col_case_insensitive(
    df, preferred=area_col_pref,
    alternatives=["建筑面积(㎡)", "建筑面积（㎡）"],
    startswith="建筑面积"
)
if area_col is None:
    raise KeyError(f"未找到建筑面积列")

print(f"列定位: Price -> {price_col}, 建筑面积 -> {area_col}")


# 3. 工具函数
def to_numeric_price(s):
    if pd.isna(s):
        return np.nan
    if isinstance(s, (int, float, np.number)):
        return float(s)
    s = str(s).strip().replace(",", "")
    m = re.findall(r"[-+]?\d*\.?\d+", s)
    if not m:
        return np.nan
    try:
        return float(m[0])
    except:
        return np.nan

def to_numeric_area(s):
    if pd.isna(s):
        return np.nan
    if isinstance(s, (int, float, np.number)):
        return float(s)
    t = str(s).strip().replace(",", "")
    nums = re.findall(r"[-+]?\d*\.?\d+", t)
    if not nums:
        return np.nan
    vals = []
    for n in nums:
        try:
            vals.append(float(n))
        except:
            pass
    if not vals:
        return np.nan
    if len(vals) >= 2 and any(sep in t for sep in ["-", "—", "~", "～", "至"]):
        return float(np.mean(vals[:2]))
    return float(vals[0])

def winsorize_series(x, q_low, q_high):
    x = pd.to_numeric(x, errors="coerce")
    mask = x.notna()
    if mask.sum() == 0:
        return x, np.nan, np.nan
    lo = x[mask].quantile(q_low)
    hi = x[mask].quantile(q_high)
    x_clip = x.clip(lower=lo, upper=hi)
    return x_clip, lo, hi


# 4. 处理 Price（只处理训练集）
print(f"\n处理列: {price_col}")
print("-" * 60)

if keep_raw_copy:
    df[f"{price_col}_raw"] = df[price_col]

# 转数字
df[price_col] = df[price_col].apply(to_numeric_price)

# 分离
is_train = (df['is_train'] == 1)
price_train = df.loc[is_train, price_col].copy()
price_test = df.loc[~is_train, price_col].copy()

# 训练集：Winsorize（不填充，保留原缺失）
price_train_wins, lo_p, hi_p = winsorize_series(price_train, *winsor_q_price)

print(f"  【训练集】")
print(f"    缺失: {price_train.isna().sum()}")
print(f"    Winsorize: [{lo_p:.2f}, {hi_p:.2f}]")
print(f"    裁剪: {((price_train != price_train_wins) & price_train.notna()).sum()} 个值")

# 测试集：不处理（应该全是NaN或者需要预测的）
test_has_price = price_test.notna().sum()
print(f"  【测试集】")
print(f"    有值样本: {test_has_price} (正常应为0)")

# 回写
df.loc[is_train, price_col] = price_train_wins
# 测试集保持原样（不变）


# 5. 处理 建筑面积（分离训练集和测试集）
print(f"\n处理列: {area_col}")
print("-" * 60)

if keep_raw_copy:
    df[f"{area_col}_raw"] = df[area_col]

# 转数字
df[area_col] = df[area_col].apply(to_numeric_area)

# 分离
area_train = df.loc[is_train, area_col].copy()
area_test = df.loc[~is_train, area_col].copy()

# 训练集：Winsorize + 中位数填充
area_train_wins, lo_a, hi_a = winsorize_series(area_train, *winsor_q_area)
median_train = area_train_wins.median(skipna=True)
area_train_final = area_train_wins.fillna(median_train) if not np.isnan(median_train) else area_train_wins

print(f"  【训练集】")
print(f"    缺失（去单位后）: {area_train.isna().sum()}")
print(f"    Winsorize: [{lo_a:.2f}, {hi_a:.2f}]")
print(f"    中位数: {median_train:.2f}")
print(f"    填充: {area_train_wins.isna().sum()} 个缺失")

# 测试集：不Winsorize，用训练集中位数填充
area_test_final = area_test.fillna(median_train) if not np.isnan(median_train) else area_test

print(f"  【测试集】")
print(f"    缺失: {area_test.isna().sum()}")
print(f"    填充: {area_test.isna().sum()} 个缺失（用训练集中位数）")

# 回写
df.loc[is_train, area_col] = area_train_final
df.loc[~is_train, area_col] = area_test_final


# 6. 保存
p_in = Path(path_in)
stem = p_in.stem
if re.search(r"clean2[_\-]?lin", stem, flags=re.I):
    stem_out = re.sub(r"clean2[_\-]?lin", "clean3", stem, flags=re.I)
elif re.search(r"clean2", stem, flags=re.I):
    stem_out = re.sub(r"clean2", "clean3", stem, flags=re.I)
else:
    stem_out = stem + "_clean3"
path_out = str(p_in.with_name(stem_out + p_in.suffix))

df.to_csv(path_out, index=False, encoding=enc or "utf-8-sig")

print(f"\n✓ 已保存: {Path(path_out).resolve()}")
print("=" * 60)
print("✓ 处理完成")
print("=" * 60)

Price + 建筑面积处理（训练集统计量）
总样本: 103871 (训练:83096, 测试:20775)
列定位: Price -> Price, 建筑面积 -> 建筑面积

处理列: Price
------------------------------------------------------------
  【训练集】
    缺失: 0
    Winsorize: [201221.66, 16830286.33]
    裁剪: 832 个值
  【测试集】
    有值样本: 20775 (正常应为0)

处理列: 建筑面积
------------------------------------------------------------
  【训练集】
    缺失（去单位后）: 0
    Winsorize: [22.88, 320.47]
    中位数: 91.16
    填充: 0 个缺失
  【测试集】
    缺失: 0
    填充: 0 个缺失（用训练集中位数）

✓ 已保存: H:\HW\ruc_Class25Q2_train_price_clean3.csv
✓ 处理完成


In [26]:
# -*- coding: utf-8 -*-
"""
智能板块聚合策略（训练-测试分离版）：
1. 训练集：
   - 大样本板块（≥min_count_big_block）：保持独立
   - 小样本板块：智能融入或聚类
2. 测试集：
   - 利用 lon/lat 找到最近的训练集样本
   - 直接复制训练集样本的聚类特征
3. 输出：聚合后簇的 One-Hot（删基准）+ 缺失标记
4. 删除：区域/板块/lon/lat 四列
"""

import pandas as pd
import numpy as np
from pathlib import Path
import warnings

# ===================== 配置区 =====================
path_in = r"H:\HW\ruc_Class25Q2_train_price_clean3.csv"

# 固定列名
REGION_COL = "区域"
BLOCK_COL  = "板块"
LON_COL    = "lon"
LAT_COL    = "lat"
IS_TRAIN_COL = "is_train"

# 大小样本阈值（仅针对训练集）
min_count_big_block = 100

# 融入判断参数
max_merge_to_big_km = 50
distance_ratio_threshold = 1.2

# 小样本聚类参数
target_small_samples_per_cluster = 200
k_small_min, k_small_max = 10, 150
random_state = 42

# 输出文件名后缀
out_suffix = "_blk_smart_features.csv"
# =================================================

print("=" * 60)
print("板块特征工程（训练集聚类 + 测试集最近邻匹配）")
print("=" * 60)


# ===================== 工具函数 =====================
def read_csv_smart(path):
    for e in ["utf-8-sig", "gbk", "utf-8"]:
        try:
            return pd.read_csv(path, encoding=e), e
        except Exception:
            pass
    raise RuntimeError("无法读取CSV")

def lonlat_to_mercator(lon_deg: np.ndarray, lat_deg: np.ndarray):
    R = 6378137.0
    MAX_LAT = 85.05112878
    lon = np.asarray(lon_deg, dtype=float)
    lat = np.asarray(lat_deg, dtype=float)
    lat = np.clip(lat, -MAX_LAT, MAX_LAT)
    lon_rad = np.deg2rad(lon)
    lat_rad = np.deg2rad(lat)
    x = R * lon_rad
    y = R * np.log(np.tan(np.pi / 4.0 + lat_rad / 2.0))
    return np.column_stack([x, y])

def haversine_km(lon1, lat1, lon2, lat2):
    R = 6371.0088
    lon1 = np.radians(lon1); lat1 = np.radians(lat1)
    lon2 = np.radians(lon2); lat2 = np.radians(lat2)
    dlon = lon2 - lon1; dlat = lat2 - lat1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    c = 2*np.arcsin(np.sqrt(a))
    return R * c

def choose_k_small(total_samples, target_per_cluster, kmin, kmax, n_blocks):
    if total_samples <= 0 or n_blocks <= 0:
        return 0
    k = int(np.ceil(total_samples / float(target_per_cluster)))
    k = max(kmin, min(k, kmax))
    k = min(k, n_blocks)
    return k

def one_hot_drop_baseline(df, col, prefix):
    """对指定列做One-Hot，删除频率最高的基准列"""
    counts = df[col].value_counts(dropna=False)
    if counts.empty:
        return df, None, []
    baseline = counts.index[0]
    dummies = pd.get_dummies(df[col], prefix=prefix, dtype=int)
    base_col = f"{prefix}_{baseline}"
    if base_col in dummies.columns:
        dummies.drop(columns=[base_col], inplace=True)
    df = pd.concat([df, dummies], axis=1)
    return df, baseline, list(dummies.columns)


# ===================== 主流程 =====================
def main():
    # 1. 读取数据
    df, enc = read_csv_smart(path_in)
    
    # 校验必需列
    for c in [BLOCK_COL, LON_COL, LAT_COL, IS_TRAIN_COL]:
        if c not in df.columns:
            raise KeyError(f"缺少必需列：{c}")
    
    # 分离训练集和测试集（保留原始索引）
    is_train_mask = (df[IS_TRAIN_COL] == 1)
    df_train = df[is_train_mask].copy()
    df_test = df[~is_train_mask].copy()
    
    print(f"总样本: {len(df)} (训练:{len(df_train)}, 测试:{len(df_test)})")
    
    
    # ==================== 训练集聚类 ====================
    print("\n【训练集】板块聚类")
    print("-" * 60)
    
    # 计算训练集板块统计
    has_coord_train = df_train[LON_COL].notna() & df_train[LAT_COL].notna()
    g_train = df_train.groupby(BLOCK_COL, dropna=False)
    blk_stats_train = g_train.agg(
        blk_count=(BLOCK_COL, "size"),
        lon_med=(LON_COL, "median"),
        lat_med=(LAT_COL, "median")
    ).reset_index()
    
    # 仅使用有中心点的板块
    valid_mask = blk_stats_train["lon_med"].notna() & blk_stats_train["lat_med"].notna()
    blk_valid = blk_stats_train[valid_mask].copy()
    
    if blk_valid.empty:
        raise RuntimeError("训练集所有板块均缺少中心点")
    
    # 划分大小样本板块
    blk_valid["is_big"] = blk_valid["blk_count"] >= min_count_big_block
    big_blocks = blk_valid[blk_valid["is_big"]].copy()
    small_blocks = blk_valid[~blk_valid["is_big"]].copy()
    
    n_big = len(big_blocks)
    n_small = len(small_blocks)
    total_small_samples = int(small_blocks["blk_count"].sum())
    
    print(f"  板块总数（有中心点）: {len(blk_valid)}")
    print(f"  大样本板块（≥{min_count_big_block}）: {n_big} 个")
    print(f"  小样本板块（<{min_count_big_block}）: {n_small} 个，共 {total_small_samples} 样本")
    
    # 初始化映射：block -> cluster_id
    block_to_cluster = {}
    next_cluster_id = 0
    
    # 大样本板块：各自独立
    for idx, row in big_blocks.iterrows():
        block_to_cluster[row[BLOCK_COL]] = next_cluster_id
        next_cluster_id += 1
    
    # 小样本板块智能分组
    if n_small > 0 and n_big > 0:
        big_lons = big_blocks["lon_med"].values
        big_lats = big_blocks["lat_med"].values
        small_lons = small_blocks["lon_med"].values
        small_lats = small_blocks["lat_med"].values
        
        # 距离矩阵
        dist_to_big = np.zeros((n_small, n_big))
        for i, (slon, slat) in enumerate(zip(small_lons, small_lats)):
            dist_to_big[i, :] = haversine_km(slon, slat, big_lons, big_lats)
        
        dist_small = np.zeros((n_small, n_small))
        for i in range(n_small):
            for j in range(n_small):
                if i != j:
                    dist_small[i, j] = haversine_km(
                        small_lons[i], small_lats[i],
                        small_lons[j], small_lats[j]
                    )
                else:
                    dist_small[i, j] = np.inf
        
        # 决策：融入大样本 vs 聚类
        small_merge_to_big = []
        small_need_cluster = []
        
        for i in range(n_small):
            d_big_min = dist_to_big[i, :].min()
            idx_big_nearest = dist_to_big[i, :].argmin()
            d_small_min = dist_small[i, :].min()
            
            if (d_big_min <= max_merge_to_big_km and 
                d_big_min < d_small_min * distance_ratio_threshold):
                small_merge_to_big.append((i, idx_big_nearest, d_big_min))
            else:
                small_need_cluster.append(i)
        
        # 执行融入
        n_merged = len(small_merge_to_big)
        for small_idx, big_idx, _ in small_merge_to_big:
            small_blk = small_blocks.iloc[small_idx][BLOCK_COL]
            big_blk = big_blocks.iloc[big_idx][BLOCK_COL]
            block_to_cluster[small_blk] = block_to_cluster[big_blk]
        
        print(f"  小样本融入大样本: {n_merged} 个板块")
        
        # 剩余小样本聚类
        n_cluster_needed = len(small_need_cluster)
        if n_cluster_needed > 0:
            cluster_samples = sum(small_blocks.iloc[i]["blk_count"] for i in small_need_cluster)
            print(f"  剩余小样本需聚类: {n_cluster_needed} 个板块，共 {cluster_samples} 样本")
            
            cluster_lons = small_blocks.iloc[small_need_cluster]["lon_med"].values
            cluster_lats = small_blocks.iloc[small_need_cluster]["lat_med"].values
            cluster_xy = lonlat_to_mercator(cluster_lons, cluster_lats)
            
            k_small = choose_k_small(
                cluster_samples, target_small_samples_per_cluster,
                k_small_min, k_small_max, n_cluster_needed
            )
            
            if k_small > 0 and k_small < n_cluster_needed:
                from sklearn.cluster import KMeans
                km = KMeans(n_clusters=k_small, random_state=random_state, n_init=10)
                small_labels = km.fit_predict(cluster_xy)
                
                for local_idx, small_idx in enumerate(small_need_cluster):
                    small_blk = small_blocks.iloc[small_idx][BLOCK_COL]
                    block_to_cluster[small_blk] = next_cluster_id + small_labels[local_idx]
                
                next_cluster_id += k_small
                print(f"  小样本聚类完成: k={k_small}")
            else:
                for small_idx in small_need_cluster:
                    small_blk = small_blocks.iloc[small_idx][BLOCK_COL]
                    block_to_cluster[small_blk] = next_cluster_id
                    next_cluster_id += 1
    
    elif n_small > 0 and n_big == 0:
        print("  无大样本板块，全部小样本聚类")
        cluster_xy = lonlat_to_mercator(
            small_blocks["lon_med"].values,
            small_blocks["lat_med"].values
        )
        k_small = choose_k_small(
            total_small_samples, target_small_samples_per_cluster,
            k_small_min, k_small_max, n_small
        )
        if k_small > 0 and k_small < n_small:
            from sklearn.cluster import KMeans
            km = KMeans(n_clusters=k_small, random_state=random_state, n_init=10)
            labels = km.fit_predict(cluster_xy)
            for i, (idx, row) in enumerate(small_blocks.iterrows()):
                block_to_cluster[row[BLOCK_COL]] = labels[i]
            next_cluster_id = k_small
        else:
            for idx, row in small_blocks.iterrows():
                block_to_cluster[row[BLOCK_COL]] = next_cluster_id
                next_cluster_id += 1
    
    total_clusters = next_cluster_id
    print(f"  训练集簇总数: {total_clusters}")
    
    
    # ==================== 训练集特征生成 ====================
    # 样本级映射
    df_train["blk_cluster"] = df_train[BLOCK_COL].map(block_to_cluster).astype("Int64")
    
    # 训练集有坐标但板块无映射的，按最近中心分配
    need_fallback_train = df_train["blk_cluster"].isna() & has_coord_train
    if need_fallback_train.any():
        cluster_centers = {}
        for blk, cid in block_to_cluster.items():
            if cid not in cluster_centers:
                cluster_centers[cid] = []
            blk_row = blk_valid[blk_valid[BLOCK_COL] == blk]
            if not blk_row.empty:
                cluster_centers[cid].append((
                    blk_row.iloc[0]["lon_med"],
                    blk_row.iloc[0]["lat_med"]
                ))
        
        cids = sorted(cluster_centers.keys())
        center_lons = np.array([np.mean([p[0] for p in cluster_centers[c]]) for c in cids])
        center_lats = np.array([np.mean([p[1] for p in cluster_centers[c]]) for c in cids])
        
        fb_lons = df_train.loc[need_fallback_train, LON_COL].values
        fb_lats = df_train.loc[need_fallback_train, LAT_COL].values
        dists = np.vstack([
            haversine_km(fb_lons, fb_lats, clon, clat)
            for clon, clat in zip(center_lons, center_lats)
        ]).T
        nearest = np.argmin(dists, axis=1)
        df_train.loc[need_fallback_train, "blk_cluster"] = [cids[i] for i in nearest]
    
    
    # ==================== 测试集最近邻匹配 ====================
    print("\n【测试集】最近邻匹配")
    print("-" * 60)
    
    # 提取训练集有坐标的样本
    train_has_coord = df_train[LON_COL].notna() & df_train[LAT_COL].notna()
    train_coord_samples = df_train[train_has_coord].copy()
    
    if len(train_coord_samples) == 0:
        raise RuntimeError("训练集无有效坐标样本")
    
    train_lons = train_coord_samples[LON_COL].values
    train_lats = train_coord_samples[LAT_COL].values
    train_clusters = train_coord_samples["blk_cluster"].values
    
    # 测试集有坐标的样本
    test_has_coord = df_test[LON_COL].notna() & df_test[LAT_COL].notna()
    test_coord_samples = df_test[test_has_coord]
    
    if len(test_coord_samples) > 0:
        test_lons = test_coord_samples[LON_COL].values
        test_lats = test_coord_samples[LAT_COL].values
        
        # 计算距离矩阵（分批处理避免内存爆炸）
        batch_size = 1000
        n_test = len(test_coord_samples)
        test_clusters = np.zeros(n_test, dtype=int)
        
        for i in range(0, n_test, batch_size):
            end_i = min(i + batch_size, n_test)
            batch_lons = test_lons[i:end_i]
            batch_lats = test_lats[i:end_i]
            
            # 距离矩阵 [batch_size, n_train]
            dists = np.zeros((end_i - i, len(train_lons)))
            for j, (tlon, tlat) in enumerate(zip(batch_lons, batch_lats)):
                dists[j, :] = haversine_km(tlon, tlat, train_lons, train_lats)
            
            # 最近训练样本
            nearest_idx = np.argmin(dists, axis=1)
            test_clusters[i:end_i] = train_clusters[nearest_idx]
        
        # 回填测试集
        df_test.loc[test_has_coord, "blk_cluster"] = test_clusters
        print(f"  匹配成功: {len(test_coord_samples)} 个测试样本")
    else:
        df_test["blk_cluster"] = pd.NA
        print(f"  测试集无有效坐标样本")
    
    # 测试集无坐标的标记为缺失
    df_test.loc[~test_has_coord, "blk_cluster"] = pd.NA
    
    
    # ==================== 合并并生成One-Hot ====================
    # 合并回原数据集（保持原顺序）
    df_combined = pd.concat([df_train, df_test], axis=0).sort_index()
    
    # 缺失标记
    df_combined["blk_missing"] = df_combined["blk_cluster"].isna().astype(int)
    
    # One-Hot（删基准）
    df_combined, baseline, ohe_cols = one_hot_drop_baseline(
        df_combined, "blk_cluster", "blk_k"
    )
    
    # 删除原始列
    drop_cols = [REGION_COL, BLOCK_COL,  "blk_cluster"]
    df_combined.drop(
        columns=[c for c in drop_cols if c in df_combined.columns],
        inplace=True, errors="ignore"
    )
    
    
    # ==================== 输出 ====================
    p_in = Path(path_in)
    path_out = str(p_in.with_name(p_in.stem + out_suffix))
    df_combined.to_csv(path_out, index=False, encoding=enc or "utf-8-sig")
    
    print("\n" + "=" * 60)
    print("✓ 处理完成")
    print("=" * 60)
    print(f"总簇数: {total_clusters}")
    print(f"One-Hot 维度（删基准后）: {len(ohe_cols)}")
    print(f"基准簇: {baseline}")
    print(f"训练集缺失: {df_combined[df_combined[IS_TRAIN_COL]==1]['blk_missing'].sum()}")
    print(f"测试集缺失: {df_combined[df_combined[IS_TRAIN_COL]==0]['blk_missing'].sum()}")
    print(f"输出文件: {Path(path_out).resolve()}")


if __name__ == "__main__":
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        main()

板块特征工程（训练集聚类 + 测试集最近邻匹配）
总样本: 103871 (训练:83096, 测试:20775)

【训练集】板块聚类
------------------------------------------------------------
  板块总数（有中心点）: 971
  大样本板块（≥100）: 223 个
  小样本板块（<100）: 748 个，共 21935 样本
  小样本融入大样本: 153 个板块
  剩余小样本需聚类: 595 个板块，共 16599 样本
  小样本聚类完成: k=83
  训练集簇总数: 306

【测试集】最近邻匹配
------------------------------------------------------------
  匹配成功: 20775 个测试样本

✓ 处理完成
总簇数: 306
One-Hot 维度（删基准后）: 305
基准簇: 62.0
训练集缺失: 0
测试集缺失: 0
输出文件: H:\HW\ruc_Class25Q2_train_price_clean3_blk_smart_features.csv


In [27]:

# -*- coding: utf-8 -*-
"""
交易时间简化处理：只保留年份和月份的One-Hot编码
"""

import pandas as pd

def process_time_simple(df, time_col='交易时间'):
    """
    简化版：只做年份和月份的One-Hot编码
    
    参数:
        df: DataFrame
        time_col: 交易时间列名
    
    返回:
        df: 处理后的DataFrame（已删除原始交易时间列）
    """
    
    print("=" * 60)
    print("交易时间处理 - 简化版（仅年月One-Hot）")
    print("=" * 60)
    
    # 1. 解析时间
    time_split = df[time_col].str.split('/', expand=True)
    df['交易年份'] = time_split[0].astype(int)
    df['交易月份'] = time_split[1].astype(int)
    
    # 统计信息
    year_range = (df['交易年份'].min(), df['交易年份'].max())
    n_years = df['交易年份'].nunique()
    print(f"\n时间范围: {year_range[0]} ~ {year_range[1]}")
    print(f"年份数: {n_years} 个")
    print(f"月份分布:\n{df['交易月份'].value_counts().sort_index()}")
    
    # 2. 年份One-Hot（删基准）
    df = pd.get_dummies(df, columns=['交易年份'], prefix='year', drop_first=True, dtype=int)
    year_cols = [col for col in df.columns if col.startswith('year_')]
    print(f"\n✅ 年份One-Hot: {len(year_cols)}列")
    print(f"   基准年: {year_range[0]}")
    if year_cols:
        print(f"   列名: {year_cols}")
    
    # 3. 月份One-Hot（删基准）
    df = pd.get_dummies(df, columns=['交易月份'], prefix='month', drop_first=True, dtype=int)
    month_cols = [col for col in df.columns if col.startswith('month_')]
    print(f"\n✅ 月份One-Hot: {len(month_cols)}列")
    print(f"   基准月: 1月")
    print(f"   列名: {month_cols}")
    
    # 4. 删除原始交易时间列
    df.drop(time_col, axis=1, inplace=True)
    print(f"\n✅ 已删除列: {time_col}")
    
    print("=" * 60)
    print(f"完成！新增 {len(year_cols) + len(month_cols)} 列")
    print("=" * 60)
    
    return df


# ========== 使用示例 ==========
if __name__ == "__main__":
    
    # 读取数据
    path_in = r"H:\HW\ruc_Class25Q2_train_price_clean3_blk_smart_features.csv"
    df = pd.read_csv(path_in)
    
    print(f"处理前: {df.shape}")
    print(f"处理前列名（前15列）: {df.columns.tolist()[:15]}")
    
    # 处理交易时间
    df = process_time_simple(df, time_col='交易时间')
    
    print(f"\n处理后: {df.shape}")
    
    # 显示时间相关列
    time_cols = [col for col in df.columns if col.startswith('year_') or col.startswith('month_')]
    print(f"\n时间特征列（共{len(time_cols)}个）:")
    print(time_cols)
    
    # 查看示例
    print(f"\n示例（前5行）:")
    print(df[time_cols].head())
    
    # 保存
    output_path = path_in.replace('.csv', '_final.csv')
    df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n✅ 已保存: {output_path}")


处理前: (103871, 351)
处理前列名（前15列）: ['Price', '建筑面积', '交易时间', 'lon', 'lat', '房屋总数', '楼栋总数', '绿 化 率', '容 积 率', '物 业 费', 'is_train', '室', '厅', '厨', '卫']
交易时间处理 - 简化版（仅年月One-Hot）

时间范围: 2018 ~ 2025
年份数: 8 个
月份分布:
交易月份
1     13122
2      4791
3      5534
4      6393
5      7269
6      7677
7      9396
8      9972
9     10713
10     7653
11     6853
12    14498
Name: count, dtype: int64

✅ 年份One-Hot: 7列
   基准年: 2018
   列名: ['year_2019', 'year_2020', 'year_2021', 'year_2022', 'year_2023', 'year_2024', 'year_2025']

✅ 月份One-Hot: 11列
   基准月: 1月
   列名: ['month_2', 'month_3', 'month_4', 'month_5', 'month_6', 'month_7', 'month_8', 'month_9', 'month_10', 'month_11', 'month_12']

✅ 已删除列: 交易时间
完成！新增 18 列

处理后: (103871, 368)

时间特征列（共18个）:
['year_2019', 'year_2020', 'year_2021', 'year_2022', 'year_2023', 'year_2024', 'year_2025', 'month_2', 'month_3', 'month_4', 'month_5', 'month_6', 'month_7', 'month_8', 'month_9', 'month_10', 'month_11', 'month_12']

示例（前5行）:
   year_2019  year_2020  year_2021  year_202

In [28]:
# -*- coding: utf-8 -*-
"""
从原始文件合并城市列（保留One-Hot）
"""

import pandas as pd

# ===================== 配置 =====================
path_current = r"H:\HW\ruc_Class25Q2_train_price_clean3_blk_smart_features_final.csv"
path_original = r"H:\HW\ruc_Class25Q2_train_price_clean1.csv"  # 包含城市列的原始文件
# =================================================

print("=" * 80)
print("从原始文件合并城市列")
print("=" * 80)

# 读取当前数据（One-Hot版本）
df_current = pd.read_csv(path_current, encoding='utf-8')
print(f"当前数据: {df_current.shape}")

# 读取原始数据
try:
    df_original = pd.read_csv(path_original, encoding='utf-8')
    print(f"原始数据: {df_original.shape}")
except FileNotFoundError:
    print(f"❌ 原始文件未找到: {path_original}")
    exit(1)

# 查找城市列
city_col_candidates = ['城市']
city_col = None

for col in city_col_candidates:
    if col in df_original.columns:
        city_col = col
        break

if city_col is None:
    print(f"❌ 原始数据中未找到城市列")
    print(f"原始列名: {df_original.columns.tolist()[:20]}")
    exit(1)

print(f"✅ 找到城市列: '{city_col}'")

# 检查行数是否匹配
if len(df_current) != len(df_original):
    print(f"\n⚠️ 警告: 行数不匹配")
    print(f"   当前数据: {len(df_current)}行")
    print(f"   原始数据: {len(df_original)}行")
    
    response = input("是否继续？(可能导致数据错位) [y/N]: ")
    if response.lower() != 'y':
        exit(0)

# 合并城市列（添加到最后）
df_current['城市'] = df_original[city_col].values

print(f"\n✅ 城市列已添加")
print(f"数据形状: {df_current.shape}")

print(f"\n城市分布:")
print(df_current['城市'].value_counts())

# 保存
output_path = path_current.replace('.csv', '_with_city.csv')
df_current.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"\n✅ 已保存: {output_path}")
print("=" * 80)

从原始文件合并城市列
当前数据: (103871, 368)
原始数据: (103871, 41)
✅ 找到城市列: '城市'

✅ 城市列已添加
数据形状: (103871, 369)

城市分布:
城市
2     24996
3     21472
0     16491
10    15057
1      6437
8      5931
4      4363
5      3582
6      2281
9      1323
7      1184
11      754
Name: count, dtype: int64

✅ 已保存: H:\HW\ruc_Class25Q2_train_price_clean3_blk_smart_features_final_with_city.csv


In [34]:
# -*- coding: utf-8 -*-
"""
线性回归建模 - 极致去共线性 + 对数Price版
改进：
1. 对Price取对数（处理右偏）
2. 再次清理异常值
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
import joblib

# ===================== 配置区 =====================
path_in = r"H:\HW\ruc_Class25Q2_train_price_clean3_blk_smart_features_final_with_city.csv"

random_state = 42
test_size = 0.2
cv_folds = 6

# 明确指定连续变量
CONTINUOUS_FEATURES = [
    '建筑面积', '房屋总数', '楼栋总数', 
    '绿 化 率', '容 积 率', '物 业 费', 
    '室', '厅', '厨', '卫', '房龄'
]

# 特征工程开关
enable_log_transform = True      
enable_interaction = True        
enable_binning = True            

# 🔥 极致版：删除所有VIF>5的特征
FEATURES_TO_REMOVE = [
    '建筑面积', '容 积 率', '物 业 费', '卫', '房龄',
    '室', '厅',
]

ENGINEERED_FEATURES_TO_REMOVE = [
    '居住房间数',
    '建筑面积_x_容积率',
    '容 积 率_log',
    '房屋总数',
    '楼栋总数',
]

# =================================================

print("=" * 80)
print("线性回归建模 - 极致去共线性 + 对数Price版")
print("=" * 80)
print("🎯 改进：对数Price + 异常值清理")
print("=" * 80)


# ============ 1. 数据加载 + 对数Price + 异常值处理 ============
print("\n" + "=" * 80)
print("Step 1: 数据加载 + 对数Price + 异常值处理")
print("=" * 80)

df = pd.read_csv(path_in, encoding='utf-8')
print(f"原始数据形状: {df.shape}")

if 'Price' not in df.columns:
    raise ValueError("数据中没有'Price'列！")

# 删除城市特征
city_cols = [col for col in df.columns if col.startswith('city_')]
if city_cols:
    print(f"删除 {len(city_cols)} 个城市特征")
    df = df.drop(columns=city_cols)

# 验证连续特征
missing_features = [f for f in CONTINUOUS_FEATURES if f not in df.columns]
if missing_features:
    print(f"\n⚠️ 以下特征不存在: {missing_features}")
    CONTINUOUS_FEATURES = [f for f in CONTINUOUS_FEATURES if f in df.columns]

print(f"\n确认的连续特征({len(CONTINUOUS_FEATURES)}个): {CONTINUOUS_FEATURES}")

# 🔥 1.1 对Price取对数
print("\n" + "-" * 80)
print("1.1 对Price取对数（处理右偏分布）")
print("-" * 80)

print(f"原始Price统计:")
print(f"  均值: {df['Price'].mean():.2f}")
print(f"  标准差: {df['Price'].std():.2f}")
print(f"  范围: [{df['Price'].min():.2f}, {df['Price'].max():.2f}]")
print(f"  偏度: {df['Price'].skew():.2f} (>1为右偏)")

# 取对数
df['Price_log'] = np.log1p(df['Price'])

print(f"\n对数Price统计:")
print(f"  均值: {df['Price_log'].mean():.4f}")
print(f"  标准差: {df['Price_log'].std():.4f}")
print(f"  范围: [{df['Price_log'].min():.4f}, {df['Price_log'].max():.4f}]")
print(f"  偏度: {df['Price_log'].skew():.2f} (对数后接近正态)")

# 🔥 1.2 清理对数Price的异常值
print("\n" + "-" * 80)
print("1.2 清理对数Price的异常值（3σ原则）")
print("-" * 80)

mean_log = df['Price_log'].mean()
std_log = df['Price_log'].std()
lower_bound = mean_log - 3 * std_log
upper_bound = mean_log + 3 * std_log

print(f"对数Price的3σ范围: [{lower_bound:.4f}, {upper_bound:.4f}]")
print(f"对应原始Price范围: [{np.expm1(lower_bound):.0f}, {np.expm1(upper_bound):.0f}]")

outlier_mask = (df['Price_log'] < lower_bound) | (df['Price_log'] > upper_bound)
n_outliers = outlier_mask.sum()

print(f"\n异常值统计:")
print(f"  异常样本数: {n_outliers} ({n_outliers/len(df)*100:.2f}%)")

if n_outliers > 0:
    print(f"  异常Price范围: [{df[outlier_mask]['Price'].min():.0f}, {df[outlier_mask]['Price'].max():.0f}]")
    
    # 删除异常值
    df = df[~outlier_mask].copy()
    print(f"\n✅ 已删除 {n_outliers} 个异常样本")
    print(f"  清理后数据: {df.shape}")
else:
    print(f"✅ 无异常值")

# 分离特征和目标（使用对数Price）
y_log = df['Price_log'].copy()
y_original = df['Price'].copy()  # 保留原始Price用于后续评估
X = df.drop(['Price', 'Price_log'], axis=1)

print(f"\n✅ 数据准备完成:")
print(f"  样本数: {len(df)}")
print(f"  目标变量: Price_log (对数)")
print(f"  特征数: {X.shape[1]}")


# ============ 2. 识别特征类型 ============
print("\n" + "=" * 80)
print("Step 2: 特征类型识别")
print("=" * 80)

continuous_features = [f for f in CONTINUOUS_FEATURES 
                       if f in X.columns and f not in ['lon', 'lat']]

onehot_features = [col for col in X.columns 
                   if col not in continuous_features 
                   and col not in ['lon', 'lat']
                   and not col.startswith('city_')]

print(f"连续特征: {len(continuous_features)} 个")
print(f"One-Hot特征: {len(onehot_features)} 个")


# ============ 3. 特征工程 ============
print("\n" + "=" * 80)
print("Step 3: 特征工程")
print("=" * 80)

X_engineered = X.copy()
new_continuous_features = []

# 3.1 对数转换
if enable_log_transform:
    print("\n3.1 对数转换")
    for col in continuous_features:
        if (X_engineered[col] > 0).all():
            skewness = X_engineered[col].skew()
            if skewness > 1:
                new_col = f'{col}_log'
                X_engineered[new_col] = np.log1p(X_engineered[col])
                new_continuous_features.append(new_col)
                print(f"  {col} (偏度={skewness:.2f}) → {new_col}")

# 3.2 交互项
if enable_interaction:
    print("\n3.2 交互项")
    
    if '室' in continuous_features and '厅' in continuous_features:
        X_engineered['居住房间数'] = X_engineered['室'] + X_engineered['厅']
        new_continuous_features.append('居住房间数')
        print(f"  居住房间数 = 室+厅")
    
    if '建筑面积' in continuous_features and '居住房间数' in X_engineered.columns:
        X_engineered['居住房间数_adjusted'] = X_engineered['居住房间数'].replace(0, 1)
        new_col = '建筑面积_per_居住房间'
        X_engineered[new_col] = X_engineered['建筑面积'] / X_engineered['居住房间数_adjusted']
        new_continuous_features.append(new_col)
        print(f"  ✅ {new_col}")

# 3.3 分箱
if enable_binning and '建筑面积' in continuous_features:
    print("\n3.3 分箱")
    X_engineered['面积_小户型'] = (X_engineered['建筑面积'] < 70).astype(int)
    X_engineered['面积_中户型'] = ((X_engineered['建筑面积'] >= 70) & (X_engineered['建筑面积'] <= 120)).astype(int)
    X_engineered['面积_大户型'] = (X_engineered['建筑面积'] > 120).astype(int)
    onehot_features.extend(['面积_小户型', '面积_中户型', '面积_大户型'])
    print(f"  面积分箱完成")

all_continuous = continuous_features + new_continuous_features

print(f"\n✅ 特征工程汇总:")
print(f"  原始连续特征: {len(continuous_features)} 个")
print(f"  新增连续特征: {len(new_continuous_features)} 个")
print(f"  连续特征总计: {len(all_continuous)} 个")


# ============ 3.5 删除共线特征 ============
print("\n" + "=" * 80)
print("Step 3.5: 删除共线特征")
print("=" * 80)

all_features_to_remove = FEATURES_TO_REMOVE + ENGINEERED_FEATURES_TO_REMOVE
features_to_remove_actual = [f for f in all_features_to_remove if f in X_engineered.columns]

print(f"删除 {len(features_to_remove_actual)} 个共线特征")
X_engineered = X_engineered.drop(columns=features_to_remove_actual)

all_continuous = [f for f in all_continuous if f not in features_to_remove_actual]

print(f"✅ 剩余连续特征: {len(all_continuous)} 个")


# ============ 4. 划分训练集和测试集 ============
print("\n" + "=" * 80)
print("Step 4: 划分数据集")
print("=" * 80)

X_train, X_test, y_train_log, y_test_log, y_train_orig, y_test_orig = train_test_split(
    X_engineered, y_log, y_original, test_size=test_size, random_state=random_state
)

print(f"训练集: {X_train.shape[0]} 样本 ({X_train.shape[0]/len(X_engineered)*100:.1f}%)")
print(f"测试集: {X_test.shape[0]} 样本 ({X_test.shape[0]/len(X_engineered)*100:.1f}%)")


# ============ 5. 标准化 ============
print("\n" + "=" * 80)
print("Step 5: 标准化")
print("=" * 80)

all_continuous_exist = [col for col in all_continuous if col in X_train.columns]
print(f"标准化特征数: {len(all_continuous_exist)} 个")

scaler = StandardScaler()
X_train[all_continuous_exist] = scaler.fit_transform(X_train[all_continuous_exist])
X_test[all_continuous_exist] = scaler.transform(X_test[all_continuous_exist])

print("✅ 标准化完成")


# ============ 5.5 VIF验证 ============
print("\n" + "=" * 80)
print("Step 5.5: VIF验证")
print("=" * 80)

if len(all_continuous_exist) <= 30:
    print(f"计算 {len(all_continuous_exist)} 个连续特征的VIF...\n")
    
    try:
        X_vif = X_train[all_continuous_exist].values
        vif_data = pd.DataFrame()
        vif_data["Feature"] = all_continuous_exist
        vif_data["VIF"] = [variance_inflation_factor(X_vif, i) for i in range(len(all_continuous_exist))]
        vif_data = vif_data.sort_values('VIF', ascending=False)
        
        print("VIF检测结果:")
        print("=" * 60)
        print(vif_data.to_string(index=False))
        print("=" * 60)
        
        vif_excellent = len(vif_data[vif_data['VIF'] < 2])
        vif_good = len(vif_data[(vif_data['VIF'] >= 2) & (vif_data['VIF'] < 5)])
        vif_acceptable = len(vif_data[(vif_data['VIF'] >= 5) & (vif_data['VIF'] < 10)])
        vif_high = len(vif_data[vif_data['VIF'] >= 10])
        
        print(f"\n📊 VIF质量:")
        print(f"  VIF < 2: {vif_excellent} 个 ({vif_excellent/len(vif_data)*100:.1f}%)")
        print(f"  2 ≤ VIF < 5: {vif_good} 个 ({vif_good/len(vif_data)*100:.1f}%)")
        print(f"  5 ≤ VIF < 10: {vif_acceptable} 个 ({vif_acceptable/len(vif_data)*100:.1f}%)")
        print(f"  VIF ≥ 10: {vif_high} 个 ({vif_high/len(vif_data)*100:.1f}%)")
        
        if vif_high == 0 and vif_acceptable == 0:
            print(f"\n🎉 完美！所有VIF < 5")
        
        vif_data.to_csv('vif_log_price.csv', index=False, encoding='utf-8-sig')
        
    except Exception as e:
        print(f"\n⚠️ VIF计算失败: {e}")


# ============ 6. 特征选择 ============
print("\n" + "=" * 80)
print("Step 6: 特征选择")
print("=" * 80)

selected_continuous = all_continuous_exist.copy()

correlation_threshold = 0.01
correlations = []
for col in selected_continuous:
    corr = np.corrcoef(X_train[col], y_train_log)[0, 1]
    correlations.append({'feature': col, 'correlation': corr})

corr_df = pd.DataFrame(correlations)
corr_df['abs_corr'] = corr_df['correlation'].abs()

print(f"\nTop 10 高相关特征:")
top_corr = corr_df.nlargest(10, 'abs_corr')
for idx, row in top_corr.iterrows():
    direction = "📈" if row['correlation'] > 0 else "📉"
    print(f"  {direction} {row['feature']}: {row['correlation']:.4f}")

low_corr = corr_df[corr_df['abs_corr'] < correlation_threshold]
if len(low_corr) > 0:
    print(f"\n低相关性特征: {len(low_corr)} 个，已删除")
    selected_continuous = [f for f in selected_continuous if f not in low_corr['feature'].tolist()]

selected_features = selected_continuous + onehot_features

X_train_final = X_train[selected_features]
X_test_final = X_test[selected_features]

print(f"\n✅ 最终特征: {len(selected_features)} 个")
print(f"  连续: {len(selected_continuous)} | One-Hot: {len(onehot_features)}")


# ============ 7. 模型训练（对数Price）============
print("\n" + "=" * 80)
print("Step 7: 模型训练（对数Price建模）")
print("=" * 80)

def evaluate_model_log(model, X_train, X_test, y_train_log, y_test_log, y_test_orig, model_name):
    """评估对数Price模型（统一在原始空间计算所有指标）"""
    
    # ============ 1. 训练集预测 ============
    y_train_pred_log = model.predict(X_train)
    y_train_pred = np.expm1(y_train_pred_log)  # 转回原始尺度
    y_train_actual = np.expm1(y_train_log)
    
    mae_train = mean_absolute_error(y_train_actual, y_train_pred)
    rmse_train = np.sqrt(mean_squared_error(y_train_actual, y_train_pred))
    r2_train = r2_score(y_train_actual, y_train_pred)
    
    # ============ 2. 测试集预测 ============
    y_test_pred_log = model.predict(X_test)
    y_test_pred = np.expm1(y_test_pred_log)  # 转回原始尺度
    
    mae_test = mean_absolute_error(y_test_orig, y_test_pred)
    rmse_test = np.sqrt(mean_squared_error(y_test_orig, y_test_pred))
    r2_test = r2_score(y_test_orig, y_test_pred)
    
    # ============ 3. 交叉验证（在原始空间计算MAE和R²）============
    kfold = KFold(n_splits=cv_folds, shuffle=True, random_state=random_state)
    cv_scores_mae = []
    cv_scores_r2 = []
    
    for train_idx, val_idx in kfold.split(X_train):
        X_cv_train, X_cv_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_cv_train_log, y_cv_val_log = y_train_log.iloc[train_idx], y_train_log.iloc[val_idx]
        
        # 训练模型
        model.fit(X_cv_train, y_cv_train_log)
        y_cv_pred_log = model.predict(X_cv_val)
        
        # 🔥 转回原始尺度计算指标
        y_cv_pred = np.expm1(y_cv_pred_log)
        y_cv_actual = np.expm1(y_cv_val_log)
        
        cv_scores_mae.append(mean_absolute_error(y_cv_actual, y_cv_pred))
        cv_scores_r2.append(r2_score(y_cv_actual, y_cv_pred))
    
    mae_cv = np.mean(cv_scores_mae)
    mae_cv_std = np.std(cv_scores_mae)
    r2_cv = np.mean(cv_scores_r2)
    r2_cv_std = np.std(cv_scores_r2)
    
    # ============ 4. 返回完整结果 ============
    return {
        'Model': model_name,
        'MAE_train': mae_train,
        'RMSE_train': rmse_train,
        'R2_train': r2_train,
        'MAE_test': mae_test,
        'RMSE_test': rmse_test,
        'R2_test': r2_test,
        'MAE_cv': mae_cv,
        'MAE_cv_std': mae_cv_std,
        'R2_cv': r2_cv,
        'R2_cv_std': r2_cv_std,
        'Overfit': mae_test - mae_train
    }
results = []  

# 7.1 OLS (对数Price)
print("\n7.1 OLS（对数Price）")
ols_model = LinearRegression()
ols_model.fit(X_train_final, y_train_log)
ols_result = evaluate_model_log(ols_model, X_train_final, X_test_final, 
                                 y_train_log, y_test_log, y_test_orig, 'OLS_对数Price')
results.append(ols_result)
print(f"  测试集: MAE={ols_result['MAE_test']:.2f} | RMSE={ols_result['RMSE_test']:.2f} | R²={ols_result['R2_test']:.4f}")
print(f"  交叉验证: MAE={ols_result['MAE_cv']:.2f}±{ols_result['MAE_cv_std']:.2f} | R²={ols_result['R2_cv']:.4f}")

# 7.2 Ridge (对数Price)
print("\n7.2 Ridge（对数Price）")
ridge_params = {'alpha': [0.001, 0.01, 0.1, 1, 10, 100]}
ridge_grid = GridSearchCV(Ridge(random_state=random_state), ridge_params, 
                          cv=cv_folds, scoring='neg_mean_absolute_error', n_jobs=-1)
ridge_grid.fit(X_train_final, y_train_log)
ridge_model = ridge_grid.best_estimator_
ridge_result = evaluate_model_log(ridge_model, X_train_final, X_test_final, 
                                   y_train_log, y_test_log, y_test_orig, 'Ridge_对数Price')
results.append(ridge_result)
print(f"  最佳alpha: {ridge_grid.best_params_['alpha']}")
print(f"  测试集: MAE={ridge_result['MAE_test']:.2f} | RMSE={ridge_result['RMSE_test']:.2f} | R²={ridge_result['R2_test']:.4f}")
print(f"  交叉验证: MAE={ridge_result['MAE_cv']:.2f}±{ridge_result['MAE_cv_std']:.2f} | R²={ridge_result['R2_cv']:.4f}")

# 7.3 Lasso (对数Price)
print("\n7.3 Lasso（对数Price）")
lasso_params = {'alpha': [0.0001, 0.001, 0.01, 0.1]}
lasso_grid = GridSearchCV(
    Lasso(random_state=random_state, max_iter=5000), 
    lasso_params, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1
)
lasso_grid.fit(X_train_final, y_train_log)
lasso_model = lasso_grid.best_estimator_
lasso_result = evaluate_model_log(lasso_model, X_train_final, X_test_final, 
                                   y_train_log, y_test_log, y_test_orig, 'Lasso_对数Price')
results.append(lasso_result)

n_nonzero = np.sum(lasso_model.coef_ != 0)
print(f"  最佳alpha: {lasso_grid.best_params_['alpha']}")
print(f"  非零系数: {n_nonzero}/{len(lasso_model.coef_)} ({n_nonzero/len(lasso_model.coef_)*100:.1f}%)")
print(f"  测试集: MAE={lasso_result['MAE_test']:.2f} | RMSE={lasso_result['RMSE_test']:.2f} | R²={lasso_result['R2_test']:.4f}")
print(f"  交叉验证: MAE={lasso_result['MAE_cv']:.2f}±{lasso_result['MAE_cv_std']:.2f} | R²={lasso_result['R2_cv']:.4f}")

# 7.4 ElasticNet (对数Price)
print("\n7.4 ElasticNet（对数Price）")
elastic_params = {'alpha': [0.001, 0.01, 0.1], 'l1_ratio': [0.3, 0.5, 0.7]}
elastic_grid = GridSearchCV(
    ElasticNet(random_state=random_state, max_iter=3000),
    elastic_params, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1
)
elastic_grid.fit(X_train_final, y_train_log)
elastic_model = elastic_grid.best_estimator_
elastic_result = evaluate_model_log(elastic_model, X_train_final, X_test_final, 
                                     y_train_log, y_test_log, y_test_orig, 'ElasticNet_对数Price')
results.append(elastic_result)
print(f"  最佳参数: alpha={elastic_grid.best_params_['alpha']}, l1_ratio={elastic_grid.best_params_['l1_ratio']}")
print(f"  测试集: MAE={elastic_result['MAE_test']:.2f} | RMSE={elastic_result['RMSE_test']:.2f} | R²={elastic_result['R2_test']:.4f}")
print(f"  交叉验证: MAE={elastic_result['MAE_cv']:.2f}±{elastic_result['MAE_cv_std']:.2f} | R²={elastic_result['R2_cv']:.4f}")


# ============ 8. 结果汇总 ============
print("\n" + "=" * 80)
print("Step 8: 结果汇总")
print("=" * 80)

results_df = pd.DataFrame(results)

# 找出最佳模型
best_idx = results_df['MAE_test'].idxmin()
best_model_name = results_df.loc[best_idx, 'Model']
best_result = results_df.loc[best_idx]

print(f"\n{'🏆' * 40}")
print(f"最佳模型: {best_model_name}")
print(f"{'🏆' * 40}")

# ========== 核心性能指标表格 ==========
print("\n" + "=" * 80)
print("📊 核心性能指标（最佳模型）")
print("=" * 80)

performance_table = pd.DataFrame({
    '评估维度': ['样本内性能', '样本外性能', '6折交叉验证'],
    'MAE': [
        f"{best_result['MAE_train']:,.0f}",
        f"{best_result['MAE_test']:,.0f}",
        f"{best_result['MAE_cv']:,.0f} ± {best_result['MAE_cv_std']:,.0f}"
    ],
    'RMSE': [
        f"{best_result['RMSE_train']:,.0f}",
        f"{best_result['RMSE_test']:,.0f}",
        "N/A"
    ],
    'R²': [
        f"{best_result['R2_train']:.4f}",
        f"{best_result['R2_test']:.4f}",
        f"{best_result['R2_cv']:.4f}"
    ]
})

print(performance_table.to_string(index=False))

# 详细指标
print(f"\n" + "=" * 80)
print("📈 详细指标")
print("=" * 80)
print(f"【样本内性能（训练集）】")
print(f"  MAE:  {best_result['MAE_train']:>15,.0f} 元")
print(f"  RMSE: {best_result['RMSE_train']:>15,.0f} 元")
print(f"  R²:   {best_result['R2_train']:>15.4f}")

print(f"\n【样本外性能（测试集）】")
print(f"  MAE:  {best_result['MAE_test']:>15,.0f} 元")
print(f"  RMSE: {best_result['RMSE_test']:>15,.0f} 元")
print(f"  R²:   {best_result['R2_test']:>15.4f}")
print(f"  相对误差: {best_result['MAE_test']/y_original.mean()*100:>11.1f}%")

print(f"\n【6折交叉验证】")
print(f"  MAE:  {best_result['MAE_cv']:>15,.0f} ± {best_result['MAE_cv_std']:,.0f} 元")
print(f"  R²:   {best_result['R2_cv']:>15.4f}")

print(f"\n【模型稳定性】")
overfit_amount = best_result['Overfit']
overfit_pct = (best_result['MAE_test'] / best_result['MAE_train'] - 1) * 100
print(f"  过拟合量: {overfit_amount:>14,.0f} 元 ({overfit_pct:+.1f}%)")

if abs(overfit_pct) < 5:
    overfit_level = "🎉 优秀（几乎无过拟合）"
elif abs(overfit_pct) < 10:
    overfit_level = "✅ 良好"
elif abs(overfit_pct) < 20:
    overfit_level = "⚠️ 可接受"
else:
    overfit_level = "❌ 过拟合严重"

print(f"  稳定性评价: {overfit_level}")

# 数据集信息
print(f"\n" + "=" * 80)
print("📦 数据集信息")
print("=" * 80)
print(f"  原始数据量: {len(df) + n_outliers:>15,} 样本")
print(f"  剔除异常值: {n_outliers:>15,} 样本 ({n_outliers/(len(df)+n_outliers)*100:.2f}%)")
print(f"  剔除后总量: {len(df):>15,} 样本 ✅")
print(f"  训练集: {len(X_train):>19,} 样本 ({len(X_train)/len(df)*100:.1f}%)")
print(f"  测试集: {len(X_test):>19,} 样本 ({len(X_test)/len(df)*100:.1f}%)")
print(f"  特征数量: {len(selected_features):>17,} 个")

# 性能对比表
print("\n" + "=" * 80)
print("📊 所有模型性能对比（对数Price建模）")
print("=" * 80)

comparison = pd.DataFrame({
    '模型': [r['Model'] for r in results],
    '样本内MAE': [f"{r['MAE_train']:,.0f}" for r in results],
    '样本外MAE': [f"{r['MAE_test']:,.0f}" for r in results],
    '样本外RMSE': [f"{r['RMSE_test']:,.0f}" for r in results],
    'CV_MAE': [f"{r['MAE_cv']:,.0f}±{r['MAE_cv_std']:.0f}" for r in results],
    '测试R²': [f"{r['R2_test']:.4f}" for r in results],
    '相对误差%': [f"{r['MAE_test']/y_original.mean()*100:.1f}" for r in results]
})

print(comparison.to_string(index=False))

# 保存结果
results_df.to_csv('model_results_log_price.csv', index=False, encoding='utf-8-sig')
comparison.to_csv('model_comparison_log_price.csv', index=False, encoding='utf-8-sig')

# 保存最佳模型
if 'OLS' in best_model_name:
    best_model = ols_model
elif 'Lasso' in best_model_name:
    best_model = lasso_model
elif 'Ridge' in best_model_name:
    best_model = ridge_model
else:
    best_model = elastic_model

joblib.dump(best_model, f'best_model_log_price.pkl')
joblib.dump(ols_model, 'model_OLS_log_price.pkl')
joblib.dump(ridge_model, 'model_Ridge_log_price.pkl')
joblib.dump(lasso_model, 'model_Lasso_log_price.pkl')
joblib.dump(elastic_model, 'model_ElasticNet_log_price.pkl')
joblib.dump(scaler, 'scaler_log_price.pkl')
joblib.dump(selected_features, 'selected_features_log_price.pkl')

print("\n" + "=" * 80)
print("💾 保存的文件")
print("=" * 80)
print("  📊 model_results_log_price.csv")
print("  📊 model_comparison_log_price.csv")
print("  📊 vif_log_price.csv")
print(f"  💾 best_model_log_price.pkl ({best_model_name})")
print("  💾 model_OLS_log_price.pkl")
print("  💾 model_Ridge_log_price.pkl")
print("  💾 model_Lasso_log_price.pkl")
print("  💾 model_ElasticNet_log_price.pkl")
print("  💾 scaler_log_price.pkl")
print("  💾 selected_features_log_price.pkl")


# ============ 9. 总结报告 ============
print("\n" + "=" * 80)
print("🎉 对数Price建模完成！")
print("=" * 80)

print(f"\n📋 数据处理总结:")
print(f"  原始数据量: {len(df) + n_outliers:,} 样本")
print(f"  对数变换: Price偏度 {df['Price'].skew():.2f} → Price_log偏度 {df['Price_log'].skew():.2f}")
print(f"  异常值清理 (3σ): 删除 {n_outliers:,} 样本 ({n_outliers/(len(df)+n_outliers)*100:.2f}%)")
print(f"  ✅ 剔除后总量: {len(df):,} 样本")
print(f"  特征工程: {X.shape[1]} → {len(selected_features)} 个特征")

print(f"\n🏆 最佳模型: {best_model_name}")
print(f"  【样本内】  MAE = {best_result['MAE_train']:,.0f},  RMSE = {best_result['RMSE_train']:,.0f},  R² = {best_result['R2_train']:.4f}")
print(f"  【样本外】  MAE = {best_result['MAE_test']:,.0f},  RMSE = {best_result['RMSE_test']:,.0f},  R² = {best_result['R2_test']:.4f}")
print(f"  【交叉验证】MAE = {best_result['MAE_cv']:,.0f} ± {best_result['MAE_cv_std']:,.0f},  R² = {best_result['R2_cv']:.4f}")
print(f"  相对误差: {best_result['MAE_test']/y_original.mean()*100:.1f}%")

# 性能评价
relative_error_pct = best_result['MAE_test']/y_original.mean()*100
overfit_pct = (best_result['MAE_test'] / best_result['MAE_train'] - 1) * 100

print(f"\n📈 性能评价:")
if relative_error_pct < 25:
    print(f"  ✅ 预测精度: 🎉 优秀！相对误差 {relative_error_pct:.1f}% < 25%")
elif relative_error_pct < 30:
    print(f"  ✅ 预测精度: 良好，相对误差 {relative_error_pct:.1f}% < 30%")
else:
    print(f"  ⚠️ 预测精度: 一般，相对误差 {relative_error_pct:.1f}% > 30%，建议优化")

if abs(overfit_pct) < 10:
    print(f"  ✅ 泛化能力: 优秀，过拟合程度 {overfit_pct:+.1f}% < 10%")
elif abs(overfit_pct) < 20:
    print(f"  ✅ 泛化能力: 良好，过拟合程度 {overfit_pct:+.1f}% < 20%")
else:
    print(f"  ⚠️ 泛化能力: 有过拟合，过拟合程度 {overfit_pct:+.1f}% > 20%")

if best_result['R2_test'] > 0.7:
    print(f"  ✅ 拟合优度: 优秀，R² = {best_result['R2_test']:.4f} > 0.7")
elif best_result['R2_test'] > 0.6:
    print(f"  ✅ 拟合优度: 良好，R² = {best_result['R2_test']:.4f} > 0.6")
else:
    print(f"  ⚠️ 拟合优度: 一般，R² = {best_result['R2_test']:.4f} < 0.6")

print(f"\n💡 使用说明:")
print("  1. 预测时需要先对新数据用scaler标准化")
print("  2. 模型输出是log(Price+1)，需要用np.expm1()转回原始价格")
print("  3. 示例代码:")
print("     model = joblib.load('best_model_log_price.pkl')")
print("     scaler = joblib.load('scaler_log_price.pkl')")
print("     features = joblib.load('selected_features_log_price.pkl')")
print("     X_new_scaled = scaler.transform(X_new[features])")
print("     y_pred_log = model.predict(X_new_scaled)")
print("     y_pred = np.expm1(y_pred_log)  # 转回原始价格")

print("\n" + "=" * 80)
print("✨ 建模完成！✨")
print("=" * 80)

线性回归建模 - 极致去共线性 + 对数Price版
🎯 改进：对数Price + 异常值清理

Step 1: 数据加载 + 对数Price + 异常值处理
原始数据形状: (103871, 369)
删除 11 个城市特征

确认的连续特征(11个): ['建筑面积', '房屋总数', '楼栋总数', '绿 化 率', '容 积 率', '物 业 费', '室', '厅', '厨', '卫', '房龄']

--------------------------------------------------------------------------------
1.1 对Price取对数（处理右偏分布）
--------------------------------------------------------------------------------
原始Price统计:
  均值: 2241952.18
  标准差: 2366670.85
  范围: [79776.88, 42114016.57]
  偏度: 3.40 (>1为右偏)

对数Price统计:
  均值: 14.2630
  标准差: 0.8239
  范围: [11.2870, 17.5559]
  偏度: 0.24 (对数后接近正态)

--------------------------------------------------------------------------------
1.2 清理对数Price的异常值（3σ原则）
--------------------------------------------------------------------------------
对数Price的3σ范围: [11.7912, 16.7347]
对应原始Price范围: [132085, 18526141]

异常值统计:
  异常样本数: 90 (0.09%)
  异常Price范围: [79777, 42114017]

✅ 已删除 90 个异常样本
  清理后数据: (103781, 359)

✅ 数据准备完成:
  样本数: 103781
  目标变量: Price_log (对数)
  特征数: 357

Step 2: 特征类型识别
连续

In [41]:
# -*- coding: utf-8 -*-
"""
预测集变量处理 - 保留指定列
"""

import pandas as pd
import numpy as np

# ===================== 配置 =====================
path_test = r"H:\HW\ruc_Class25Q2_test_price.csv"  # 预测集路径

# 需要保留的列
KEEP_COLUMNS = [
    'ID',
    '城市',
    '房屋户型',
    '所在楼层',
    '建筑面积',
    '房屋朝向',
    '建筑结构',
    '装修情况',
    '配备电梯',
    '交易时间',
    'lon',
    'lat',
    '建筑年代',
    '房屋总数',
    '楼栋总数',
    '绿 化 率',
    '容 积 率',
    '物 业 费'
]
# =================================================

print("=" * 80)
print("预测集变量处理")
print("=" * 80)

# 读取数据
try:
    df_test = pd.read_csv(path_test, encoding='utf-8')
    print(f"✅ 数据加载成功")
except FileNotFoundError:
    # 尝试其他可能的文件名
    alternative_paths = [
        r"H:\HW\ruc_Class25Q2_test_price.csv",
        r"H:\HW\ruc_Class25Q2_test.csv",
        r"H:\HW\test_price.csv"
    ]
    df_test = None
    for path in alternative_paths:
        try:
            df_test = pd.read_csv(path, encoding='utf-8')
            path_test = path
            print(f"✅ 数据加载成功: {path}")
            break
        except:
            continue
    
    if df_test is None:
        print(f"❌ 文件未找到，请检查路径")
        exit(1)

print(f"原始数据形状: {df_test.shape}")
print(f"原始列数: {len(df_test.columns)}")

# 显示现有列
print(f"\n现有列名({len(df_test.columns)}个):")
for i, col in enumerate(df_test.columns, 1):
    print(f"  {i}. {col}")

# 检查哪些列存在
existing_columns = []
missing_columns = []

for col in KEEP_COLUMNS:
    if col in df_test.columns:
        existing_columns.append(col)
    else:
        missing_columns.append(col)

print(f"\n" + "=" * 80)
print(f"列检查结果:")
print(f"=" * 80)
print(f"✅ 存在的列: {len(existing_columns)}/{len(KEEP_COLUMNS)}")
for col in existing_columns:
    print(f"   {col}")

if len(missing_columns) > 0:
    print(f"\n⚠️ 缺失的列: {len(missing_columns)}个")
    for col in missing_columns:
        print(f"   {col}")

# 保留存在的列
df_test_clean = df_test[existing_columns].copy()

print(f"\n" + "=" * 80)
print(f"处理后的数据:")
print(f"=" * 80)
print(f"数据形状: {df_test_clean.shape}")
print(f"保留列数: {len(existing_columns)}")
print(f"删除列数: {len(df_test.columns) - len(existing_columns)}")

# 数据质量检查
print(f"\n" + "=" * 80)
print(f"数据质量检查:")
print(f"=" * 80)

# 缺失值统计
missing_stats = df_test_clean.isnull().sum()
missing_cols = missing_stats[missing_stats > 0]

if len(missing_cols) > 0:
    print(f"\n⚠️ 有缺失值的列:")
    for col, count in missing_cols.items():
        pct = count / len(df_test_clean) * 100
        print(f"   {col}: {count}个 ({pct:.2f}%)")
else:
    print(f"✅ 无缺失值")

# 数据类型
print(f"\n数据类型分布:")
print(df_test_clean.dtypes.value_counts())

# 基本统计（数值列）
numeric_cols = df_test_clean.select_dtypes(include=[np.number]).columns
if len(numeric_cols) > 0:
    print(f"\n数值列统计({len(numeric_cols)}个):")
    print(df_test_clean[numeric_cols].describe())

# 保存
output_path = path_test.replace('.csv', '_clean.csv')
df_test_clean.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"\n" + "=" * 80)
print(f"✅ 处理完成")
print(f"=" * 80)
print(f"输出文件: {output_path}")
print(f"数据形状: {df_test_clean.shape}")
print(f"保留列: {existing_columns}")
print("=" * 80)

预测集变量处理
✅ 数据加载成功
原始数据形状: (34017, 55)
原始列数: 55

现有列名(55个):
  1. ID
  2. 城市
  3. 区域
  4. 板块
  5. 环线
  6. 房屋户型
  7. 所在楼层
  8. 建筑面积
  9. 套内面积
  10. 房屋朝向
  11. 建筑结构
  12. 装修情况
  13. 梯户比例
  14. 配备电梯
  15. 别墅类型
  16. 交易时间
  17. 交易权属
  18. 上次交易
  19. 房屋用途
  20. 房屋年限
  21. 产权所属
  22. 抵押信息
  23. 房屋优势
  24. 核心卖点
  25. 户型介绍
  26. 周边配套
  27. 交通出行
  28. lon
  29. lat
  30. 年份
  31. 区县
  32. 板块_comm
  33. 环线位置
  34. 物业类别
  35. 建筑年代
  36. 开发商
  37. 房屋总数
  38. 楼栋总数
  39. 物业公司
  40. 绿 化 率
  41. 容 积 率
  42. 物 业 费
  43. 建筑结构_comm
  44. 物业办公电话
  45. 产权描述
  46. 供水
  47. 供暖
  48. 供电
  49. 燃气费
  50. 供热费
  51. 停车位
  52. 停车费用
  53. coord_x
  54. coord_y
  55. 客户反馈

列检查结果:
✅ 存在的列: 18/18
   ID
   城市
   房屋户型
   所在楼层
   建筑面积
   房屋朝向
   建筑结构
   装修情况
   配备电梯
   交易时间
   lon
   lat
   建筑年代
   房屋总数
   楼栋总数
   绿 化 率
   容 积 率
   物 业 费

处理后的数据:
数据形状: (34017, 18)
保留列数: 18
删除列数: 37

数据质量检查:

⚠️ 有缺失值的列:
   房屋户型: 14个 (0.04%)
   建筑结构: 14个 (0.04%)
   装修情况: 14个 (0.04%)
   配备电梯: 4092个 (12.03%)
   建筑年代: 9406个 (27.65%)
   房屋总数: 3715

In [43]:
#python
# -*- coding: utf-8 -*-
"""
预测集处理：添加室厅厨卫变量
"""

import pandas as pd
import numpy as np
import re

# ===================== 配置 =====================
path_test = r"H:\HW\ruc_Class25Q2_test_price_clean.csv"  # 上一步的输出文件

# 缺失值默认填充
DEFAULT_VALUES = {
    '室': 3,
    '厅': 2,
    '厨': 1,
    '卫': 1
}
# =================================================

print("=" * 80)
print("添加室厅厨卫变量（并删除房屋户型列）")
print("=" * 80)

# 读取数据
df = pd.read_csv(path_test, encoding='utf-8')
print(f"数据形状: {df.shape}")
print(f"列数: {len(df.columns)}")

# 检查是否有房屋户型列
if '房屋户型' not in df.columns:
    print("❌ 未找到'房屋户型'列")
    print(f"现有列: {df.columns.tolist()}")
    exit(1)

print(f"\n✅ 找到'房屋户型'列")
print(f"样本数量: {len(df)}")
print(f"缺失值: {df['房屋户型'].isnull().sum()} ({df['房屋户型'].isnull().sum()/len(df)*100:.2f}%)")

# 显示样例
print(f"\n房屋户型样例:")
sample_values = df['房屋户型'].dropna().head(10)
for i, val in enumerate(sample_values, 1):
    print(f"  {i}. {val}")

# 定义提取函数
def extract_room_info(户型):
    """
    从房屋户型字符串中提取室厅厨卫数量
    例如: "3室2厅1厨2卫" -> {'室': 3, '厅': 2, '厨': 1, '卫': 2}
    """
    if pd.isnull(户型):
        return DEFAULT_VALUES.copy()
    
    户型 = str(户型).strip()
    
    if 户型 == '' or 户型.lower() in ['nan', 'none', '未知', '暂无']:
        return DEFAULT_VALUES.copy()
    
    result = {}
    
    # 匹配模式：数字+室/厅/厨/卫
    patterns = {
        '室': r'(\d+)\s*室',
        '厅': r'(\d+)\s*厅',
        '厨': r'(\d+)\s*厨',
        '卫': r'(\d+)\s*卫'
    }
    
    for key, pattern in patterns.items():
        match = re.search(pattern, 户型)
        if match:
            result[key] = int(match.group(1))
        else:
            result[key] = DEFAULT_VALUES[key]
    
    return result

# 提取室厅厨卫
print(f"\n" + "=" * 80)
print(f"开始提取室厅厨卫...")
print(f"=" * 80)

room_info_list = df['房屋户型'].apply(extract_room_info)

# 转换为DataFrame
room_df = pd.DataFrame(room_info_list.tolist())

# 添加到原数据
df['室'] = room_df['室']
df['厅'] = room_df['厅']
df['厨'] = room_df['厨']
df['卫'] = room_df['卫']

print(f"✅ 提取完成")

# 统计
print(f"\n室厅厨卫统计:")
print(f"\n室的分布:")
print(df['室'].value_counts().sort_index())

print(f"\n厅的分布:")
print(df['厅'].value_counts().sort_index())

print(f"\n厨的分布:")
print(df['厨'].value_counts().sort_index())

print(f"\n卫的分布:")
print(df['卫'].value_counts().sort_index())

# 统计使用默认值的数量
default_count = 0
for idx, row in df.iterrows():
    if (row['室'] == DEFAULT_VALUES['室'] and 
        row['厅'] == DEFAULT_VALUES['厅'] and 
        row['厨'] == DEFAULT_VALUES['厨'] and 
        row['卫'] == DEFAULT_VALUES['卫']):
        
        original = df.loc[idx, '房屋户型']
        if pd.isnull(original) or str(original).strip() == '':
            default_count += 1

print(f"\n使用默认值(3室2厅1厨1卫)的记录: {default_count} ({default_count/len(df)*100:.2f}%)")

# 删除房屋户型列
print(f"\n" + "=" * 80)
print(f"删除'房屋户型'列")
print(f"=" * 80)

df = df.drop('房屋户型', axis=1)

print(f"✅ 已删除'房屋户型'列")
print(f"新的列数: {len(df.columns)}")

# 数据质量检查
print(f"\n" + "=" * 80)
print(f"数据质量检查:")
print(f"=" * 80)

# 检查异常值
print(f"\n异常值检查:")

abnormal_室 = df[df['室'] > 10]
abnormal_厅 = df[df['厅'] > 5]
abnormal_厨 = df[df['厨'] > 3]
abnormal_卫 = df[df['卫'] > 5]

if len(abnormal_室) > 0:
    print(f"⚠️ 室 > 10: {len(abnormal_室)}条 (最大={df['室'].max()})")
if len(abnormal_厅) > 0:
    print(f"⚠️ 厅 > 5: {len(abnormal_厅)}条 (最大={df['厅'].max()})")
if len(abnormal_厨) > 0:
    print(f"⚠️ 厨 > 3: {len(abnormal_厨)}条 (最大={df['厨'].max()})")
if len(abnormal_卫) > 0:
    print(f"⚠️ 卫 > 5: {len(abnormal_卫)}条 (最大={df['卫'].max()})")

if (len(abnormal_室) + len(abnormal_厅) + len(abnormal_厨) + len(abnormal_卫)) == 0:
    print(f"✅ 无异常值")

# 基本统计
print(f"\n基本统计:")
print(df[['室', '厅', '厨', '卫']].describe())

# 保存
output_path = path_test.replace('_clean.csv', '_clean_with_rooms.csv')
if output_path == path_test:  # 如果文件名没变
    output_path = path_test.replace('.csv', '_with_rooms.csv')

df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"\n" + "=" * 80)
print(f"✅ 处理完成！")
print(f"=" * 80)
print(f"输出文件: {output_path}")
print(f"数据形状: {df.shape}")
print(f"\n变化:")
print(f"   - 删除列: 房屋户型")
print(f"   - 新增列: 室、厅、厨、卫")
print(f"   - 默认值: 3室2厅1厨1卫")

print(f"\n最终列名({len(df.columns)}个):")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

print("=" * 80)

添加室厅厨卫变量（并删除房屋户型列）
数据形状: (34017, 18)
列数: 18

✅ 找到'房屋户型'列
样本数量: 34017
缺失值: 14 (0.04%)

房屋户型样例:
  1. 3室2厅1厨2卫
  2. 2室1厅1厨1卫
  3. 3室1厅1厨2卫
  4. 2室1厅1厨1卫
  5. 3室2厅1厨2卫
  6. 3室1厅1厨1卫
  7. 3室2厅1厨3卫
  8. 2室1厅1厨1卫
  9. 3室2厅1厨2卫
  10. 3室1厅1厨2卫

开始提取室厅厨卫...
✅ 提取完成

室厅厨卫统计:

室的分布:
室
0        6
1     3489
2    13094
3    13660
4     3189
5      478
6       83
7       12
8        6
Name: count, dtype: int64

厅的分布:
厅
0      733
1    15988
2    17109
3      167
4       17
5        3
Name: count, dtype: int64

厨的分布:
厨
0      296
1    33517
2      201
3        3
Name: count, dtype: int64

卫的分布:
卫
0       72
1    22609
2    10286
3      826
4      185
5       35
6        4
Name: count, dtype: int64

使用默认值(3室2厅1厨1卫)的记录: 14 (0.04%)

删除'房屋户型'列
✅ 已删除'房屋户型'列
新的列数: 21

数据质量检查:

异常值检查:
⚠️ 卫 > 5: 4条 (最大=6)

基本统计:
                  室             厅             厨             卫
count  34017.000000  34017.000000  34017.000000  34017.000000
mean       2.540877      1.493077      0.997384      1.369844
std        0.87

In [44]:
# -*- coding: utf-8 -*-
"""
预测集特征工程：One-Hot编码
以样本量最大的类别为基准（删除该类别的变量，避免多重共线性）
朝向除外：保留所有4个方向
"""

import pandas as pd
import numpy as np
import re

# ===================== 配置 =====================
path_test = r"H:\HW\ruc_Class25Q2_test_price_clean_with_rooms.csv"
# =================================================

print("=" * 80)
print("预测集特征工程：One-Hot编码（删除最大类别）")
print("=" * 80)

# 读取数据
df = pd.read_csv(path_test, encoding='utf-8')
print(f"原始数据: {df.shape}")
print(f"原始列数: {len(df.columns)}")

# 用于记录删除的基准类别
dropped_categories = []

# ============================================================
# 1. 楼层处理（6选5，删除最多的）
# ============================================================
print("\n" + "=" * 80)
print("1. 楼层处理（One-Hot编码）")
print("=" * 80)

if '所在楼层' not in df.columns:
    print("⚠️ 未找到'所在楼层'列")
else:
    print(f"样例数据:")
    print(df['所在楼层'].value_counts().head(10))
    
    # 定义楼层分类函数
    def classify_floor(floor_str):
        """分类楼层"""
        result = {
            '地下室': 0,
            '底层': 0,
            '低楼层': 0,
            '中楼层': 0,
            '高楼层': 0,
            '顶层': 0
        }
        
        if pd.isnull(floor_str):
            result['中楼层'] = 1  # 默认中楼层
            return result
        
        floor_str = str(floor_str).strip().lower()
        
        # 地下室
        if '地下' in floor_str or 'basement' in floor_str or floor_str.startswith('-'):
            result['地下室'] = 1
        # 底层
        elif '底层' in floor_str or floor_str in ['1层', '一层', '1']:
            result['底层'] = 1
        # 低楼层
        elif '低层' in floor_str or '低楼层' in floor_str:
            result['低楼层'] = 1
        # 中楼层
        elif '中层' in floor_str or '中楼层' in floor_str:
            result['中楼层'] = 1
        # 高楼层
        elif '高层' in floor_str or '高楼层' in floor_str:
            result['高楼层'] = 1
        # 顶层
        elif '顶层' in floor_str or 'top' in floor_str:
            result['顶层'] = 1
        else:
            # 默认中楼层
            result['中楼层'] = 1
        
        return result
    
    # 应用分类
    floor_data = df['所在楼层'].apply(classify_floor)
    floor_df = pd.DataFrame(floor_data.tolist())
    
    # 统计各类别样本数
    print(f"\n楼层分布:")
    floor_counts = {}
    for col in floor_df.columns:
        count = floor_df[col].sum()
        floor_counts[col] = count
        pct = count / len(df) * 100
        print(f"  {col:10s}: {count:5d} ({pct:5.2f}%)")
    
    # 找到样本最多的类别（作为基准）
    baseline_floor = max(floor_counts, key=floor_counts.get)
    print(f"\n📊 基准类别（删除）: {baseline_floor} ({floor_counts[baseline_floor]}样本)")
    dropped_categories.append(f"楼层-{baseline_floor}")
    
    # 添加到数据集（删除基准类别）
    for col in floor_df.columns:
        if col != baseline_floor:
            df[f'{col}_01'] = floor_df[col]
            print(f"  ✅ 保留: {col}_01")
    
    print(f"\n✅ 已添加{len(floor_df.columns)-1}个楼层变量（6选5）")

# ============================================================
# 2. 电梯处理（2选1，删除最多的）
# ============================================================
print("\n" + "=" * 80)
print("2. 电梯处理（One-Hot编码）")
print("=" * 80)

if '配备电梯' not in df.columns:
    print("⚠️ 未找到'配备电梯'列")
else:
    print(f"原始电梯数据:")
    if df['配备电梯'].notnull().any():
        print(df['配备电梯'].value_counts())
    else:
        print("  全部为空")
    
    def classify_elevator(elevator_str, floor_str):
        """
        判断是否有电梯
        规则：无数据时，楼层>6则有，否则无
        """
        # 如果有明确数据
        if pd.notnull(elevator_str):
            elevator_str = str(elevator_str).strip().lower()
            if elevator_str in ['有', '是', 'yes', '1', 'true', '配备']:
                return '有'
            elif elevator_str in ['无', '否', 'no', '0', 'false', '不配备']:
                return '无'
        
        # 无数据，根据楼层判断
        if pd.notnull(floor_str):
            floor_str = str(floor_str).strip().lower()
            
            # 提取楼层数字
            match = re.search(r'(\d+)', floor_str)
            if match:
                floor_num = int(match.group(1))
                if floor_num > 6:
                    return '有'
                else:
                    return '无'
            
            # 根据关键词判断
            if '高' in floor_str or '顶' in floor_str:
                return '有'
            elif '底' in floor_str or '低' in floor_str or '地下' in floor_str:
                return '无'
        
        # 默认：有电梯
        return '有'
    
    # 应用分类
    elevator_category = df.apply(
        lambda row: classify_elevator(row['配备电梯'], row.get('所在楼层', None)), 
        axis=1
    )
    
    # 统计
    elevator_counts = elevator_category.value_counts()
    print(f"\n电梯分布:")
    for cat, count in elevator_counts.items():
        pct = count / len(df) * 100
        print(f"  {cat:5s}: {count:5d} ({pct:5.2f}%)")
    
    # 找到样本最多的类别（作为基准）
    baseline_elevator = elevator_counts.idxmax()
    print(f"\n📊 基准类别（删除）: {baseline_elevator} ({elevator_counts[baseline_elevator]}样本)")
    dropped_categories.append(f"电梯-{baseline_elevator}")
    
    # 添加变量（只保留非基准类别）
    for cat in elevator_counts.index:
        if cat != baseline_elevator:
            df[f'配备电梯_{cat}_01'] = (elevator_category == cat).astype(int)
            print(f"  ✅ 保留: 配备电梯_{cat}_01")
    
    print(f"\n✅ 已添加{len(elevator_counts)-1}个电梯变量（2选1）")

# ============================================================
# 3. 朝向处理（保留全部4个，不删除）
# ============================================================
print("\n" + "=" * 80)
print("3. 朝向处理（保留全部4个方向）")
print("=" * 80)

if '房屋朝向' not in df.columns:
    print("⚠️ 未找到'房屋朝向'列")
else:
    print(f"原始朝向分布:")
    print(df['房屋朝向'].value_counts().head(10))
    
    def parse_orientation(ori_str):
        """解析朝向，返回东南西北四个方向"""
        result = {
            '朝向_东_01': 0,
            '朝向_南_01': 0,
            '朝向_西_01': 0,
            '朝向_北_01': 0
        }
        
        if pd.isnull(ori_str):
            return result
        
        ori_str = str(ori_str).strip().lower()
        
        if '东' in ori_str or 'east' in ori_str:
            result['朝向_东_01'] = 1
        if '南' in ori_str or 'south' in ori_str:
            result['朝向_南_01'] = 1
        if '西' in ori_str or 'west' in ori_str:
            result['朝向_西_01'] = 1
        if '北' in ori_str or 'north' in ori_str:
            result['朝向_北_01'] = 1
        
        return result
    
    # 应用映射
    orientation_data = df['房屋朝向'].apply(parse_orientation)
    orientation_df = pd.DataFrame(orientation_data.tolist())
    
    # 添加到数据集（全部保留）
    for col in ['朝向_东_01', '朝向_南_01', '朝向_西_01', '朝向_北_01']:
        df[col] = orientation_df[col]
    
    print(f"\n✅ 已添加4个朝向变量（保留全部）")
    print(f"朝向分布:")
    for col in ['朝向_东_01', '朝向_南_01', '朝向_西_01', '朝向_北_01']:
        count = df[col].sum()
        pct = count / len(df) * 100
        print(f"  {col:15s}: {count:5d} ({pct:5.2f}%)")

# ============================================================
# 4. 建筑结构处理（6选5，删除最多的）
# ============================================================
print("\n" + "=" * 80)
print("4. 建筑结构处理（One-Hot编码）")
print("=" * 80)

if '建筑结构' not in df.columns:
    print("⚠️ 未找到'建筑结构'列")
else:
    print(f"原始建筑结构分布:")
    print(df['建筑结构'].value_counts())
    
    def classify_structure(struct_str):
        """
        分类建筑结构
        砖木结构和无数据归为混合结构
        """
        if pd.isnull(struct_str):
            return '混合结构'
        
        struct_str = str(struct_str).strip()
        
        if '钢混' in struct_str or '钢筋混凝土' in struct_str:
            return '钢混结构'
        elif '框架' in struct_str:
            return '框架结构'
        elif '砖混' in struct_str:
            return '砖混结构'
        elif '钢结构' in struct_str or struct_str == '钢':
            return '钢结构'
        elif '砖木' in struct_str or '混合' in struct_str:
            return '混合结构'
        elif '未知' in struct_str or struct_str in ['', 'nan', 'none', '暂无']:
            return '未知结构'
        else:
            # 其他未识别的归为混合结构
            return '混合结构'
    
    # 应用分类
    structure_category = df['建筑结构'].apply(classify_structure)
    
    # 统计
    structure_counts = structure_category.value_counts()
    print(f"\n建筑结构分布:")
    for cat, count in structure_counts.items():
        pct = count / len(df) * 100
        print(f"  {cat:10s}: {count:5d} ({pct:5.2f}%)")
    
    # 找到样本最多的类别（作为基准）
    baseline_structure = structure_counts.idxmax()
    print(f"\n📊 基准类别（删除）: {baseline_structure} ({structure_counts[baseline_structure]}样本)")
    dropped_categories.append(f"建筑结构-{baseline_structure}")
    
    # 添加变量（只保留非基准类别）
    for cat in structure_counts.index:
        if cat != baseline_structure:
            df[f'建筑结构_{cat}_01'] = (structure_category == cat).astype(int)
            print(f"  ✅ 保留: 建筑结构_{cat}_01")
    
    print(f"\n✅ 已添加{len(structure_counts)-1}个建筑结构变量")

# ============================================================
# 5. 装修处理（4选3，删除最多的）
# ============================================================
print("\n" + "=" * 80)
print("5. 装修处理（One-Hot编码）")
print("=" * 80)

if '装修情况' not in df.columns:
    print("⚠️ 未找到'装修情况'列")
else:
    print(f"原始装修分布:")
    print(df['装修情况'].value_counts())
    
    def classify_decoration(deco_str):
        """
        分类装修情况
        无数据归为其他
        """
        if pd.isnull(deco_str):
            return '其他'
        
        deco_str = str(deco_str).strip()
        
        if '精装' in deco_str or '豪华' in deco_str:
            return '精装'
        elif '简装' in deco_str or '中装' in deco_str:
            return '简装'
        elif '毛坯' in deco_str or '清水' in deco_str:
            return '毛坯'
        else:
            return '其他'
    
    # 应用分类
    decoration_category = df['装修情况'].apply(classify_decoration)
    
    # 统计
    decoration_counts = decoration_category.value_counts()
    print(f"\n装修分布:")
    for cat, count in decoration_counts.items():
        pct = count / len(df) * 100
        print(f"  {cat:5s}: {count:5d} ({pct:5.2f}%)")
    
    # 找到样本最多的类别（作为基准）
    baseline_decoration = decoration_counts.idxmax()
    print(f"\n📊 基准类别（删除）: {baseline_decoration} ({decoration_counts[baseline_decoration]}样本)")
    dropped_categories.append(f"装修-{baseline_decoration}")
    
    # 添加变量（只保留非基准类别）
    for cat in decoration_counts.index:
        if cat != baseline_decoration:
            df[f'装修_{cat}_01'] = (decoration_category == cat).astype(int)
            print(f"  ✅ 保留: 装修_{cat}_01")
    
    print(f"\n✅ 已添加{len(decoration_counts)-1}个装修变量")

# ============================================================
# 6. 删除原列
# ============================================================
print("\n" + "=" * 80)
print("6. 删除原列")
print("=" * 80)

columns_to_drop = ['所在楼层', '配备电梯', '房屋朝向', '建筑结构', '装修情况']
existing_drops = [col for col in columns_to_drop if col in df.columns]

if len(existing_drops) > 0:
    print(f"待删除的列: {existing_drops}")
    df = df.drop(existing_drops, axis=1)
    print(f"✅ 已删除{len(existing_drops)}列")
else:
    print(f"⚠️ 没有需要删除的列")

# ============================================================
# 7. 汇总
# ============================================================
print("\n" + "=" * 80)
print("7. 处理汇总")
print("=" * 80)

print(f"处理后数据: {df.shape}")
print(f"处理后列数: {len(df.columns)}")

print(f"\n删除的基准类别（避免多重共线性）:")
for i, cat in enumerate(dropped_categories, 1):
    print(f"  {i}. {cat}")

print(f"\n新增的One-Hot变量:")
new_cols = [col for col in df.columns if col.endswith('_01')]
print(f"共{len(new_cols)}个:")
for i, col in enumerate(sorted(new_cols), 1):
    count = df[col].sum()
    pct = count / len(df) * 100
    print(f"  {i:2d}. {col:30s}: {count:5d} ({pct:5.2f}%)")

print(f"\n删除的原列({len(existing_drops)}个): {existing_drops}")

# 验证01变量
print(f"\n验证One-Hot变量:")
all_valid = True
for col in new_cols:
    unique_vals = sorted(df[col].unique())
    if not set(unique_vals).issubset({0, 1}):
        print(f"  ⚠️ {col}: {unique_vals} (非01值!)")
        all_valid = False

if all_valid:
    print(f"  ✅ 所有变量都是0/1编码")

# 保存
output_path = path_test.replace('.csv', '_onehot.csv')
df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"\n" + "=" * 80)
print(f"✅ 处理完成！")
print(f"=" * 80)
print(f"输出文件: {output_path}")
print(f"数据形状: {df.shape}")
print(f"\n变化总结:")
print(f"   - 删除原列: {len(existing_drops)}个")
print(f"   - 新增变量: {len(new_cols)}个")
print(f"   - 删除基准: {len(dropped_categories)}个类别")
print(f"   - 净增列数: {len(new_cols) - len(existing_drops)}")
print("=" * 80)

预测集特征工程：One-Hot编码（删除最大类别）
原始数据: (34017, 21)
原始列数: 21

1. 楼层处理（One-Hot编码）
样例数据:
所在楼层
中楼层 (共6层)     2209
高楼层 (共6层)     2000
低楼层 (共6层)     1470
中楼层 (共18层)     950
高楼层 (共18层)     880
低楼层 (共18层)     838
中楼层 (共5层)      828
高楼层 (共33层)     723
中楼层 (共11层)     697
低楼层 (共33层)     686
Name: count, dtype: int64

楼层分布:
  地下室       :    28 ( 0.08%)
  底层        :   623 ( 1.83%)
  低楼层       :  9302 (27.35%)
  中楼层       : 13214 (38.85%)
  高楼层       : 10130 (29.78%)
  顶层        :   720 ( 2.12%)

📊 基准类别（删除）: 中楼层 (13214样本)
  ✅ 保留: 地下室_01
  ✅ 保留: 底层_01
  ✅ 保留: 低楼层_01
  ✅ 保留: 高楼层_01
  ✅ 保留: 顶层_01

✅ 已添加5个楼层变量（6选5）

2. 电梯处理（One-Hot编码）
原始电梯数据:
配备电梯
有    21713
无     8212
Name: count, dtype: int64

电梯分布:
  有    : 24343 (71.56%)
  无    :  9674 (28.44%)

📊 基准类别（删除）: 有 (24343样本)
  ✅ 保留: 配备电梯_无_01

✅ 已添加1个电梯变量（2选1）

3. 朝向处理（保留全部4个方向）
原始朝向分布:
房屋朝向
南      13239
南 北     9887
东南      3305
东       1250
西南      1212
北       1115
西        582
东北       524
西北       507
东 西      435
Name: count, dtype: int64

✅ 已添加4个朝向变量（保留全

In [11]:
# -*- coding: utf-8 -*-
"""
预测集处理：计算房龄
支持建筑年代区间格式：
- 区间格式（2012-2021）→ 取最后一年（2021）
- 单个年份（2022）→ 直接使用（2022）
"""

import pandas as pd
import numpy as np
import re
from datetime import datetime

# ===================== 配置 =====================
path_test = r"H:\HW\ruc_Class25Q2_test_price_clean_with_rooms_onehot.csv"

# 缺失值默认房龄
DEFAULT_AGE = 11
# =================================================

print("=" * 80)
print("计算房龄（区间取最后一年）")
print("=" * 80)

# 读取数据
df = pd.read_csv(path_test, encoding='utf-8')
print(f"数据形状: {df.shape}")
print(f"列数: {len(df.columns)}")

# 检查必要的列
required_cols = ['交易时间', '建筑年代']
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    print(f"❌ 缺少必要的列: {missing_cols}")
    exit(1)

print(f"✅ 找到必要的列: {required_cols}")

# ============================================================
# 1. 查看原始数据
# ============================================================
print("\n" + "=" * 80)
print("原始数据预览")
print("=" * 80)

print(f"\n交易时间样例（前10条）:")
for i, val in enumerate(df['交易时间'].head(10), 1):
    print(f"  {i:2d}. {val}")
print(f"缺失值: {df['交易时间'].isnull().sum()} ({df['交易时间'].isnull().sum()/len(df)*100:.2f}%)")

print(f"\n建筑年代样例（前20条）:")
for i, val in enumerate(df['建筑年代'].head(20), 1):
    print(f"  {i:2d}. {val}")
print(f"缺失值: {df['建筑年代'].isnull().sum()} ({df['建筑年代'].isnull().sum()/len(df)*100:.2f}%)")

# 统计建筑年代格式
print(f"\n建筑年代格式分析（去重后前30个）:")
unique_formats = df['建筑年代'].dropna().unique()[:30]
for i, fmt in enumerate(unique_formats, 1):
    print(f"  {i:2d}. {fmt}")

# ============================================================
# 2. 提取交易年份
# ============================================================
print("\n" + "=" * 80)
print("提取交易年份")
print("=" * 80)

def extract_transaction_year(date_str):
    """
    从交易时间提取年份
    支持格式：
    - 2023-01-15
    - 2023/01/15
    - 20230115
    - 2023年1月15日
    - 2023
    """
    if pd.isnull(date_str):
        return None
    
    date_str = str(date_str).strip()
    
    # 空值判断
    if date_str in ['', 'nan', 'NaN', 'None', '无', '未知', '暂无']:
        return None
    
    # 尝试匹配年份（4位数字，优先匹配20xx和19xx）
    patterns = [
        r'(20[0-2]\d)',           # 2000-2029
        r'(19[5-9]\d)',           # 1950-1999
        r'(\d{4})',               # 任意4位数字
    ]
    
    for pattern in patterns:
        match = re.search(pattern, date_str)
        if match:
            year = int(match.group(1))
            if 1950 <= year <= 2025:
                return year
    
    # 尝试标准日期格式解析
    try:
        parsed_date = pd.to_datetime(date_str, errors='coerce')
        if pd.notnull(parsed_date):
            year = parsed_date.year
            if 1950 <= year <= 2025:
                return year
    except:
        pass
    
    return None

df['交易年份'] = df['交易时间'].apply(extract_transaction_year)

# 统计
valid_transaction_years = df['交易年份'].dropna()
missing_transaction = df['交易年份'].isnull().sum()

print(f"提取结果:")
print(f"  ✅ 成功提取: {len(valid_transaction_years)} ({len(valid_transaction_years)/len(df)*100:.2f}%)")
print(f"  ❌ 提取失败: {missing_transaction} ({missing_transaction/len(df)*100:.2f}%)")

if len(valid_transaction_years) > 0:
    print(f"\n交易年份统计:")
    print(f"  最小值: {valid_transaction_years.min():.0f}")
    print(f"  最大值: {valid_transaction_years.max():.0f}")
    print(f"  均值: {valid_transaction_years.mean():.1f}")
    print(f"  中位数: {valid_transaction_years.median():.0f}")
    
    print(f"\n交易年份分布:")
    year_dist = valid_transaction_years.value_counts().sort_index()
    for year, count in year_dist.items():
        pct = count / len(valid_transaction_years) * 100
        print(f"  {year:.0f}: {count:5d} ({pct:5.2f}%)")

# ============================================================
# 3. 提取建筑年代（支持区间，取最后一年）
# ============================================================
print("\n" + "=" * 80)
print("提取建筑年代（区间取最后一年）")
print("=" * 80)

def extract_build_year_last(year_str):
    """
    从建筑年代提取年份
    【关键】支持区间格式，取最后一年
    
    支持格式：
    - 2012-2021 → 2021
    - 2012~2021 → 2021
    - 2012—2021 → 2021
    - 2012至2021 → 2021
    - 2022 → 2022
    - 2022年 → 2022
    """
    if pd.isnull(year_str):
        return None
    
    year_str = str(year_str).strip()
    
    # 空值判断
    if year_str in ['', 'nan', 'NaN', 'None', '无', '未知', '暂无', '待定']:
        return None
    
    # 【关键】处理区间：找出所有4位年份，取最大值（最后一年）
    # 匹配所有符合条件的年份
    all_years = re.findall(r'(20[0-2]\d|19[5-9]\d)', year_str)
    
    if len(all_years) >= 2:
        # 区间格式：取最后一个（最大的）
        years = [int(y) for y in all_years if 1950 <= int(y) <= 2025]
        if years:
            last_year = max(years)
            return last_year
    elif len(all_years) == 1:
        # 单个年份
        year = int(all_years[0])
        if 1950 <= year <= 2025:
            return year
    
    # 尝试匹配"XX年代"格式
    match = re.search(r'(\d{2})年代', year_str)
    if match:
        decade = int(match.group(1))
        if decade >= 50:
            year = 1900 + decade
        else:
            year = 2000 + decade
        if 1950 <= year <= 2025:
            return year
    
    # 尝试直接转换为数字
    try:
        year = int(float(year_str))
        if 1950 <= year <= 2025:
            return year
    except:
        pass
    
    return None

# 测试示例
print(f"\n提取规则测试:")
test_cases = [
    '2012-2021',
    '2012~2021', 
    '2012—2021',
    '2012至2021年',
    '2022',
    '2022年',
    '90年代',
    '2010-2015年建',
]
for test in test_cases:
    result = extract_build_year_last(test)
    print(f"  '{test}' → {result}")

# 应用提取
df['建筑年份'] = df['建筑年代'].apply(extract_build_year_last)

# 统计
valid_build_years = df['建筑年份'].dropna()
missing_build = df['建筑年份'].isnull().sum()

print(f"\n提取结果:")
print(f"  ✅ 成功提取: {len(valid_build_years)} ({len(valid_build_years)/len(df)*100:.2f}%)")
print(f"  ❌ 提取失败: {missing_build} ({missing_build/len(df)*100:.2f}%)")

if len(valid_build_years) > 0:
    print(f"\n建筑年份统计:")
    print(f"  最小值: {valid_build_years.min():.0f}")
    print(f"  最大值: {valid_build_years.max():.0f}")
    print(f"  均值: {valid_build_years.mean():.1f}")
    print(f"  中位数: {valid_build_years.median():.0f}")
    
    print(f"\n建筑年份分布（Top 20）:")
    build_dist = valid_build_years.value_counts().sort_index()
    for year, count in build_dist.tail(20).items():
        pct = count / len(valid_build_years) * 100
        print(f"  {year:.0f}: {count:5d} ({pct:5.2f}%)")

# 显示一些提取样例
print(f"\n提取样例（前20条）:")
sample_df = df[['建筑年代', '建筑年份']].head(20)
for idx, row in sample_df.iterrows():
    orig = row['建筑年代']
    extracted = row['建筑年份']
    status = '✅' if pd.notnull(extracted) else '❌'
    print(f"  {status} '{orig}' → {extracted}")

# ============================================================
# 4. 计算房龄
# ============================================================
print("\n" + "=" * 80)
print("计算房龄")
print("=" * 80)

# 统计异常情况
error_log = []

def calculate_age(transaction_year, build_year, index):
    """
    计算房龄
    房龄 = 交易年份 - 建筑年份
    """
    if pd.notnull(transaction_year) and pd.notnull(build_year):
        age = transaction_year - build_year
        
        # 异常值处理
        if age < 0:
            error_log.append(f"第{index}行: 房龄为负 (交易={transaction_year}, 建筑={build_year}, 房龄={age})")
            return None
        elif age > 100:
            error_log.append(f"第{index}行: 房龄过大 (交易={transaction_year}, 建筑={build_year}, 房龄={age})")
            return None
        else:
            return age
    else:
        return None

# 计算房龄
df['房龄_raw'] = df.apply(
    lambda row: calculate_age(row['交易年份'], row['建筑年份'], row.name), 
    axis=1
)

# 显示异常日志
if len(error_log) > 0:
    print(f"\n⚠️ 发现{len(error_log)}个异常:")
    for log in error_log[:10]:  # 只显示前10个
        print(f"  {log}")
    if len(error_log) > 10:
        print(f"  ... 还有{len(error_log)-10}个异常")

# 统计计算结果
valid_ages = df['房龄_raw'].dropna()
missing_ages = df['房龄_raw'].isnull().sum()

print(f"\n计算结果:")
print(f"  ✅ 成功计算: {len(valid_ages)} ({len(valid_ages)/len(df)*100:.2f}%)")
print(f"  ❌ 计算失败: {missing_ages} ({missing_ages/len(df)*100:.2f}%)")

if len(valid_ages) > 0:
    print(f"\n房龄统计（填充前）:")
    print(df['房龄_raw'].describe())
    
    print(f"\n房龄分布（Top 30）:")
    age_dist = df['房龄_raw'].value_counts().sort_index()
    for age, count in age_dist.head(30).items():
        pct = count / len(valid_ages) * 100
        bar = '█' * int(pct)
        print(f"  {age:3.0f}年: {count:5d} ({pct:5.2f}%) {bar}")

# ============================================================
# 5. 填充缺失值
# ============================================================
print("\n" + "=" * 80)
print(f"填充缺失值（默认房龄={DEFAULT_AGE}）")
print("=" * 80)

# 填充缺失值
df['房龄'] = df['房龄_raw'].fillna(DEFAULT_AGE)

# 统计填充情况
filled_count = df['房龄_raw'].isnull().sum()
print(f"使用默认值({DEFAULT_AGE})的记录: {filled_count} ({filled_count/len(df)*100:.2f}%)")

print(f"\n房龄统计（填充后）:")
print(df['房龄'].describe())

print(f"\n房龄分布（填充后，Top 30）:")
age_dist_filled = df['房龄'].value_counts().sort_index()
for age, count in age_dist_filled.head(30).items():
    pct = count / len(df) * 100
    bar = '█' * int(pct / 2)
    marker = ' 🎯' if age == DEFAULT_AGE else ''
    print(f"  {age:3.0f}年: {count:5d} ({pct:5.2f}%) {bar}{marker}")

# 显示计算样例
print(f"\n计算样例（前20条）:")
sample_calc = df[[ '交易年份', '建筑年代', '建筑年份', '房龄']].head(20)
for idx, row in sample_calc.iterrows():
    trans = row['交易年份']
    build = row['建筑年份']
    age = row['房龄']
    if pd.notnull(trans) and pd.notnull(build):
        print(f"  {idx+1:2d}. 交易{trans:.0f} - 建筑{build:.0f} = {age:.0f}年")
    else:
        print(f"  {idx+1:2d}. 缺失数据 → {age:.0f}年 (默认)")

# ============================================================
# 6. 删除临时列和原列
# ============================================================
print("\n" + "=" * 80)
print("删除原列和临时列")
print("=" * 80)

columns_to_drop = ['建筑年代', '交易年份', '建筑年份', '房龄_raw']
existing_drops = [col for col in columns_to_drop if col in df.columns]

if len(existing_drops) > 0:
    print(f"待删除的列: {existing_drops}")
    df = df.drop(existing_drops, axis=1)
    print(f"✅ 已删除{len(existing_drops)}列")
else:
    print(f"⚠️ 没有需要删除的列")

# ============================================================
# 7. 验证和保存
# ============================================================
print("\n" + "=" * 80)
print("验证和保存")
print("=" * 80)

# 验证房龄列
print(f"房龄列验证:")
print(f"  数据类型: {df['房龄'].dtype}")
print(f"  缺失值: {df['房龄'].isnull().sum()}")
print(f"  最小值: {df['房龄'].min():.1f}")
print(f"  最大值: {df['房龄'].max():.1f}")
print(f"  均值: {df['房龄'].mean():.2f}")
print(f"  中位数: {df['房龄'].median():.1f}")

# 检查异常值
abnormal_ages = df[(df['房龄'] < 0) | (df['房龄'] > 100)]
if len(abnormal_ages) > 0:
    print(f"  ⚠️ 异常值: {len(abnormal_ages)}条 (房龄<0或>100)")
    print(abnormal_ages[['房龄']].head())
else:
    print(f"  ✅ 无异常值")

print(f"\n处理后数据:")
print(f"  数据形状: {df.shape}")
print(f"  列数: {len(df.columns)}")

# 显示最终列名
print(f"\n最终列名({len(df.columns)}个):")
for i, col in enumerate(df.columns, 1):
    marker = '🆕' if col == '房龄' else '  '
    print(f"  {i:2d}. {marker} {col}")

# 保存
output_path = path_test.replace('.csv', '_with_age.csv')
df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"\n" + "=" * 80)
print(f"✅ 处理完成！")
print(f"=" * 80)
print(f"输出文件: {output_path}")
print(f"数据形状: {df.shape}")
print(f"\n变化总结:")
print(f"   ❌ 删除列: 交易时间、建筑年代")
print(f"   ✅ 新增列: 房龄")
print(f"   📊 提取规则: 区间取最后一年")
print(f"   📊 默认值: {DEFAULT_AGE}年")
print(f"   📊 使用默认值: {filled_count}条 ({filled_count/len(df)*100:.2f}%)")
print(f"   📊 成功计算: {len(valid_ages)}条 ({len(valid_ages)/len(df)*100:.2f}%)")
print("=" * 80)

计算房龄（区间取最后一年）
数据形状: (34017, 34)
列数: 34
✅ 找到必要的列: ['交易时间', '建筑年代']

原始数据预览

交易时间样例（前10条）:
   1. 2025-02-21
   2. 2025-01-17
   3. 2025-04-03
   4. 2025-01-31
   5. 2025-01-22
   6. 2025-01-24
   7. 2025-04-05
   8. 2025-01-23
   9. 2025-01-20
  10. 2025-01-24
缺失值: 0 (0.00%)

建筑年代样例（前20条）:
   1. 2002-2006年
   2. 2007-2009年
   3. 1999-2001年
   4. 2004-2008年
   5. 1993-1997年
   6. 2003-2006年
   7. 2014-2019年
   8. 1993-2009年
   9. 2018-2022年
  10. 2005-2006年
  11. 1994-2003年
  12. 1999年
  13. 2009-2013年
  14. 2007-2009年
  15. 2008-2014年
  16. 1950-1994年
  17. 2006-2009年
  18. 2005-2008年
  19. 2001-2003年
  20. 2007-2009年
缺失值: 9406 (27.65%)

建筑年代格式分析（去重后前30个）:
   1. 2002-2006年
   2. 2007-2009年
   3. 1999-2001年
   4. 2004-2008年
   5. 1993-1997年
   6. 2003-2006年
   7. 2014-2019年
   8. 1993-2009年
   9. 2018-2022年
  10. 2005-2006年
  11. 1994-2003年
  12. 1999年
  13. 2009-2013年
  14. 2008-2014年
  15. 1950-1994年
  16. 2006-2009年
  17. 2005-2008年
  18. 2001-2003年
  19. 2000-2003年
  20. 1981-1989年
  

In [12]:
# -*- coding: utf-8 -*-
"""
预测集处理：房屋总数和楼栋总数
- 去除单位（户、栋）
- 填充空白值为中位数
- 最终输出：ruc_Class25Q2_test_price1.csv
"""

import pandas as pd
import numpy as np
import re

# ===================== 配置 =====================
path_test = r"H:\HW\ruc_Class25Q2_test_price_clean_with_rooms_onehot_with_age.csv"

# 输出文件名
output_filename = "ruc_Class25Q2_test_price1.csv"

# 填充的中位数
MEDIAN_HOUSE_COUNT = 1372.0
MEDIAN_BUILDING_COUNT = 15.0
# =================================================

print("=" * 80)
print("处理房屋总数和楼栋总数")
print("=" * 80)

# 读取数据
df = pd.read_csv(path_test, encoding='utf-8')
print(f"原始数据: {df.shape}")
print(f"列数: {len(df.columns)}")

# 检查必要的列
required_cols = ['房屋总数', '楼栋总数']
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    print(f"❌ 缺少必要的列: {missing_cols}")
    print(f"现有列: {df.columns.tolist()}")
    exit(1)

print(f"✅ 找到必要的列: {required_cols}")

# ============================================================
# 1. 处理房屋总数
# ============================================================
print("\n" + "=" * 80)
print("1. 处理房屋总数")
print("=" * 80)

print(f"原始数据样例（前20条）:")
for i, val in enumerate(df['房屋总数'].head(20), 1):
    print(f"  {i:2d}. {val}")

print(f"\n原始缺失值: {df['房屋总数'].isnull().sum()} ({df['房屋总数'].isnull().sum()/len(df)*100:.2f}%)")

def clean_house_count(value):
    """
    清洗房屋总数
    - 去除单位"户"
    - 去除空格、逗号等
    - 转换为数字
    """
    if pd.isnull(value):
        return None
    
    value_str = str(value).strip()
    
    # 空值判断
    if value_str in ['', 'nan', 'NaN', 'None', '无', '未知', '暂无']:
        return None
    
    # 去除单位和其他字符
    value_str = value_str.replace('户', '')
    value_str = value_str.replace('个', '')
    value_str = value_str.replace('套', '')
    value_str = value_str.replace(',', '')
    value_str = value_str.replace('，', '')
    value_str = value_str.replace(' ', '')
    
    # 尝试提取数字
    match = re.search(r'(\d+\.?\d*)', value_str)
    if match:
        try:
            num = float(match.group(1))
            return num
        except:
            return None
    
    # 尝试直接转换
    try:
        num = float(value_str)
        return num
    except:
        return None

# 应用清洗
df['房屋总数_clean'] = df['房屋总数'].apply(clean_house_count)

# 统计清洗结果
valid_house = df['房屋总数_clean'].dropna()
missing_house = df['房屋总数_clean'].isnull().sum()

print(f"\n清洗结果:")
print(f"  ✅ 成功提取: {len(valid_house)} ({len(valid_house)/len(df)*100:.2f}%)")
print(f"  ❌ 提取失败/缺失: {missing_house} ({missing_house/len(df)*100:.2f}%)")

if len(valid_house) > 0:
    print(f"\n房屋总数统计（填充前）:")
    print(f"  最小值: {valid_house.min():.0f}")
    print(f"  最大值: {valid_house.max():.0f}")
    print(f"  均值: {valid_house.mean():.2f}")
    print(f"  中位数: {valid_house.median():.0f}")
    print(f"  标准差: {valid_house.std():.2f}")

# 填充缺失值
print(f"\n填充缺失值（中位数={MEDIAN_HOUSE_COUNT}）...")
df['房屋总数'] = df['房屋总数_clean'].fillna(MEDIAN_HOUSE_COUNT)

filled_house = df['房屋总数_clean'].isnull().sum()
print(f"  使用中位数填充: {filled_house}条 ({filled_house/len(df)*100:.2f}%)")

print(f"\n房屋总数统计（填充后）:")
print(df['房屋总数'].describe())

# 显示处理样例
print(f"\n处理样例（前20条）:")
sample_house = df[['房屋总数']].head(20)
for idx, row in sample_house.iterrows():
    val = row['房屋总数']
    marker = '🎯' if val == MEDIAN_HOUSE_COUNT else '  '
    print(f"  {idx+1:2d}. {marker} {val:.0f}")

# ============================================================
# 2. 处理楼栋总数
# ============================================================
print("\n" + "=" * 80)
print("2. 处理楼栋总数")
print("=" * 80)

print(f"原始数据样例（前20条）:")
for i, val in enumerate(df['楼栋总数'].head(20), 1):
    print(f"  {i:2d}. {val}")

print(f"\n原始缺失值: {df['楼栋总数'].isnull().sum()} ({df['楼栋总数'].isnull().sum()/len(df)*100:.2f}%)")

def clean_building_count(value):
    """
    清洗楼栋总数
    - 去除单位"栋"
    - 去除空格、逗号等
    - 转换为数字
    """
    if pd.isnull(value):
        return None
    
    value_str = str(value).strip()
    
    # 空值判断
    if value_str in ['', 'nan', 'NaN', 'None', '无', '未知', '暂无']:
        return None
    
    # 去除单位和其他字符
    value_str = value_str.replace('栋', '')
    value_str = value_str.replace('幢', '')
    value_str = value_str.replace('座', '')
    value_str = value_str.replace('个', '')
    value_str = value_str.replace(',', '')
    value_str = value_str.replace('，', '')
    value_str = value_str.replace(' ', '')
    
    # 尝试提取数字
    match = re.search(r'(\d+\.?\d*)', value_str)
    if match:
        try:
            num = float(match.group(1))
            return num
        except:
            return None
    
    # 尝试直接转换
    try:
        num = float(value_str)
        return num
    except:
        return None

# 应用清洗
df['楼栋总数_clean'] = df['楼栋总数'].apply(clean_building_count)

# 统计清洗结果
valid_building = df['楼栋总数_clean'].dropna()
missing_building = df['楼栋总数_clean'].isnull().sum()

print(f"\n清洗结果:")
print(f"  ✅ 成功提取: {len(valid_building)} ({len(valid_building)/len(df)*100:.2f}%)")
print(f"  ❌ 提取失败/缺失: {missing_building} ({missing_building/len(df)*100:.2f}%)")

if len(valid_building) > 0:
    print(f"\n楼栋总数统计（填充前）:")
    print(f"  最小值: {valid_building.min():.0f}")
    print(f"  最大值: {valid_building.max():.0f}")
    print(f"  均值: {valid_building.mean():.2f}")
    print(f"  中位数: {valid_building.median():.0f}")
    print(f"  标准差: {valid_building.std():.2f}")

# 填充缺失值
print(f"\n填充缺失值（中位数={MEDIAN_BUILDING_COUNT}）...")
df['楼栋总数'] = df['楼栋总数_clean'].fillna(MEDIAN_BUILDING_COUNT)

filled_building = df['楼栋总数_clean'].isnull().sum()
print(f"  使用中位数填充: {filled_building}条 ({filled_building/len(df)*100:.2f}%)")

print(f"\n楼栋总数统计（填充后）:")
print(df['楼栋总数'].describe())

# 显示处理样例
print(f"\n处理样例（前20条）:")
sample_building = df[['楼栋总数']].head(20)
for idx, row in sample_building.iterrows():
    val = row['楼栋总数']
    marker = '🎯' if val == MEDIAN_BUILDING_COUNT else '  '
    print(f"  {idx+1:2d}. {marker} {val:.0f}")

# ============================================================
# 3. 删除临时列
# ============================================================
print("\n" + "=" * 80)
print("3. 删除临时列")
print("=" * 80)

temp_cols = ['房屋总数_clean', '楼栋总数_clean']
existing_temp = [col for col in temp_cols if col in df.columns]

if len(existing_temp) > 0:
    print(f"待删除的临时列: {existing_temp}")
    df = df.drop(existing_temp, axis=1)
    print(f"✅ 已删除{len(existing_temp)}列")

# ============================================================
# 4. 验证和保存
# ============================================================
print("\n" + "=" * 80)
print("4. 验证和保存")
print("=" * 80)

# 验证数据
print(f"数据验证:")
print(f"  房屋总数 - 缺失值: {df['房屋总数'].isnull().sum()}")
print(f"  房屋总数 - 最小值: {df['房屋总数'].min():.0f}")
print(f"  房屋总数 - 最大值: {df['房屋总数'].max():.0f}")
print(f"  房屋总数 - 均值: {df['房屋总数'].mean():.2f}")
print(f"  房屋总数 - 中位数: {df['房屋总数'].median():.0f}")

print(f"\n  楼栋总数 - 缺失值: {df['楼栋总数'].isnull().sum()}")
print(f"  楼栋总数 - 最小值: {df['楼栋总数'].min():.0f}")
print(f"  楼栋总数 - 最大值: {df['楼栋总数'].max():.0f}")
print(f"  楼栋总数 - 均值: {df['楼栋总数'].mean():.2f}")
print(f"  楼栋总数 - 中位数: {df['楼栋总数'].median():.0f}")

# 异常值检查
abnormal_house = df[(df['房屋总数'] < 0) | (df['房屋总数'] > 100000)]
abnormal_building = df[(df['楼栋总数'] < 0) | (df['楼栋总数'] > 1000)]

if len(abnormal_house) > 0:
    print(f"\n  ⚠️ 房屋总数异常值: {len(abnormal_house)}条")
else:
    print(f"\n  ✅ 房屋总数无异常值")

if len(abnormal_building) > 0:
    print(f"  ⚠️ 楼栋总数异常值: {len(abnormal_building)}条")
else:
    print(f"  ✅ 楼栋总数无异常值")

print(f"\n处理后数据:")
print(f"  数据形状: {df.shape}")
print(f"  列数: {len(df.columns)}")

# 显示所有列名
print(f"\n最终列名({len(df.columns)}个):")
for i, col in enumerate(df.columns, 1):
    marker = '📊' if col in ['房屋总数', '楼栋总数'] else '  '
    print(f"  {i:2d}. {marker} {col}")

# 保存到指定目录
import os
output_dir = r"H:\HW"
output_path = os.path.join(output_dir, output_filename)

df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"\n" + "=" * 80)
print(f"✅ 处理完成！")
print(f"=" * 80)
print(f"输出文件: {output_path}")
print(f"数据形状: {df.shape}")
print(f"\n变化总结:")
print(f"   📊 房屋总数:")
print(f"      - 去除单位: 户、个、套")
print(f"      - 填充中位数: {MEDIAN_HOUSE_COUNT}")
print(f"      - 填充数量: {filled_house}条 ({filled_house/len(df)*100:.2f}%)")
print(f"\n   📊 楼栋总数:")
print(f"      - 去除单位: 栋、幢、座")
print(f"      - 填充中位数: {MEDIAN_BUILDING_COUNT}")
print(f"      - 填充数量: {filled_building}条 ({filled_building/len(df)*100:.2f}%)")
print(f"\n   💾 输出文件: {output_filename}")
print("=" * 80)

处理房屋总数和楼栋总数
原始数据: (34017, 34)
列数: 34
✅ 找到必要的列: ['房屋总数', '楼栋总数']

1. 处理房屋总数
原始数据样例（前20条）:
   1. 458户
   2. 3465户
   3. 144户
   4. 320户
   5. 340户
   6. 575户
   7. 238户
   8. 2831户
   9. 386户
  10. 559户
  11. 3433户
  12. 320户
  13. 1021户
  14. 1887户
  15. 2261户
  16. 6830户
  17. 1675户
  18. 775户
  19. 712户
  20. 1887户

原始缺失值: 3715 (10.92%)

清洗结果:
  ✅ 成功提取: 30302 (89.08%)
  ❌ 提取失败/缺失: 3715 (10.92%)

房屋总数统计（填充前）:
  最小值: 1
  最大值: 12669
  均值: 1962.30
  中位数: 1409
  标准差: 1924.48

填充缺失值（中位数=1372.0）...
  使用中位数填充: 3715条 (10.92%)

房屋总数统计（填充后）:
count    34017.000000
mean      1897.833671
std       1825.664093
min          1.000000
25%        800.000000
50%       1372.000000
75%       2339.000000
max      12669.000000
Name: 房屋总数, dtype: float64

处理样例（前20条）:
   1.    458
   2.    3465
   3.    144
   4.    320
   5.    340
   6.    575
   7.    238
   8.    2831
   9.    386
  10.    559
  11.    3433
  12.    320
  13.    1021
  14.    1887
  15.    2261
  16.    6830
  17.    1675
  18.    775
  19

In [13]:
# -*- coding: utf-8 -*-
import pandas as pd
import re

# 读取
path = r"H:\HW\ruc_Class25Q2_test_price1.csv"
df = pd.read_csv(path, encoding='utf-8')

print(f"原始: {df.shape}")

# 清洗函数
def clean_num(x, remove_chars=[]):
    if pd.isnull(x):
        return None
    s = str(x).strip()
    for char in remove_chars + [',', '，', ' ']:
        s = s.replace(char, '')
    try:
        match = re.search(r'(\d+\.?\d*)', s)
        return float(match.group(1)) if match else None
    except:
        return None

# 处理绿化率（去除%）
df['绿 化 率'] = df['绿 化 率'].apply(lambda x: clean_num(x, ['%', '％']))
df['绿 化 率'] = df['绿 化 率'].fillna(34.0)

# 处理容积率
df['容 积 率'] = df['容 积 率'].apply(lambda x: clean_num(x))
df['容 积 率'] = df['容 积 率'].fillna(2.5)

print(f"处理后: {df.shape}")
print(f"绿化率: 缺失={df['绿 化 率'].isnull().sum()}, 中位数={df['绿 化 率'].median():.1f}")
print(f"容积率: 缺失={df['容 积 率'].isnull().sum()}, 中位数={df['容 积 率'].median():.1f}")

# 保存
output = r"H:\HW\ruc_Class25Q2_test_price12.csv"
df.to_csv(output, index=False, encoding='utf-8-sig')
print(f"✅ 已保存: {output}")

原始: (34017, 34)
处理后: (34017, 34)
绿化率: 缺失=0, 中位数=34.0
容积率: 缺失=0, 中位数=2.5
✅ 已保存: H:\HW\ruc_Class25Q2_test_price12.csv


In [14]:
# -*- coding: utf-8 -*-
import pandas as pd
import re

# 读取
path = r"H:\HW\ruc_Class25Q2_test_price12.csv"
df = pd.read_csv(path, encoding='utf-8')

print(f"原始: {df.shape}")

# 清洗物业费
def clean_property_fee(x):
    if pd.isnull(x):
        return None
    s = str(x).strip()
    # 去除单位
    for unit in ['元/月/㎡', '元/㎡/月', '元/平米/月', '元/平方米/月', '元', '㎡', '/', '月', '平米', '平方米']:
        s = s.replace(unit, '')
    s = s.replace(',', '').replace('，', '').replace(' ', '')
    
    # 提取所有数字
    nums = re.findall(r'(\d+\.?\d*)', s)
    if len(nums) == 0:
        return None
    elif len(nums) == 1:
        # 单个值
        return float(nums[0])
    else:
        # 区间：取平均值
        values = [float(n) for n in nums]
        avg = sum(values) / len(values)
        return avg

# 处理物业费
print("处理物业费...")
print(f"样例数据（前10条）:")
for i, val in enumerate(df['物 业 费'].head(10), 1):
    cleaned = clean_property_fee(val)
    print(f"  {i:2d}. 原始: {val} -> 清洗后: {cleaned}")

df['物 业 费'] = df['物 业 费'].apply(clean_property_fee)
df['物 业 费'] = df['物 业 费'].fillna(1.9)

print(f"\n处理后: {df.shape}")
print(f"物业费统计: 缺失={df['物 业 费'].isnull().sum()}, 最小={df['物 业 费'].min():.2f}, 最大={df['物 业 费'].max():.2f}, 中位数={df['物 业 费'].median():.2f}")

# 保存
output = r"H:\HW\ruc_Class25Q2_test_price123.csv"
df.to_csv(output, index=False, encoding='utf-8-sig')
print(f"\n✅ 已保存: {output}")

原始: (34017, 34)
处理物业费...
样例数据（前10条）:
   1. 原始: 2.8元/月/㎡ -> 清洗后: 2.8
   2. 原始: 1.8元/月/㎡ -> 清洗后: 1.8
   3. 原始: 1.2-1.96元/月/㎡ -> 清洗后: 1.58
   4. 原始: 0.5元/月/㎡ -> 清洗后: 0.5
   5. 原始: 1.61元/月/㎡ -> 清洗后: 1.61
   6. 原始: 0.5-1.5元/月/㎡ -> 清洗后: 1.0
   7. 原始: 0.86-7.8元/月/㎡ -> 清洗后: 4.33
   8. 原始: 0.68-4元/月/㎡ -> 清洗后: 2.34
   9. 原始: 4.55元/月/㎡ -> 清洗后: 4.55
  10. 原始: 2.2-2.5元/月/㎡ -> 清洗后: 2.35

处理后: (34017, 34)
物业费统计: 缺失=0, 最小=0.20, 最大=76.45, 中位数=1.90

✅ 已保存: H:\HW\ruc_Class25Q2_test_price123.csv


In [15]:
# -*- coding: utf-8 -*-
import pandas as pd

# 读取
path = r"H:\HW\ruc_Class25Q2_test_price123.csv"
df = pd.read_csv(path, encoding='utf-8')

print(f"原始: {df.shape}")

# 查看城市列的值
print(f"\n城市分布:")
print(df['城市'].value_counts().sort_index())

# 获取所有唯一城市
cities = sorted(df['城市'].unique())

# 创建one-hot编码（以城市2为基准，跳过它）
for city in cities:
    if city == 2:
        continue  # 跳过城市2（基准）
    col_name = f'city_{city:02d}'  # 格式化为两位数字
    df[col_name] = (df['城市'] == city).astype(int)

# 获取生成的列名
city_cols = [f'city_{c:02d}' for c in cities if c != 2]

print(f"\n生成的虚拟变量列:")
for col in city_cols:
    count = df[col].sum()
    print(f"  - {col}: {count} 条记录")

print(f"\n处理后: {df.shape}")
print(f"新增列数: {len(city_cols)}")

# 保存
output = r"H:\HW\ruc_Class25Q2_test_price1234.csv"
df.to_csv(output, index=False, encoding='utf-8-sig')
print(f"\n✅ 已保存: {output}")

原始: (34017, 34)

城市分布:
城市
0     7057
1     1985
2     5431
3     3961
4     5602
5     1057
6      933
7      879
8     2051
9     1182
10    3694
11     185
Name: count, dtype: int64

生成的虚拟变量列:
  - city_00: 7057 条记录
  - city_01: 1985 条记录
  - city_03: 3961 条记录
  - city_04: 5602 条记录
  - city_05: 1057 条记录
  - city_06: 933 条记录
  - city_07: 879 条记录
  - city_08: 2051 条记录
  - city_09: 1182 条记录
  - city_10: 3694 条记录
  - city_11: 185 条记录

处理后: (34017, 45)
新增列数: 11

✅ 已保存: H:\HW\ruc_Class25Q2_test_price1234.csv


In [19]:
# -*- coding: utf-8 -*-
import pandas as pd

# 读取
path = r"H:\HW\ruc_Class25Q2_test_price1234.csv"
df = pd.read_csv(path, encoding='utf-8')

print(f"原始: {df.shape}")

# 查看交易时间列
print(f"\n交易时间样例:")
print(df['交易时间'].head(10))

# 转换为日期格式（格式：年/月/日）
df['交易时间'] = pd.to_datetime(df['交易时间'], format='%Y/%m/%d', errors='coerce')

# 提取年份和月份
df['year'] = df['交易时间'].dt.year
df['month'] = df['交易时间'].dt.month

print(f"\n年份分布:")
print(df['year'].value_counts().sort_index())

print(f"\n月份分布:")
print(df['month'].value_counts().sort_index())

# 创建年份one-hot（2019-2025，以2018为基准）
for year in range(2019, 2026):  # 2019到2025
    col_name = f'year_{year}'
    df[col_name] = (df['year'] == year).astype(int)

# 创建月份one-hot（2-12月，以1月为基准）
for month in range(2, 13):  # 2到12月
    col_name = f'month_{month}'
    df[col_name] = (df['month'] == month).astype(int)

# 删除临时列
df = df.drop(['year', 'month'], axis=1)

# 生成的列名
year_cols = [f'year_{y}' for y in range(2019, 2026)]
month_cols = [f'month_{m}' for m in range(2, 13)]

print(f"\n生成的年份虚拟变量 (以2018为基准): {year_cols}")
print(f"生成的月份虚拟变量 (以1月为基准): {month_cols}")

print(f"\n处理后: {df.shape}")
print(f"新增列数: {len(year_cols) + len(month_cols)} (7个年份 + 11个月份)")

# 保存
output = r"H:\HW\ruc_Class25Q2_test_price12345.csv"
try:
    df.to_csv(output, index=False, encoding='utf-8-sig')
    print(f"\n✅ 已保存: {output}")
except PermissionError:
    output = r"H:\HW\ruc_Class25Q2_test_price12345_new.csv"
    df.to_csv(output, index=False, encoding='utf-8-sig')
    print(f"\n✅ 原文件被占用，已保存为: {output}")

原始: (34017, 45)

交易时间样例:
0    2025-02-21
1    2025-01-17
2    2025-04-03
3    2025-01-31
4    2025-01-22
5    2025-01-24
6    2025-04-05
7    2025-01-23
8    2025-01-20
9    2025-01-24
Name: 交易时间, dtype: object

年份分布:
Series([], Name: count, dtype: int64)

月份分布:
Series([], Name: count, dtype: int64)

生成的年份虚拟变量 (以2018为基准): ['year_2019', 'year_2020', 'year_2021', 'year_2022', 'year_2023', 'year_2024', 'year_2025']
生成的月份虚拟变量 (以1月为基准): ['month_2', 'month_3', 'month_4', 'month_5', 'month_6', 'month_7', 'month_8', 'month_9', 'month_10', 'month_11', 'month_12']

处理后: (34017, 63)
新增列数: 18 (7个年份 + 11个月份)

✅ 已保存: H:\HW\ruc_Class25Q2_test_price12345.csv


In [72]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist

# 读取test数据集
test_path = r"H:\HW\ruc_Class25Q2_test_price12345.csv"
df_test = pd.read_csv(test_path, encoding='utf-8')

# 读取train数据集（包含聚类变量）
train_path = r"H:\HW\ruc_Class25Q2_train_price_clean3_blk_smart_features_final_with_city.csv"
df_train = pd.read_csv(train_path, encoding='utf-8')

print(f"Test数据: {df_test.shape}")
print(f"Train数据: {df_train.shape}")

# 找到所有聚类变量列
blk_cols = [col for col in df_train.columns if col.startswith('blk_k_')]
print(f"\n找到 {len(blk_cols)} 个聚类变量")
print(f"聚类变量样例: {blk_cols[:5]}")

# 查找城市列名
city_col_test = [col for col in df_test.columns if '城' in col][0]
city_col_train = [col for col in df_train.columns if '城' in col][0]

print(f"\nTest城市列: {city_col_test}")
print(f"Train城市列: {city_col_train}")

# 初始化聚类变量列（全部设为0）
for col in blk_cols:
    df_test[col] = 0

# 按城市匹配
unique_cities = df_test[city_col_test].unique()
print(f"\n开始匹配，共 {len(unique_cities)} 个城市...")

for city in sorted(unique_cities):
    # 筛选该城市的test和train数据
    test_city_idx = df_test[city_col_test] == city
    train_city = df_train[df_train[city_col_train] == city].copy()
    
    if len(train_city) == 0:
        print(f"  城市 {city}: Train中无数据，跳过")
        continue
    
    # 提取经纬度
    test_coords = df_test.loc[test_city_idx, ['lon', 'lat']].values
    train_coords = train_city[['lon', 'lat']].values
    
    # 计算距离矩阵
    distances = cdist(test_coords, train_coords, metric='euclidean')
    
    # 找到每个test样本最近的train样本索引
    nearest_train_idx = np.argmin(distances, axis=1)
    
    # 获取最近的train记录的聚类变量
    matched_features = train_city.iloc[nearest_train_idx][blk_cols].values
    
    # 赋值给test数据
    df_test.loc[test_city_idx, blk_cols] = matched_features
    
    print(f"  城市 {city}: 匹配了 {test_city_idx.sum()} 条test数据 (Train有 {len(train_city)} 条)")

print(f"\n处理后Test数据: {df_test.shape}")
print(f"新增列数: {len(blk_cols)}")

# 验证聚类变量
print(f"\n聚类变量赋值验证:")
for col in blk_cols[:3]:
    print(f"  {col}: sum = {df_test[col].sum()}")

# 保存
output = r"H:\HW\ruc_Class25Q2_test_price_final.csv"
try:
    df_test.to_csv(output, index=False, encoding='utf-8-sig')
    print(f"\n✅ 已保存: {output}")
except PermissionError:
    output = r"H:\HW\ruc_Class25Q2_test_price_final_new.csv"
    df_test.to_csv(output, index=False, encoding='utf-8-sig')
    print(f"\n✅ 原文件被占用，已保存为: {output}")

Test数据: (34017, 63)
Train数据: (103871, 369)

找到 305 个聚类变量
聚类变量样例: ['blk_k_0.0', 'blk_k_1.0', 'blk_k_2.0', 'blk_k_3.0', 'blk_k_4.0']

Test城市列: 城市
Train城市列: 城市

开始匹配，共 12 个城市...
  城市 0: 匹配了 7057 条test数据 (Train有 16491 条)
  城市 1: 匹配了 1985 条test数据 (Train有 6437 条)
  城市 2: 匹配了 5431 条test数据 (Train有 24996 条)
  城市 3: 匹配了 3961 条test数据 (Train有 21472 条)
  城市 4: 匹配了 5602 条test数据 (Train有 4363 条)
  城市 5: 匹配了 1057 条test数据 (Train有 3582 条)
  城市 6: 匹配了 933 条test数据 (Train有 2281 条)
  城市 7: 匹配了 879 条test数据 (Train有 1184 条)
  城市 8: 匹配了 2051 条test数据 (Train有 5931 条)
  城市 9: 匹配了 1182 条test数据 (Train有 1323 条)
  城市 10: 匹配了 3694 条test数据 (Train有 15057 条)
  城市 11: 匹配了 185 条test数据 (Train有 754 条)

处理后Test数据: (34017, 368)
新增列数: 305

聚类变量赋值验证:
  blk_k_0.0: sum = 71
  blk_k_1.0: sum = 116
  blk_k_2.0: sum = 66

✅ 已保存: H:\HW\ruc_Class25Q2_test_price_final.csv


In [73]:
# -*- coding: utf-8 -*-
import pandas as pd

# 读取文件
path = r"H:\HW\ruc_Class25Q2_test_price_final.csv"
df = pd.read_csv(path, encoding='utf-8')

print(f"数据: {df.shape}")

# 查找建筑面积列
area_cols = [col for col in df.columns if '面积' in col]
print(f"\n包含'面积'的列: {area_cols}")

# 找到建筑面积列
building_area_col = [col for col in area_cols if '建筑' in col][0]
print(f"建筑面积列: {building_area_col}")

# 查看处理前的数据
print(f"\n处理前样例:")
print(df[building_area_col].head(10))

# 去掉"m²"和可能的空格
df[building_area_col] = df[building_area_col].astype(str).str.replace('m²', '').str.replace('㎡', '').str.strip()

# 转换为数值类型
df[building_area_col] = pd.to_numeric(df[building_area_col], errors='coerce')

# 查看处理后的数据
print(f"\n处理后样例:")
print(df[building_area_col].head(10))

# 保存到原文件
df.to_csv(path, index=False, encoding='utf-8-sig')
print(f"\n✅ 已更新原文件: {path}")

数据: (34017, 368)

包含'面积'的列: ['建筑面积']
建筑面积列: 建筑面积

处理前样例:
0    282.02㎡
1     88.42㎡
2    175.52㎡
3    106.13㎡
4     116.8㎡
5    107.21㎡
6     209.2㎡
7     53.59㎡
8    112.98㎡
9    136.29㎡
Name: 建筑面积, dtype: object

处理后样例:
0    282.02
1     88.42
2    175.52
3    106.13
4    116.80
5    107.21
6    209.20
7     53.59
8    112.98
9    136.29
Name: 建筑面积, dtype: float64

✅ 已更新原文件: H:\HW\ruc_Class25Q2_test_price_final.csv


In [2]:
# -*- coding: utf-8 -*-
"""
批量预测 - 四大Price模型（智能动态裁剪）
"""

import pandas as pd
import numpy as np
import joblib
import warnings
import gc
import os
from datetime import datetime
warnings.filterwarnings('ignore')

print("=" * 80)
print("四模型批量预测 - 对数Price模型（智能动态裁剪）")
print("=" * 80)

# ===================== 配置四个模型 =====================
MODELS = [
    {'name': 'OLS', 'model_file': 'model_OLS_log_price.pkl'},
    {'name': 'Ridge', 'model_file': 'model_Ridge_log_price.pkl'},
    {'name': 'Lasso', 'model_file': 'model_Lasso_log_price.pkl'},
    {'name': 'ElasticNet', 'model_file': 'model_ElasticNet_log_price.pkl'}
]

# ===================== 路径配置 =====================
path_test = r"H:\HW\ruc_Class25Q2_test_price_final.csv"
path_train = r"H:\HW\ruc_Class25Q2_train_price_clean3_blk_smart_features_final_with_city.csv"
output_dir = r"H:\HW"

scaler_path = 'scaler_log_price.pkl'
features_path = 'selected_features_log_price.pkl'

# ===================== 特征配置 =====================
CONTINUOUS_FEATURES = [
    '建筑面积', '房屋总数', '楼栋总数', 
    '绿 化 率', '容 积 率', '物 业 费', 
    '室', '厅', '厨', '卫', '房龄'
]

# ===================== 开关 =====================
enable_clipping = True
enable_calibration = True
enable_min_fix = True


# ===================== 1. 加载公共资源 =====================
print("\nStep 1: 加载公共资源（Scaler + Features）")
print("=" * 80)

try:
    scaler = joblib.load(scaler_path)
    print(f"✅ Scaler: {scaler_path}")
    
    selected_features = joblib.load(features_path)
    print(f"✅ Selected Features: {len(selected_features)} 个")
    
    if hasattr(scaler, 'feature_names_in_'):
        scaler_features = list(scaler.feature_names_in_)
        print(f"✅ Scaler实际特征: {len(scaler_features)}")
    else:
        scaler_features = []
        print(f"ℹ️ Scaler没有feature_names_in_")
        
except Exception as e:
    print(f"❌ 加载失败: {e}")
    exit()


# ===================== 2. 计算裁剪界限（基于训练集）=====================
print("\nStep 2: 计算裁剪界限（基于训练集）")
print("=" * 80)

log_lower_bound = 11.0
log_upper_bound = 16.5
train_mean = 2000000
train_median = 1800000
min_price_threshold = 150000
target_mean_ratio = 1.20

try:
    df_train_stats = pd.read_csv(path_train, encoding='utf-8', usecols=['Price'])
    
    train_log_price = np.log1p(df_train_stats['Price'])
    train_log_mean = train_log_price.mean()
    train_log_std = train_log_price.std()
    train_log_min = train_log_price.min()
    train_log_max = train_log_price.max()
    train_log_p99 = train_log_price.quantile(0.99)
    
    train_mean = df_train_stats['Price'].mean()
    train_median = df_train_stats['Price'].median()
    train_p5 = df_train_stats['Price'].quantile(0.05)
    train_min = df_train_stats['Price'].min()
    train_max = df_train_stats['Price'].max()
    
    print(f"训练集统计:")
    print(f"  Price范围: [{train_min:,.0f}, {train_max:,.0f}]")
    print(f"  均价: {train_mean:,.0f}  中位数: {train_median:,.0f}")
    print(f"  log(Price+1)范围: [{train_log_min:.4f}, {train_log_max:.4f}]")
    print(f"  均值±3.5σ: [{train_log_mean - 3.5*train_log_std:.4f}, {train_log_mean + 3.5*train_log_std:.4f}]")
    
    strategy1_lower = train_log_mean - 3.5 * train_log_std
    strategy2_lower = train_log_min - 0.5
    strategy2_upper = train_log_max + 0.3
    strategy3_upper = train_log_p99 + 0.2
    
    log_lower_bound = max(strategy1_lower, strategy2_lower, 11.0)
    log_upper_bound = min(strategy2_upper, strategy3_upper, 16.8)
    
    min_price_threshold = train_p5 * 0.85
    
    print(f"\n✅ 裁剪界限:")
    print(f"  对数空间: [{log_lower_bound:.4f}, {log_upper_bound:.4f}]")
    print(f"  对应Price: [{np.expm1(log_lower_bound):,.0f}, {np.expm1(log_upper_bound):,.0f}]")
    print(f"  最小值阈值: {min_price_threshold:,.0f}")
    print(f"  校准目标: 均价 = 训练集 * {target_mean_ratio}")
    
    del df_train_stats, train_log_price
    gc.collect()
    
except Exception as e:
    print(f"⚠️ 统计失败: {e}")
    print(f"使用默认值")


# ===================== 3. 加载测试集 + ID处理 =====================
print("\nStep 3: 加载测试集 + ID处理")
print("=" * 80)

try:
    df_test_raw = pd.read_csv(path_test, encoding='utf-8')
    print(f"✅ 测试集: {df_test_raw.shape}")
    print(f"   列名: {list(df_test_raw.columns[:5])}")
    
    # 智能ID检测
    id_col_name = None
    possible_id_cols = ['ID', 'id', 'Id', 'index', '编号', 'Unnamed: 0']
    
    for col in possible_id_cols:
        if col in df_test_raw.columns:
            id_col_name = col
            break
    
    if id_col_name:
        test_ids = df_test_raw[id_col_name].copy()
        print(f"✅ 检测到ID列: '{id_col_name}' ({len(test_ids)})")
        df_test = df_test_raw.drop(columns=[id_col_name])
    else:
        print(f"⚠️ 未检测到ID列，自动生成")
        test_ids = pd.Series(range(len(df_test_raw)), name='ID')
        df_test = df_test_raw.copy()
    
    # 删除多余列
    cols_to_drop = [c for c in ['城市', '交易时间'] if c in df_test.columns]
    if cols_to_drop:
        df_test = df_test.drop(columns=cols_to_drop)
        print(f"✅ 删除列: {cols_to_drop}")
    
    print(f"✅ 建模数据: {df_test.shape}")
    
except Exception as e:
    print(f"❌ 测试集加载失败: {e}")
    raise


# ===================== 4. 特征工程（一次性）=====================
print("\nStep 4: 特征工程（一次性处理）")
print("=" * 80)

def prepare_test_data(df):
    """特征工程"""
    df = df.copy()
    
    # 删除城市特征
    city_cols = [col for col in df.columns if col.startswith('city_')]
    if city_cols:
        df = df.drop(columns=city_cols)
        print(f"✅ 删除 {len(city_cols)} 个城市特征")
    
    # 对数转换
    log_features_created = []
    for col in CONTINUOUS_FEATURES:
        if col in df.columns:
            log_col = f'{col}_log'
            if log_col in selected_features:
                df[log_col] = np.log1p(df[col].clip(lower=0))
                log_features_created.append(log_col)
    
    if log_features_created:
        print(f"✅ 对数特征: {len(log_features_created)}")
    
    # 交互项
    if '室' in df.columns and '厅' in df.columns:
        df['居住房间数'] = df['室'] + df['厅']
        
        if '建筑面积' in df.columns and '建筑面积_per_居住房间' in selected_features:
            df['居住房间数_adjusted'] = df['居住房间数'].replace(0, 1)
            df['建筑面积_per_居住房间'] = df['建筑面积'] / df['居住房间数_adjusted']
            df = df.drop(columns=['居住房间数_adjusted'])
            print(f"✅ 建筑面积_per_居住房间")
    
    # 面积分箱
    if '建筑面积' in df.columns:
        area_bins = [f for f in selected_features if f.startswith('面积_') and '户型' in f]
        if area_bins:
            df['面积_小户型'] = (df['建筑面积'] < 70).astype(int)
            df['面积_中户型'] = ((df['建筑面积'] >= 70) & (df['建筑面积'] <= 120)).astype(int)
            df['面积_大户型'] = (df['建筑面积'] > 120).astype(int)
            print(f"✅ 面积分箱")
    
    # 特征对齐
    missing_count = 0
    for f in selected_features:
        if f not in df.columns:
            df[f] = 0
            missing_count += 1
    
    if missing_count > 0:
        print(f"⚠️ 填充 {missing_count} 个缺失特征")
    
    df = df[selected_features].copy()
    
    return df

X_test_base = prepare_test_data(df_test)
print(f"✅ 特征工程完成: {X_test_base.shape}")


# ===================== 5. 标准化（一次性）=====================
print("\nStep 5: 标准化（一次性）")
print("=" * 80)

scaler_features_clean = [f for f in scaler_features if not f.startswith('city_')]

if len(scaler_features_clean) > 0:
    X_for_scaling = pd.DataFrame(index=X_test_base.index)
    for f in scaler_features_clean:
        if f in X_test_base.columns:
            X_for_scaling[f] = X_test_base[f]
        else:
            X_for_scaling[f] = 0
    
    X_for_scaling = X_for_scaling[scaler_features_clean].fillna(0)
    
    try:
        scaled_data = scaler.transform(X_for_scaling)
        
        X_test_scaled = X_test_base.copy()
        for i, f in enumerate(scaler_features_clean):
            if f in X_test_scaled.columns:
                X_test_scaled[f] = scaled_data[:, i]
        
        print(f"✅ 标准化完成")
        
        del X_for_scaling
        
    except Exception as e:
        print(f"❌ 标准化失败: {e}")
        X_test_scaled = X_test_base
else:
    X_test_scaled = X_test_base

del X_test_base, df_test
gc.collect()


# ===================== 6. 批量预测 =====================
print("\n" + "=" * 80)
print("开始批量预测（4个模型）")
print("=" * 80)

results = {}

for model_config in MODELS:
    model_name = model_config['name']
    model_file = model_config['model_file']
    
    print(f"\n{'='*60}")
    print(f"模型: {model_name}")
    print(f"{'='*60}")
    
    # 6.1 加载模型
    try:
        model = joblib.load(model_file)
        print(f"✅ 已加载: {model_file}")
        
        # 从模型获取特征顺序
        if hasattr(model, 'feature_names_in_'):
            model_features = list(model.feature_names_in_)
            print(f"✅ 模型特征数: {len(model_features)}")
            
            # 按模型要求顺序对齐
            X_test_aligned = pd.DataFrame()
            missing_features = []
            for feat in model_features:
                if feat in X_test_scaled.columns:
                    X_test_aligned[feat] = X_test_scaled[feat]
                else:
                    X_test_aligned[feat] = 0
                    missing_features.append(feat)
            
            if missing_features:
                print(f"⚠️ 缺失 {len(missing_features)} 个特征，已填充0")
            
            X_test_final = X_test_aligned.fillna(0)
        else:
            X_test_final = X_test_scaled.fillna(0)
        
    except Exception as e:
        print(f"❌ 加载失败: {e}")
        continue
    
    # 6.2 预测
    try:
        y_pred_log = model.predict(X_test_final)
        print(f"✅ 预测完成")
        
        print(f"\n对数预测:")
        print(f"  均值: {y_pred_log.mean():.4f}")
        print(f"  范围: [{y_pred_log.min():.4f}, {y_pred_log.max():.4f}]")
        
        # 6.3 智能裁剪
        if enable_clipping:
            too_low = (y_pred_log < log_lower_bound).sum()
            too_high = (y_pred_log > log_upper_bound).sum()
            
            if too_low > 0 or too_high > 0:
                print(f"\n对数空间裁剪:")
                print(f"  < {log_lower_bound:.4f}: {too_low} ({too_low/len(y_pred_log)*100:.2f}%)")
                print(f"  > {log_upper_bound:.4f}: {too_high} ({too_high/len(y_pred_log)*100:.2f}%)")
                
                y_pred_log_clipped = np.clip(y_pred_log, log_lower_bound, log_upper_bound)
                print(f"  ✅ 已裁剪")
            else:
                y_pred_log_clipped = y_pred_log
                print(f"  ✅ 无需裁剪")
        else:
            y_pred_log_clipped = y_pred_log
        
        # 6.4 反变换
        y_pred = np.expm1(y_pred_log_clipped)
        
        # 6.5 智能校准
        if enable_calibration:
            test_mean_before = y_pred.mean()
            calibration_factor = min((train_mean * target_mean_ratio) / test_mean_before, 1.0)
            
            if calibration_factor < 0.95:
                y_pred = y_pred * calibration_factor
                print(f"\n校准:")
                print(f"  校准系数: {calibration_factor:.4f}")
                print(f"  校准后均价: {y_pred.mean():,.0f} (目标: {train_mean * target_mean_ratio:,.0f})")
            else:
                print(f"\n校准: 无需校准 (系数={calibration_factor:.4f})")
        
        # 6.6 最小值修正
        if enable_min_fix:
            too_low_mask = y_pred < min_price_threshold
            if too_low_mask.sum() > 0:
                print(f"\n最小值修正:")
                print(f"  修正 {too_low_mask.sum()} 个低值 -> {min_price_threshold:,.0f}")
                y_pred[too_low_mask] = min_price_threshold
        
        # 6.7 统计
        print(f"\n最终统计:")
        print(f"  均价: {y_pred.mean():,.0f}")
        print(f"  中位数: {np.median(y_pred):,.0f}")
        print(f"  范围: [{y_pred.min():,.0f}, {y_pred.max():,.0f}]")
        print(f"  标准差: {y_pred.std():,.0f}")
        
        # 与训练集对比
        mean_diff = (y_pred.mean() - train_mean) / train_mean * 100
        median_diff = (np.median(y_pred) - train_median) / train_median * 100
        print(f"  vs训练集: 均价{mean_diff:+.1f}%, 中位数{median_diff:+.1f}%")
        
        # 保存结果
        results[model_name] = y_pred
        
        print(f"✅ {model_name} 完成")
        
    except Exception as e:
        print(f"❌ {model_name} 预测失败: {e}")
        import traceback
        traceback.print_exc()
        continue
    
    gc.collect()


# ===================== 7. 保存文件 =====================
print("\n" + "=" * 80)
print("保存预测文件")
print("=" * 80)

for model_name, y_pred in results.items():
    # 简洁版：只有Price列
    output_simple = os.path.join(output_dir, f"pred_{model_name}.csv")
    df_simple = pd.DataFrame({'Price': y_pred})
    
    try:
        df_simple.to_csv(output_simple, index=False, encoding='utf-8-sig')
        print(f"✅ {model_name}: pred_{model_name}.csv")
    except Exception as e:
        print(f"❌ {model_name} 保存失败: {e}")
    
    # 带ID的版本
    output_with_id = os.path.join(output_dir, f"pred_{model_name}_id.csv")
    df_with_id = pd.DataFrame({
        test_ids.name: test_ids,
        'Price': y_pred
    })
    
    try:
        df_with_id.to_csv(output_with_id, index=False, encoding='utf-8-sig')
        print(f"   + pred_{model_name}_id.csv (带ID)")
    except:
        pass


# ===================== 8. 对比分析 =====================
print("\n" + "=" * 80)
print("四模型对比分析")
print("=" * 80)

if len(results) > 0:
    comparison = []
    for model_name, y_pred in results.items():
        comparison.append({
            '模型': model_name,
            '均价': f"{y_pred.mean():,.0f}",
            '中位数': f"{np.median(y_pred):,.0f}",
            '最小值': f"{y_pred.min():,.0f}",
            '最大值': f"{y_pred.max():,.0f}",
            '标准差': f"{y_pred.std():,.0f}"
        })
    
    df_comparison = pd.DataFrame(comparison)
    print(df_comparison.to_string(index=False))
    
    # 计算模型间差异
    if len(results) >= 2:
        pred_array = np.array([results[name] for name in results.keys()])
        pred_std = pred_array.std(axis=0).mean()
        pred_range = pred_array.max(axis=0) - pred_array.min(axis=0)
        
        print(f"\n模型间差异:")
        print(f"  平均标准差: {pred_std:,.0f}")
        print(f"  平均极差: {pred_range.mean():,.0f}")
        print(f"  极差中位数: {np.median(pred_range):,.0f}")
        
        if pred_std < 50000:
            print(f"  🎉 模型高度一致！")
        elif pred_std < 100000:
            print(f"  ✅ 模型较为一致")
        else:
            print(f"  ⚠️ 模型差异较大，建议集成")
    
    # 保存对比结果
    comparison_file = os.path.join(output_dir, 'pred_comparison.csv')
    try:
        df_comparison.to_csv(comparison_file, index=False, encoding='utf-8-sig')
        print(f"\n💾 对比结果: pred_comparison.csv")
    except:
        pass


# ===================== 9. 质量检查 =====================
print("\n" + "=" * 80)
print("质量检查（vs训练集）")
print("=" * 80)

if len(results) > 0:
    # 使用第一个模型的结果做质量检查
    first_model = list(results.keys())[0]
    y_pred_check = results[first_model]
    
    test_stats = {
        'mean': y_pred_check.mean(),
        'median': np.median(y_pred_check),
        'p25': np.percentile(y_pred_check, 25),
        'p75': np.percentile(y_pred_check, 75),
        'p95': np.percentile(y_pred_check, 95)
    }
    
    train_stats = {
        'mean': train_mean,
        'median': train_median,
    }
    
    mean_diff = (test_stats['mean'] - train_stats['mean']) / train_stats['mean'] * 100
    median_diff = (test_stats['median'] - train_stats['median']) / train_stats['median'] * 100
    
    print(f"以 {first_model} 为例:")
    print(f"  均价差异: {mean_diff:+.1f}%")
    print(f"  中位数差异: {median_diff:+.1f}%")
    
    if abs(mean_diff) < 15 and abs(median_diff) < 20:
        print(f"  🎉 优秀！分布非常接近训练集")
    elif abs(mean_diff) < 25 and abs(median_diff) < 30:
        print(f"  ✅ 良好！分布较接近训练集")
    else:
        print(f"  ⚠️ 一般，分布有些偏离")


print("\n" + "=" * 80)
print("🎉 批量预测完成！")
print("=" * 80)
print(f"\n生成文件:")
for model_name in results.keys():
    print(f"  - pred_{model_name}.csv")
    print(f"  - pred_{model_name}_id.csv")
print(f"  - pred_comparison.csv (对比表)")
print(f"\nID来源: {test_ids.name} ({test_ids.iloc[0]} - {test_ids.iloc[-1]})")
print("=" * 80)

四模型批量预测 - 对数Price模型（智能动态裁剪）

Step 1: 加载公共资源（Scaler + Features）
✅ Scaler: scaler_log_price.pkl
✅ Selected Features: 398 个
✅ Scaler实际特征: 9

Step 2: 计算裁剪界限（基于训练集）
训练集统计:
  Price范围: [79,777, 42,114,017]
  均价: 2,241,952  中位数: 1,479,407
  log(Price+1)范围: [11.2870, 17.5559]
  均值±3.5σ: [11.3793, 17.1467]

✅ 裁剪界限:
  对数空间: [11.3793, 16.5797]
  对应Price: [87,487, 15,866,680]
  最小值阈值: 364,925
  校准目标: 均价 = 训练集 * 1.2

Step 3: 加载测试集 + ID处理
✅ 测试集: (34017, 368)
   列名: ['ID', '城市', '建筑面积', '交易时间', 'lon']
✅ 检测到ID列: 'ID' (34017)
✅ 删除列: ['城市', '交易时间']
✅ 建模数据: (34017, 365)

Step 4: 特征工程（一次性处理）
✅ 删除 11 个城市特征
✅ 对数特征: 6
✅ 建筑面积_per_居住房间
✅ 面积分箱
⚠️ 填充 350 个缺失特征
✅ 特征工程完成: (34017, 398)

Step 5: 标准化（一次性）
✅ 标准化完成

开始批量预测（4个模型）

模型: OLS
✅ 已加载: model_OLS_log_price.pkl
✅ 模型特征数: 398
✅ 预测完成

对数预测:
  均值: 13.4226
  范围: [11.3821, 27.6117]

对数空间裁剪:
  < 11.3793: 0 (0.00%)
  > 16.5797: 10 (0.03%)
  ✅ 已裁剪

校准: 无需校准 (系数=1.0000)

最小值修正:
  修正 3857 个低值 -> 364,925

最终统计:
  均价: 763,742
  中位数: 696,805
  范围: [364,925, 15,866,680]
  标准差: 